In [ ]:
%%writefile example.c
#include <stdio.h>
#include <stdlib.h>

// Define the vector_int structure
typedef struct {
    int* data;
    size_t size;
} vector_int;

// Function to initialize vector_int
void init_vector_int(vector_int* vec, size_t size) {
    vec->data = (int*)malloc(size * sizeof(int));
    vec->size = size;
}

// Function to free memory used by vector_int
void free_vector_int(vector_int* vec) {
    free(vec->data);
    vec->data = NULL;
    vec->size = 0;
}

// Define the tuple_int_int structure
typedef struct {
    vector_int first;
    vector_int second;
} tuple_int_int;

// Function to create a tuple_int_int
tuple_int_int make_tuple(vector_int first, vector_int second) {
    tuple_int_int t;
    t.first = first;
    t.second = second;
    return t;
}

// Function equivalent to highLevel_conversion_For_MatrixB
tuple_int_int highLevel_conversion_For_MatrixB(vector_int rowPtr, vector_int Colidx) {
    // Initialize colPtr and rowIdx
    vector_int colPtr, rowIdx;
    init_vector_int(&colPtr, Colidx.size);  // colPtr size is based on Colidx size
    init_vector_int(&rowIdx, Colidx.size);  // rowIdx size is also based on Colidx size
    int max = 0;

    // Populate colPtr and rowIdx
    for (int i = 0; i < rowPtr.size - 1; i++) {
        for (int j = rowPtr.data[i]; j < rowPtr.data[i + 1]; j++) {
            int col = Colidx.data[j];
            for (int k = col + 1; k < colPtr.size; k++) {
                colPtr.data[k]++;
            }
            int insertInd = colPtr.data[col];
            for (int k = max; k > insertInd; k--) {
                rowIdx.data[k] = rowIdx.data[k - 1];
            }
            rowIdx.data[insertInd] = i;
            max++;
        }
    }

    // Sort rowIdx within each column range (if needed)
    // For simplicity, we can skip sorting here since it's not implemented in the original C++ code

    // Return the result as a tuple
    tuple_int_int result = make_tuple(colPtr, rowIdx);

    // Free dynamically allocated memory
    free_vector_int(&colPtr);
    // No need to free rowIdx here since it's returned and will be managed by the caller

    return result;
}

int main() {
    // Sample usage of highLevel_conversion_For_MatrixB

    // Sample input (simulated rowPtr and Colidx)
    int rowPtr_data[] = {0, 2, 5, 7, 8};
    int Colidx_data[] = {2, 0, 1, 3, 1, 2, 0, 3};

    // Create vector_int from array data
    vector_int rowPtr, Colidx;
    init_vector_int(&rowPtr, sizeof(rowPtr_data) / sizeof(rowPtr_data[0]));
    init_vector_int(&Colidx, sizeof(Colidx_data) / sizeof(Colidx_data[0]));

    // Copy data to vector_int arrays
    for (size_t i = 0; i < rowPtr.size; i++) {
        rowPtr.data[i] = rowPtr_data[i];
    }
    for (size_t i = 0; i < Colidx.size; i++) {
        Colidx.data[i] = Colidx_data[i];
    }

    // Call the function
    tuple_int_int result = highLevel_conversion_For_MatrixB(rowPtr, Colidx);

    // Accessing the result (for demonstration)
    printf("colPtr: ");
    for (size_t i = 0; i < result.first.size; i++) {
        printf("%d ", result.first.data[i]);
    }
    printf("\nrowIdx: ");
    for (size_t i = 0; i < result.second.size; i++) {
        printf("%d ", result.second.data[i]);
    }
    printf("\n");

    // Free dynamically allocated memory
    free_vector_int(&rowPtr);
    // result.second will be freed by the caller after use

    return 0;
}

Overwriting example.c


In [ ]:
%%writefile example.c
#include <stdio.h>
#include <stdlib.h>

typedef struct {
    int* data;
    size_t size;
} vector;

typedef struct {
    vector colPtr;
    vector rowIdx;
} tuple;

tuple highLevel_conversion_For_MatrixB(vector rowPtr, vector Colidx) {
    vector colPtr;
    colPtr.data = (int*)calloc(rowPtr.size, sizeof(int));
    colPtr.size = rowPtr.size;

    vector rowIdx;
    rowIdx.data = (int*)calloc(Colidx.size, sizeof(int));
    rowIdx.size = Colidx.size;

    int max_val = 0;

    for (int i = 0; i < rowPtr.size - 1; i++) {
        for (int j = rowPtr.data[i]; j < rowPtr.data[i + 1]; j++) {
            int col = Colidx.data[j];
            for (int k = col + 1; k < colPtr.size; k++) {
                colPtr.data[k]++;
            }
            int insertInd = colPtr.data[col];
            for (int k = insertInd + 1; k < max_val; k++) {
                rowIdx.data[k] = rowIdx.data[k - 1];
            }
            rowIdx.data[insertInd] = i;
            max_val++;
        }
    }

    for (int i = 0; i < colPtr.size - 1; i++) {
        // Sort rowIdx within the range [colPtr[i], colPtr[i + 1])
        for (int j = colPtr.data[i]; j < colPtr.data[i + 1]; j++) {
            for (int k = j + 1; k < colPtr.data[i + 1]; k++) {
                if (rowIdx.data[j] > rowIdx.data[k]) {
                    int temp = rowIdx.data[j];
                    rowIdx.data[j] = rowIdx.data[k];
                    rowIdx.data[k] = temp;
                }
            }
        }
    }

    tuple result;
    result.colPtr = colPtr;
    result.rowIdx = rowIdx;

    return result;
}

int main() {
    // Example usage
    int rowPtr_data[] = {0, 2, 3, 4, 6};
    int Colidx_data[] = {0, 3, 1, 3, 1, 3};

    vector rowPtr;
    rowPtr.data = rowPtr_data;
    rowPtr.size = sizeof(rowPtr_data) / sizeof(int);

    vector Colidx;
    Colidx.data = Colidx_data;
    Colidx.size = sizeof(Colidx_data) / sizeof(int);

    tuple result = highLevel_conversion_For_MatrixB(rowPtr, Colidx);

    printf("colPtr: ");
    for (size_t i = 0; i < result.colPtr.size; i++) {
        printf("%d ", result.colPtr.data[i]);
    }
    printf("\n");

    printf("rowIdx: ");
    for (size_t i = 0; i < result.rowIdx.size; i++) {
        printf("%d ", result.rowIdx.data[i]);
    }
    printf("\n");

    // Free allocated memory
    free(result.colPtr.data);
    free(result.rowIdx.data);

    return 0;
}

Overwriting example.c


In [ ]:
%%writefile example2.c
#include <stdio.h>
#include <stdlib.h>

typedef struct {
    int* data;
    size_t size;
} vector;

typedef struct {
    vector colPtr;
    vector rowIdx;
} tuple;

tuple highLevel_conversion_For_MatrixB(vector rowPtr, vector Colidx) {
    vector colPtr;
    colPtr.data = (int*)calloc(rowPtr.size, sizeof(int));
    colPtr.size = rowPtr.size;

    vector rowIdx;
    rowIdx.data = (int*)calloc(Colidx.size, sizeof(int));
    rowIdx.size = Colidx.size;

    int max_val = 0;

    for (int i = 0; i < rowPtr.size - 1; i++) {
        for (int j = rowPtr.data[i]; j < rowPtr.data[i + 1]; j++) {
            int col = Colidx.data[j];
            for (int k = col + 1; k < colPtr.size; k++) {
                colPtr.data[k]++;
            }
            int insertInd = colPtr.data[col];
            for (int k = insertInd + 1; k < max_val; k++) {
                rowIdx.data[k] = rowIdx.data[k - 1];
            }
            rowIdx.data[insertInd] = i;
            max_val++;
        }
    }

    for (int i = 0; i < colPtr.size - 1; i++) {
        // Sort rowIdx within the range [colPtr[i], colPtr[i + 1])
        for (int j = colPtr.data[i]; j < colPtr.data[i + 1]; j++) {
            for (int k = j + 1; k < colPtr.data[i + 1]; k++) {
                if (rowIdx.data[j] > rowIdx.data[k]) {
                    int temp = rowIdx.data[j];
                    rowIdx.data[j] = rowIdx.data[k];
                    rowIdx.data[k] = temp;
                }
            }
        }
    }

    tuple result;
    result.colPtr = colPtr;
    result.rowIdx = rowIdx;

    return result;
}

int main() {
    // Example usage
    int rowPtr_data[] = {0, 2, 3, 4, 6};
    int Colidx_data[] = {0, 3, 1, 3, 1, 3};

    vector rowPtr;
    rowPtr.data = rowPtr_data;
    rowPtr.size = sizeof(rowPtr_data) / sizeof(int);

    vector Colidx;
    Colidx.data = Colidx_data;
    Colidx.size = sizeof(Colidx_data) / sizeof(int);

    tuple result = highLevel_conversion_For_MatrixB(rowPtr, Colidx);

    printf("colPtr: ");
    for (size_t i = 0; i < result.colPtr.size; i++) {
        printf("%d ", result.colPtr.data[i]);
    }
    printf("\n");

    printf("rowIdx: ");
    for (size_t i = 0; i < result.rowIdx.size; i++) {
        printf("%d ", result.rowIdx.data[i]);
    }
    printf("\n");

    // Free allocated memory
    free(result.colPtr.data);
    free(result.rowIdx.data);

    return 0;
}

Writing example2.c


In [ ]:
!gcc example3.c -o example1

In [ ]:
!./example1

colPtr: 0 1 3 3 6 
rowIdx: 0 1 3 0 2 3 


In [ ]:
%%writefile example3.c
#include <stdio.h>
#include <stdlib.h>

typedef struct {
    int* data;
    size_t size;
} vector;

typedef struct {
    vector colPtr;
    vector rowIdx;
} tuple;

tuple highLevel_conversion_For_MatrixB(vector rowPtr, vector Colidx) {
    vector colPtr;
    colPtr.data = (int*)calloc(rowPtr.size, sizeof(int));
    colPtr.size = rowPtr.size;

    vector rowIdx;
    rowIdx.data = (int*)calloc(Colidx.size, sizeof(int));
    rowIdx.size = Colidx.size;

    for (int i = 0; i < rowPtr.size - 1; i++) {
        for (int j = rowPtr.data[i]; j < rowPtr.data[i + 1]; j++) {
            int col = Colidx.data[j];
            colPtr.data[col + 1]++;
        }
    }

    for (int i = 0; i < colPtr.size - 1; i++) {
        colPtr.data[i + 1] += colPtr.data[i];
    }

    int* counter = (int*)calloc(colPtr.size, sizeof(int));
    for (int i = 0; i < rowPtr.size - 1; i++) {
        for (int j = rowPtr.data[i]; j < rowPtr.data[i + 1]; j++) {
            int col = Colidx.data[j];
            int idx = colPtr.data[col] + counter[col];
            rowIdx.data[idx] = i;
            counter[col]++;
        }
    }

    free(counter);

    tuple result;
    result.colPtr = colPtr;
    result.rowIdx = rowIdx;

    return result;
}

int main() {
    // Example usage
    int rowPtr_data[] = {0, 2, 3, 4, 6};
    int Colidx_data[] = {0, 3, 1, 3, 1, 3};

    vector rowPtr;
    rowPtr.data = rowPtr_data;
    rowPtr.size = sizeof(rowPtr_data) / sizeof(int);

    vector Colidx;
    Colidx.data = Colidx_data;
    Colidx.size = sizeof(Colidx_data) / sizeof(int);

    tuple result = highLevel_conversion_For_MatrixB(rowPtr, Colidx);

    printf("colPtr: ");
    for (size_t i = 0; i < result.colPtr.size; i++) {
        printf("%d ", result.colPtr.data[i]);
    }
    printf("\n");

    printf("rowIdx: ");
    for (size_t i = 0; i < result.rowIdx.size; i++) {
        printf("%d ", result.rowIdx.data[i]);
    }
    printf("\n");

    // Free allocated memory
    free(result.colPtr.data);
    free(result.rowIdx.data);

    return 0;
}

Overwriting example3.c


In [ ]:
%%cuda
#include <algorithm>
#include <vector>
#include <iostream>
#include <bits/stdc++.h>
using namespace std;

tuple<vector<int>, vector<int>> highLevel_conversion_For_MatrixB(vector<int> rowPtr, vector<int> Colidx)
{
    vector<int> colPtr(rowPtr.size(), 0);
    vector<int> rowIdx(Colidx.size(), 0);
    int max = 0;

    for (int i = 0; i < rowPtr.size() - 1; i++)
    {
        for (int j = rowPtr[i]; j < rowPtr[i + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < colPtr.size(); k++)
            {
                colPtr[k]++;
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < max; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = i;
            max++;
        }
    }
    for (int i = 0; i < colPtr.size() - 1; i++)
    {
        sort(rowIdx.begin() + colPtr[i], rowIdx.begin() + colPtr[i + 1]);
    }
    return make_tuple(colPtr, rowIdx);
}

int main()
{
    int tile_size = 4;
    vector<int> tileColPtr_B, tileRowidx_B;
    vector<int>tilerowPtr_B = {0, 1, 3, 4, 5};
    vector<int>tileColidx_B = {3, 0, 2, 1, 2};
    tie(tileColPtr_B, tileRowidx_B) = highLevel_conversion_For_MatrixB(tilerowPtr_B, tileColidx_B);
    for (auto it: tileColPtr_B){
        cout<<it<<" ";
    }
    cout<<endl;
    for(auto jt: tileRowidx_B){
      cout<<jt<<" ";
    }
    return 0;
}

0 1 2 4 5 
1 2 1 3 0 


In [ ]:
%%cuda
#include <iostream>
#include <vector>

// CUDA kernel for high-level conversion of Matrix B
__global__ void highLevelConversionForMatrixB(int* rowPtr, int* Colidx, int* colPtr, int* rowIdx, int num_rows)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < num_rows - 1)
    {
        for (int j = rowPtr[tid]; j < rowPtr[tid + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < num_rows; k++)
            {
                atomicAdd(&colPtr[k], 1);
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < num_rows; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = tid;
        }
    }

    // Custom sorting for rowIdx within the specified range
    if (tid < num_rows - 1)
    {
        for (int i = colPtr[tid]; i < colPtr[tid + 1] - 1; i++)
        {
            for (int j = i + 1; j < colPtr[tid + 1]; j++)
            {
                if (rowIdx[i] > rowIdx[j])
                {
                    int temp = rowIdx[i];
                    rowIdx[i] = rowIdx[j];
                    rowIdx[j] = temp;
                }
            }
        }
    }
}

int main()
{
    int tile_size = 4;
    int num_rows = 5; // Example: replace with the actual number of rows

    // Initialize input data (tileRowPtr_B and tileColidx_B)
    int tilerowPtr_B[] = {0, 1, 3, 4, 5};
    int tileColidx_B[] = {3, 0, 2, 1, 2};

    // Allocate memory on the device
    int* d_rowPtr, *d_Colidx, *d_colPtr, *d_rowIdx;
    cudaMalloc(&d_rowPtr, num_rows * sizeof(int));
    cudaMalloc(&d_Colidx, tilerowPtr_B[num_rows - 1] * sizeof(int));
    cudaMalloc(&d_colPtr, (num_rows + 1) * sizeof(int)); // Add +1 for the last element
    cudaMalloc(&d_rowIdx, tilerowPtr_B[num_rows - 1] * sizeof(int));

    // Initialize colPtr and rowIdx with zeros
    cudaMemset(d_colPtr, 0, (num_rows + 1) * sizeof(int));
    cudaMemset(d_rowIdx, 0, tilerowPtr_B[num_rows - 1] * sizeof(int));

    // Copy input data to the device
    cudaMemcpy(d_rowPtr, tilerowPtr_B, num_rows * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_Colidx, tileColidx_B, tilerowPtr_B[num_rows - 1] * sizeof(int), cudaMemcpyHostToDevice);

    // Launch the CUDA kernel
    int num_blocks = (num_rows + tile_size - 1) / tile_size;
    highLevelConversionForMatrixB<<<num_blocks, tile_size>>>(d_rowPtr, d_Colidx, d_colPtr, d_rowIdx, num_rows);

    // Copy results back to the host
    int colPtr[num_rows + 1], rowIdx[tilerowPtr_B[num_rows - 1]];
    cudaMemcpy(colPtr, d_colPtr, (num_rows + 1) * sizeof(int), cudaMemcpyDeviceToHost);
    cudaMemcpy(rowIdx, d_rowIdx, tilerowPtr_B[num_rows - 1] * sizeof(int), cudaMemcpyDeviceToHost);

    // Print the results
    for (int i = 0; i < num_rows; i++)
    {
        std::cout << "colPtr[" << i << "] = " << colPtr[i] << std::endl;
    }
    for (int i = 0; i < tilerowPtr_B[num_rows - 1]; i++)
    {
        std::cout << "rowIdx[" << i << "] = " << rowIdx[i] << std::endl;
    }

    // Free allocated memory
    cudaFree(d_rowPtr);
    cudaFree(d_Colidx);
    cudaFree(d_colPtr);
    cudaFree(d_rowIdx);

    return 0;
}

colPtr[0] = 0
colPtr[1] = 1
colPtr[2] = 2
colPtr[3] = 4
colPtr[4] = 5
rowIdx[0] = 1
rowIdx[1] = 2
rowIdx[2] = 1
rowIdx[3] = 3
rowIdx[4] = 3



In [ ]:
%%cuda
#include <iostream>
#include <vector>

// CUDA kernel for high-level conversion of Matrix B
__global__ void highLevelConversionForMatrixB(int* rowPtr, int* Colidx, int* colPtr, int* rowIdx, int num_rows)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < num_rows - 1)
    {
        for (int j = rowPtr[tid]; j < rowPtr[tid + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < num_rows; k++)
            {
                atomicAdd(&colPtr[k], 1);
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < num_rows; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = tid;
        }
    }
}

int main()
{
    int tile_size = 4;
    int num_rows = 5; // Example: replace with the actual number of rows

    // Initialize input data (tileRowPtr_B and tileColidx_B)
    int tilerowPtr_B[] = {0, 1, 3, 4, 5};
    int tileColidx_B[] = {3, 0, 2, 1, 2};

    // Allocate memory on the device
    int* d_rowPtr, *d_Colidx, *d_colPtr, *d_rowIdx;
    cudaMalloc(&d_rowPtr, num_rows * sizeof(int));
    cudaMalloc(&d_Colidx, tilerowPtr_B[num_rows - 1] * sizeof(int));
    cudaMalloc(&d_colPtr, (num_rows + 1) * sizeof(int)); // Add +1 for the last element
    cudaMalloc(&d_rowIdx, tilerowPtr_B[num_rows - 1] * sizeof(int));

    // Initialize colPtr with zeros
    cudaMemset(d_colPtr, 0, (num_rows + 1) * sizeof(int));

    // Copy input data to the device
    cudaMemcpy(d_rowPtr, tilerowPtr_B, num_rows * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_Colidx, tileColidx_B, tilerowPtr_B[num_rows - 1] * sizeof(int), cudaMemcpyHostToDevice);

    // Launch the CUDA kernel
    int num_blocks = (num_rows + tile_size - 1) / tile_size;
    highLevelConversionForMatrixB<<<num_blocks, tile_size>>>(d_rowPtr, d_Colidx, d_colPtr, d_rowIdx, num_rows);

    // Copy results back to the host
    int colPtr[num_rows + 1], rowIdx[tilerowPtr_B[num_rows - 1]];
    cudaMemcpy(colPtr, d_colPtr, (num_rows + 1) * sizeof(int), cudaMemcpyDeviceToHost);
    cudaMemcpy(rowIdx, d_rowIdx, tilerowPtr_B[num_rows - 1] * sizeof(int), cudaMemcpyDeviceToHost);

    // Print the results
    for (int i = 0; i < num_rows; i++)
    {
        std::cout << "colPtr[" << i << "] = " << colPtr[i] << std::endl;
    }
    for (int i = 0; i < tilerowPtr_B[num_rows - 1]; i++)
    {
        std::cout << "rowIdx[" << i << "] = " << rowIdx[i] << std::endl;
    }

    // Free allocated memory
    cudaFree(d_rowPtr);
    cudaFree(d_Colidx);
    cudaFree(d_colPtr);
    cudaFree(d_rowIdx);

    return 0;
}

colPtr[0] = 0
colPtr[1] = 1
colPtr[2] = 2
colPtr[3] = 4
colPtr[4] = 5
rowIdx[0] = 1
rowIdx[1] = 2
rowIdx[2] = 1
rowIdx[3] = 3
rowIdx[4] = 3



In [ ]:
%%cuda
#include <iostream>
#include <vector>

// CUDA kernel for high-level conversion of Matrix B
__global__ void highLevelConversionForMatrixB(int* rowPtr, int* Colidx, int* colPtr, int* rowIdx, int num_rows)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < num_rows - 1)
    {
        for (int j = rowPtr[tid]; j < rowPtr[tid + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < num_rows; k++)
            {
                atomicAdd(&colPtr[k], 1);
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < num_rows; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = tid;
        }
    }
}

int main()
{
    int tile_size = 4;
    int num_rows = 5; // Example: replace with the actual number of rows

    // Initialize input data (tileRowPtr_B and tileColidx_B)
    int tilerowPtr_B[] = {0, 1, 3, 4, 5};
    int tileColidx_B[] = {3, 0, 2, 1, 2};

    // Allocate memory on the device
    int* d_rowPtr, *d_Colidx, *d_colPtr, *d_rowIdx;
    cudaMalloc(&d_rowPtr, num_rows * sizeof(int));
    cudaMalloc(&d_Colidx, tilerowPtr_B[num_rows - 1] * sizeof(int));
    cudaMalloc(&d_colPtr, (num_rows + 1) * sizeof(int)); // Add +1 for the last element
    cudaMalloc(&d_rowIdx, tilerowPtr_B[num_rows - 1] * sizeof(int));

    // Initialize colPtr with zeros
    cudaMemset(d_colPtr, 0, (num_rows + 1) * sizeof(int));

    // Copy input data to the device
    cudaMemcpy(d_rowPtr, tilerowPtr_B, num_rows * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_Colidx, tileColidx_B, tilerowPtr_B[num_rows - 1] * sizeof(int), cudaMemcpyHostToDevice);

    // Launch the CUDA kernel
    int num_blocks = (num_rows + tile_size - 1) / tile_size;
    highLevelConversionForMatrixB<<<num_blocks, tile_size>>>(d_rowPtr, d_Colidx, d_colPtr, d_rowIdx, num_rows);

    // Copy results back to the host
    int colPtr[num_rows + 1], rowIdx[tilerowPtr_B[num_rows - 1]];
    cudaMemcpy(colPtr, d_colPtr, (num_rows + 1) * sizeof(int), cudaMemcpyDeviceToHost);
    cudaMemcpy(rowIdx, d_rowIdx, tilerowPtr_B[num_rows - 1] * sizeof(int), cudaMemcpyDeviceToHost);

    // Print the results
    for (int i = 0; i < num_rows; i++)
    {
        std::cout << "colPtr[" << i << "] = " << colPtr[i] << std::endl;
    }
    for (int i = 0; i < tilerowPtr_B[num_rows - 1]; i++)
    {
        std::cout << "rowIdx[" << i << "] = " << rowIdx[i] << std::endl;
    }

    // Free allocated memory
    cudaFree(d_rowPtr);
    cudaFree(d_Colidx);
    cudaFree(d_colPtr);
    cudaFree(d_rowIdx);

    return 0;
}


colPtr[0] = 0
colPtr[1] = 1
colPtr[2] = 2
colPtr[3] = 4
colPtr[4] = 5
rowIdx[0] = 1
rowIdx[1] = 2
rowIdx[2] = 1
rowIdx[3] = 3
rowIdx[4] = 3



In [ ]:
%%cuda
#include <iostream>
#include <vector>

// CUDA kernel for high-level conversion of Matrix B
__global__ void highLevelConversionForMatrixB(int* rowPtr, int* Colidx, int* colPtr, int* rowIdx, int num_rows)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < num_rows - 1)
    {
        for (int j = rowPtr[tid]; j < rowPtr[tid + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < num_rows; k++)
            {
                atomicAdd(&colPtr[k], 1);
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < num_rows; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = tid;
        }
    }

    // Custom sorting for rowIdx within the specified range
    if (tid < num_rows - 1)
    {
        for (int i = colPtr[tid]; i < colPtr[tid + 1] - 1; i++)
        {
            for (int j = i + 1; j < colPtr[tid + 1]; j++)
            {
                if (rowIdx[i] > rowIdx[j])
                {
                    int temp = rowIdx[i];
                    rowIdx[i] = rowIdx[j];
                    rowIdx[j] = temp;
                }
            }
        }
    }
}

int main()
{
    int tile_size = 4;
    int num_rows = 5; // Example: replace with the actual number of rows

    // Initialize input data (tileRowPtr_B and tileColidx_B)
    int tilerowPtr_B[] = {0, 1, 3, 4, 5};
    int tileColidx_B[] = {3, 0, 2, 1, 2};

    // Allocate memory on the device
    int* d_rowPtr, *d_Colidx, *d_colPtr, *d_rowIdx;
    cudaMalloc(&d_rowPtr, num_rows * sizeof(int));
    cudaMalloc(&d_Colidx, tilerowPtr_B[num_rows - 1] * sizeof(int));
    cudaMalloc(&d_colPtr, (num_rows + 1) * sizeof(int)); // Add +1 for the last element
    cudaMalloc(&d_rowIdx, tilerowPtr_B[num_rows - 1] * sizeof(int));

    // Initialize colPtr with zeros
    cudaMemset(d_colPtr, 0, (num_rows + 1) * sizeof(int));

    // Copy input data to the device
    cudaMemcpy(d_rowPtr, tilerowPtr_B, num_rows * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_Colidx, tileColidx_B, tilerowPtr_B[num_rows - 1] * sizeof(int), cudaMemcpyHostToDevice);

    // Launch the CUDA kernel
    int num_blocks = (num_rows + tile_size - 1) / tile_size;
    highLevelConversionForMatrixB<<<num_blocks, tile_size>>>(d_rowPtr, d_Colidx, d_colPtr, d_rowIdx, num_rows);

    // Copy results back to the host
    int colPtr[num_rows + 1], rowIdx[tilerowPtr_B[num_rows - 1]];
    cudaMemcpy(colPtr, d_colPtr, (num_rows + 1) * sizeof(int), cudaMemcpyDeviceToHost);
    cudaMemcpy(rowIdx, d_rowIdx, tilerowPtr_B[num_rows - 1] * sizeof(int), cudaMemcpyDeviceToHost);

    // Print the results
    for (int i = 0; i < num_rows; i++)
    {
        std::cout << "colPtr[" << i << "] = " << colPtr[i] << std::endl;
    }
    for (int i = 0; i < tilerowPtr_B[num_rows - 1]; i++)
    {
        std::cout << "rowIdx[" << i << "] = " << rowIdx[i] << std::endl;
    }

    // Free allocated memory
    cudaFree(d_rowPtr);
    cudaFree(d_Colidx);
    cudaFree(d_colPtr);
    cudaFree(d_rowIdx);

    return 0;
}


colPtr[0] = 0
colPtr[1] = 1
colPtr[2] = 2
colPtr[3] = 4
colPtr[4] = 5
rowIdx[0] = 1
rowIdx[1] = 2
rowIdx[2] = 1
rowIdx[3] = 3
rowIdx[4] = 3



In [ ]:
%%cuda
#include <iostream>
#include <vector>

// CUDA kernel for high-level conversion of Matrix B
__global__ void highLevelConversionForMatrixB(int* rowPtr, int* Colidx, int* colPtr, int* rowIdx, int num_rows)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < num_rows - 1)
    {
        for (int j = rowPtr[tid]; j < rowPtr[tid + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < num_rows; k++)
            {
                atomicAdd(&colPtr[k], 1);
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < num_rows; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = tid;
        }
    }

    // Custom sorting for rowIdx within the specified range
    if (tid < num_rows - 1)
    {
        for (int i = colPtr[tid]; i < colPtr[tid + 1] - 1; i++)
        {
            for (int j = i + 1; j < colPtr[tid + 1]; j++)
            {
                if (rowIdx[i] > rowIdx[j])
                {
                    int temp = rowIdx[i];
                    rowIdx[i] = rowIdx[j];
                    rowIdx[j] = temp;
                }
            }
        }
    }
}

int main()
{
    int tile_size = 4;
    int num_rows = 5; // Example: replace with the actual number of rows

    // Initialize input data (tileRowPtr_B and tileColidx_B)
    int tilerowPtr_B[] = {0, 1, 3, 4, 5};
    int tileColidx_B[] = {3, 0, 2, 1, 2};

    // Allocate memory on the device
    int* d_rowPtr, *d_Colidx, *d_colPtr, *d_rowIdx;
    cudaMalloc(&d_rowPtr, num_rows * sizeof(int));
    cudaMalloc(&d_Colidx, tilerowPtr_B[num_rows - 1] * sizeof(int));
    cudaMalloc(&d_colPtr, (num_rows + 1) * sizeof(int)); // Add +1 for the last element
    cudaMalloc(&d_rowIdx, tilerowPtr_B[num_rows - 1] * sizeof(int));

    // Initialize colPtr with zeros
    cudaMemset(d_colPtr, 0, (num_rows + 1) * sizeof(int));

    // Copy input data to the device
    cudaMemcpy(d_rowPtr, tilerowPtr_B, num_rows * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_Colidx, tileColidx_B, tilerowPtr_B[num_rows - 1] * sizeof(int), cudaMemcpyHostToDevice);

    // Launch the CUDA kernel
    int num_blocks = (num_rows + tile_size - 1) / tile_size;
    highLevelConversionForMatrixB<<<num_blocks, tile_size>>>(d_rowPtr, d_Colidx, d_colPtr, d_rowIdx, num_rows);

    // Copy results back to the host
    int colPtr[num_rows + 1], rowIdx[tilerowPtr_B[num_rows - 1]];
    cudaMemcpy(colPtr, d_colPtr, (num_rows + 1) * sizeof(int), cudaMemcpyDeviceToHost);
    cudaMemcpy(rowIdx, d_rowIdx, tilerowPtr_B[num_rows - 1] * sizeof(int), cudaMemcpyDeviceToHost);

    // Print the results
    for (int i = 0; i < num_rows; i++)
    {
        std::cout << "colPtr[" << i << "] = " << colPtr[i] << std::endl;
    }
    for (int i = 0; i < tilerowPtr_B[num_rows - 1]; i++)
    {
        std::cout << "rowIdx[" << i << "] = " << rowIdx[i] << std::endl;
    }

    // Free allocated memory
    cudaFree(d_rowPtr);
    cudaFree(d_Colidx);
    cudaFree(d_colPtr);
    cudaFree(d_rowIdx);

    return 0;
}

colPtr[0] = 0
colPtr[1] = 1
colPtr[2] = 2
colPtr[3] = 4
colPtr[4] = 5
rowIdx[0] = 1
rowIdx[1] = 2
rowIdx[2] = 1
rowIdx[3] = 3
rowIdx[4] = 3



# Experiments

In [ ]:
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
!nvcc --version
!nvidia-smi
%load_ext nvcc4jupyter

  Cloning https://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-h46rnb2c
  Running command git clone --filter=blob:none --quiet https://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-h46rnb2c
  Resolved https://github.com/andreinechaev/nvcc4jupyter.git to commit 5741c522547756ac4bb7a16df32106a15efb8a57
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for nvcc4jupyter: filename=nvcc4jupyter-1.2.1-py3-none-any.whl size=10741 sha256=1c43b610d84440f376c57bb0b3d20f87e2433fdb6f06eb98d5fd7c81c461a971
  Stored in directory: /tmp/pip-ephem-wheel-cache-ne1wruza/wheels/a8/b9/18/23f8ef71ceb0f63297dd1903aedd067e6243a68ea756d6feea
Successfully built nvcc4jupyter
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2023 NVIDIA Corporation
Built on Tue_Aug_15_22:02:13_PDT_2023
Cuda compilation tools, release 12.2, V12.2.140
Build cuda_12.2.r12.2/compiler.33191640_0


# PARALLEL TILESPGEMM EXPERIMENTATION

In [ ]:
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
!nvcc --version
!nvidia-smi
%load_ext nvcc4jupyter

  Cloning https://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-4ur_24nz
  Running command git clone --filter=blob:none --quiet https://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-4ur_24nz
  Resolved https://github.com/andreinechaev/nvcc4jupyter.git to commit 5741c522547756ac4bb7a16df32106a15efb8a57
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for nvcc4jupyter: filename=nvcc4jupyter-1.2.1-py3-none-any.whl size=10741 sha256=1c43b610d84440f376c57bb0b3d20f87e2433fdb6f06eb98d5fd7c81c461a971
  Stored in directory: /tmp/pip-ephem-wheel-cache-ced5y2n5/wheels/a8/b9/18/23f8ef71ceb0f63297dd1903aedd067e6243a68ea756d6feea
Successfully built nvcc4jupyter
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2023 NVIDIA Corporation
Built on Tue_Aug_15_22:02:13_PDT_2023
Cuda compilation tools, release 12.2, V12.2.140
Build cuda_12.2.r12.2/compiler.33191640_0


In [ ]:
%%cuda
#include <random>
#include<bits/stdc++.h>
#include <algorithm>
#include <vector>
#include <iostream>
#include <sys/time.h>
#include <cuda_runtime.h>
#define MAX_MATCHED_POSITIONS 16

using namespace std;

tuple<vector<int>, vector<int>> symbolic_SpGEMM_For_CSRFormat(vector<int> tilerowPtr_A, vector<int> tileColidx_A, vector<int> tilerowPtr_B, vector<int> tileColidx_B)
{
    int n = tilerowPtr_A.size();
    int m = tilerowPtr_B.size();
    vector<vector<int>> C_prime;
    vector<int> rowIdx, colIdx;

    for (int i = 0; i < n - 1; ++i)
    {
        set<int> unique_col;
        for (int j = tilerowPtr_A[i]; j < tilerowPtr_A[i + 1]; ++j)
        {
            int col = tileColidx_A[j];
            for (int i1 = tilerowPtr_B[col]; i1 < tilerowPtr_B[col + 1]; ++i1)
            {
                int col1 = tileColidx_B[i1];
                unique_col.insert(col1);
            }
        }
        for (int col : unique_col)
        {
            colIdx.push_back(col);
            rowIdx.push_back(i);
        }
    }
    C_prime.push_back(rowIdx);
    C_prime.push_back(colIdx);
    return make_tuple(rowIdx, colIdx);
}

// CUDA kernel for bitmask conversion

__global__ void bitmask_conversion_kernel(int *matrix_B, int *matrix_mask_B, int num_elements)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < num_elements)
    {
        matrix_mask_B[tid] = (matrix_B[tid] != 0) ? 1 : 0;
    }
}

// CUDA kernel for Step 2 and Step 3

__global__ void step2_and_step3_kernel(int *matrix_A, int *matrix_B, int *tileRowidx_C, int *tileColidx_C, int *tilePtr_A,
                                      int *tilePtr_B, int *tileColPtr_B, int *tileColidx_A,
                                      int *tileRowidx_B, int numtileC, int tile_size, int *matrix_C, int *maskc,
                                      int *matrix_mask_B, int num_rows, int num_cols)
{

    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < numtileC)
    {

        int tile_i = tileRowidx_C[tid];
        int tile_j = tileColidx_C[tid];
        int lena = tilePtr_A[tile_i + 1] - tilePtr_A[tile_i];
        int lenb = tileColPtr_B[tile_j + 1] - tileColPtr_B[tile_j];

        // Array to store matched positions
        int matched_posA[MAX_MATCHED_POSITIONS];
        int num_matched = 0;

        // Applying Binary Search to find the matched positions
        if (lena <= lenb)
        {
            for (int value = tilePtr_A[tile_i]; value < tilePtr_A[tile_i + 1]; value++)
            {
                int low = tileColPtr_B[tile_j];
                int high = tileColPtr_B[tile_j + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileRowidx_B[mid] == tileColidx_A[value])
                    {
                        matched_posA[num_matched++] = tileColidx_A[value];
                        break;
                    }
                    else if (tileRowidx_B[mid] < tileColidx_A[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }
        else
        {
            for (int value = tileColPtr_B[tile_j]; value < tileColPtr_B[tile_j + 1]; value++)
            {
                int low = tilePtr_A[tile_i];
                int high = tilePtr_A[tile_i + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileColidx_A[mid] == tileRowidx_B[value])
                    {
                        matched_posA[num_matched++] = tileRowidx_B[value];
                        break;
                    }
                    else if (tileColidx_A[mid] < tileRowidx_B[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }

        for (int matched = 0; matched < num_matched; matched++)
        {
            int j = matched_posA[matched];
            for (int r = 0; r < tile_size; r++)
            {
                for (int s = 0; s < tile_size; s++)
                {
                    int row = tile_size * tile_i + r;
                    int col = tile_size * j + s;
                    if (matrix_A[row * num_cols + col] != 0)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            atomicOr(&maskc[row * num_cols + (tile_size * tile_j + t)], matrix_mask_B[(tile_size * j + s) * num_cols + (tile_size * tile_j + t)]);
                        }
                    }
                }
            }
        }
        int nnz = 0;
        for (int i = 0; i < tile_size; i++)
        {
            for (int j = 0; j < tile_size; j++)
            {
                // printf("%d",maskc[(tile_i * tile_size + i) * num_cols + (tile_j * tile_size + j)]);
                if (maskc[(tile_i * tile_size + i) * num_cols + (tile_j * tile_size + j)] != 0)
                {
                    nnz++;
                }
            }
        }

        if (nnz >= 12)
        {
            for (int matched = 0; matched < num_matched; matched++)
            {
                int j = matched_posA[matched];
                for (int r = 0; r < tile_size; r++)
                {
                    for (int s = 0; s < tile_size; s++)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            matrix_C[(tile_i * tile_size + r) * num_cols + (tile_size * tile_j + s)] += matrix_A[(tile_size * tile_i + r) * num_cols + tile_size * j + t] * matrix_B[(tile_size * j + t) * num_cols + (tile_size * tile_j + s)];
                        }
                    }
                }
            }
        }
        else
        {
            for (int matched = 0; matched < num_matched; matched++)
            {
                int matched_tiles = matched_posA[matched];
                for (int r = 0; r < tile_size; ++r)
                {
                    for (int s = 0; s < tile_size; ++s)
                    {
                        if(matrix_A[(tile_i*tile_size + r)*num_cols + matched_tiles*tile_size +s] == 0){
                            continue;
                        }

                        for (int t = 0; t < tile_size; ++t)
                        {

  matrix_C[(tile_i*tile_size + r)*num_cols + (tile_j*tile_size + t)] += matrix_A[(tile_i*tile_size + r)*num_cols + matched_tiles*tile_size +s] * matrix_B[(tile_size*matched_tiles + s )*num_cols + tile_j*tile_size + t];

                        }
                    }
                }
            }
        }
    }
}
/*
tuple<vector<int>, vector<int>> highLevel_conversion(vector<vector<int>> &matrix, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix.size(); i += tile_size)
    {
        for (int j = 0; j < matrix[i].size(); j += tile_size)
        {
            for (int k = 0; k < tile_size * tile_size; k++)
            {
                int row = k / tile_size;
                int col = k % tile_size;
                if (matrix[i + row][j + col] != 0)
                {
                    colIdx.push_back(j / tile_size);
                    break;
                }
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}
*/


tuple<vector<int>, vector<int>> highLevel_conversion(vector<int> &matrix, int matrix_size, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix_size; i += tile_size)
    {
        for (int j = 0; j < matrix_size; j += tile_size)
        {
            bool hasNonZero = false;
            for (int x = 0; x < tile_size; ++x)
            {
                for (int y = 0; y < tile_size; ++y)
                {
                    int row = i + x;
                    int col = j + y;
                    if (row < matrix_size && col < matrix_size && matrix[row * matrix_size + col] != 0)
                    {
                        colIdx.push_back(j / tile_size);
                        hasNonZero = true;
                        break;
                    }
                }
                if (hasNonZero)
                    break;
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}


tuple<vector<int>, vector<int>> highLevel_conversion_For_MatrixB(vector<int> rowPtr, vector<int> Colidx)
{
    vector<int> colPtr(rowPtr.size(), 0);
    vector<int> rowIdx(Colidx.size(), 0);
    int max = 0;

    for (int i = 0; i < rowPtr.size() - 1; i++)
    {
        for (int j = rowPtr[i]; j < rowPtr[i + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < colPtr.size(); k++)
            {
                colPtr[k]++;
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < max; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = i;
            max++;
        }
    }
    for (int i = 0; i < colPtr.size() - 1; i++)
    {
        sort(rowIdx.begin() + colPtr[i], rowIdx.begin() + colPtr[i + 1]);
    }
    return make_tuple(colPtr, rowIdx);
}
int main() {
    vector<int> tile_sizes = {4};
    vector<int> matrix_sizes = {8};
    vector<double>time;
    double density = 0.1;

    // Loop over tile sizes
    for (int tile_size : tile_sizes) {
        cout << "Tile Size: " << tile_size << endl;

        // Loop over matrix sizes
        for (int matrix_size : matrix_sizes) {
            cout << "Matrix Size: " << matrix_size << "x" << matrix_size << endl;

        vector<int>matrix_A = {1, 2, 0, 4, 0, 6, 0, 8,
                               0, 0, 0, 0, 0, 0, 1, 1,
                               1, 1, 0, 1, 1, 0, 0, 0,
                               0, 0, 3, 3, 0, 3, 1, 1,
                               1, 3, 0, 3, 1, 13,0, 3,
                               0, 0, 0, 0, 0, 0, 0, 0,
                               0, 0, 0, 0, 3, 0, 0, 0,
                               0, 2, 0, 0, 0, 0 ,1, 0};
        vector<int>matrix_B = {1, 0, 0, 0, 0, 0, 0, 2,
                               0, 0 ,0 ,2, 0, 0, 0, 0,
                               0, 0, 0, 0, 0, 0, 0, 0,
                               0, 0, 0 ,0, 0, 7, 0, 0,
                               0, 0, 0, 0, 0, 0, 0, 0,
                               0, 4, 0, 0, 0, 0, 0, 0,
                               0, 0, 0, 0, 4, 0, 0, 0,
                               0, 0, 0, 0, 0, 0, 0, 1};

            // Start measuring time
            struct timeval begin, end;
            gettimeofday(&begin, 0);

            // High-level conversion for matrix A
            vector<int> tilerowPtr_A, tileColidx_A;
            tie(tilerowPtr_A, tileColidx_A) = highLevel_conversion(matrix_A, matrix_size, tile_size);

            // High-level conversion for matrix B
            vector<int> tilerowPtr_B, tileColidx_B;
            tie(tilerowPtr_B, tileColidx_B) = highLevel_conversion(matrix_B, matrix_size, tile_size);

            // Symbolic SpGEMM for CSR Format
            vector<int> tileRowidx_C, tileColidx_C;
            tie(tileRowidx_C, tileColidx_C) = symbolic_SpGEMM_For_CSRFormat(tilerowPtr_A, tileColidx_A, tilerowPtr_B, tileColidx_B);

            int numtileC = tileRowidx_C.size();
            vector<int> tileColPtr_B, tileRowidx_B;
            tie(tileColPtr_B, tileRowidx_B) = highLevel_conversion_For_MatrixB(tilerowPtr_B, tileColidx_B);




             // Allocate memory for matrices and vectors on device
              int *d_matrix_A, *d_matrix_B, *d_tileRowidx_C, *d_tileColidx_C, *d_tilePtr_A,
                  *d_tilePtr_B, *d_tileColPtr_B, *d_tileColidx_A,
                  *d_tileRowidx_B, *d_matrix_C, *d_maskc, *d_matrix_mask_B;

              // CUDA memory allocation
              cudaMalloc((void **)&d_matrix_A, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_matrix_B, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_tileRowidx_C, sizeof(int) * tileRowidx_C.size());
              cudaMalloc((void **)&d_tileColidx_C, sizeof(int) * tileColidx_C.size());
              cudaMalloc((void **)&d_tilePtr_A, sizeof(int) * tilerowPtr_A.size());
              cudaMalloc((void **)&d_tilePtr_B, sizeof(int) * tilerowPtr_B.size());
              cudaMalloc((void **)&d_tileColPtr_B, sizeof(int) * tileColPtr_B.size());
              cudaMalloc((void **)&d_tileColidx_A, sizeof(int) * tileColidx_A.size());
              cudaMalloc((void **)&d_tileRowidx_B, sizeof(int) * tileRowidx_B.size());
              cudaMalloc((void **)&d_matrix_C, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_maskc, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_matrix_mask_B, sizeof(int) * matrix_size * matrix_size);

              // Copy data from host to device
              cudaMemcpy(d_matrix_A, matrix_A.data(), sizeof(int) * matrix_size * matrix_size, cudaMemcpyHostToDevice);
              cudaMemcpy(d_matrix_B, matrix_B.data(), sizeof(int) * matrix_size * matrix_size, cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileRowidx_C, tileRowidx_C.data(), sizeof(int) * tileRowidx_C.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColidx_C, tileColidx_C.data(), sizeof(int) * tileColidx_C.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tilePtr_A, tilerowPtr_A.data(), sizeof(int) * tilerowPtr_A.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tilePtr_B, tilerowPtr_B.data(), sizeof(int) * tilerowPtr_B.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColPtr_B, tileColPtr_B.data(), sizeof(int) * tileColPtr_B.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColidx_A, tileColidx_A.data(), sizeof(int) * tileColidx_A.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileRowidx_B, tileRowidx_B.data(), sizeof(int) * tileRowidx_B.size(), cudaMemcpyHostToDevice);

              // Define grid and block dimensions
              int numThreads = 256;
              int numBlocks = (numtileC + numThreads - 1) / numThreads;

              // Launch bitmask conversion kernel
              bitmask_conversion_kernel<<<numBlocks, numThreads>>>(d_matrix_B, d_matrix_mask_B, matrix_size * matrix_size);
              cudaDeviceSynchronize();

              // Launch Step 2 and Step 3 kernel
              step2_and_step3_kernel<<<numBlocks, numThreads>>>(d_matrix_A, d_matrix_B, d_tileRowidx_C, d_tileColidx_C, d_tilePtr_A,
                                                                d_tilePtr_B, d_tileColPtr_B, d_tileColidx_A,
                                                                d_tileRowidx_B, numtileC, tile_size, d_matrix_C, d_maskc, d_matrix_mask_B, matrix_size, matrix_size);
              cudaDeviceSynchronize();

              gettimeofday(&end, 0);
            long seconds = end.tv_sec - begin.tv_sec;
            long microseconds = end.tv_usec - begin.tv_usec;
            double elapsed = seconds + microseconds*1e-6;
            time.push_back(elapsed);
            printf("Time measured: %.6f seconds.\n", elapsed);

              int *matrix_C_result = new int[matrix_size * matrix_size];
              cudaMemcpy(matrix_C_result, d_matrix_C, sizeof(int) * matrix_size * matrix_size, cudaMemcpyDeviceToHost);
            // Print matrix_C
            cout << endl<<"Matrix C:" << endl;


            for (int i = 0; i < matrix_size; ++i)
            {
                for (int j = 0; j < matrix_size; ++j)
                {
                    cout << matrix_C_result[i * matrix_size + j] << " ";
                }
                cout << endl;
            }
            // Free host memory
            delete[] matrix_C_result;

            // Copy result matrix from device to host
            // Free device memory
            cudaFree(d_matrix_A);
            cudaFree(d_matrix_B);
            cudaFree(d_tileRowidx_C);
            cudaFree(d_tileColidx_C);
            cudaFree(d_tilePtr_A);
            cudaFree(d_tilePtr_B);
            cudaFree(d_tileColPtr_B);
            cudaFree(d_tileColidx_A);
            cudaFree(d_tileRowidx_B);
            cudaFree(d_matrix_C);
            cudaFree(d_maskc);
            cudaFree(d_matrix_mask_B);
        }


    }
    return 0;
}

Tile Size: 4
Matrix Size: 8x8
6  9  4  3  Time measured: 0.458498 seconds.

Matrix C:
1 24 0 4 0 28 0 10 
0 0 0 0 4 0 0 1 
1 0 0 2 0 7 0 2 
0 12 0 0 4 21 0 1 
1 52 0 6 0 21 0 5 
0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 
0 0 0 4 4 0 0 0 



# Density 0.1 (very sparse)

# Tile 64

In [ ]:
%%cuda
#include <random>
#include<bits/stdc++.h>
#include <algorithm>
#include <vector>
#include <iostream>
#include <sys/time.h>
#include <cuda_runtime.h>
#define MAX_MATCHED_POSITIONS 16

using namespace std;

vector<int> generateRandomSparseMatrix(int rows, int cols, double density) {
    vector<int> matrix(rows*cols);
    std::random_device rd;
    std::mt19937 gen(rd());
    std::uniform_real_distribution<> dis(0.0, 1.0);
    for (int i = 0; i < rows; ++i) {
        for (int j = 0; j < cols; ++j) {
            double randNum = dis(gen);
            if (randNum <= density) {
                matrix[i*rows+j] = 1;
            }
        }
    }
    return matrix;
}

tuple<vector<int>, vector<int>> symbolic_SpGEMM_For_CSRFormat(vector<int> tilerowPtr_A, vector<int> tileColidx_A, vector<int> tilerowPtr_B, vector<int> tileColidx_B)
{
    int n = tilerowPtr_A.size();
    int m = tilerowPtr_B.size();
    vector<vector<int>> C_prime;
    vector<int> rowIdx, colIdx;

    for (int i = 0; i < n - 1; ++i)
    {
        set<int> unique_col;
        for (int j = tilerowPtr_A[i]; j < tilerowPtr_A[i + 1]; ++j)
        {
            int col = tileColidx_A[j];
            for (int i1 = tilerowPtr_B[col]; i1 < tilerowPtr_B[col + 1]; ++i1)
            {
                int col1 = tileColidx_B[i1];
                unique_col.insert(col1);
            }
        }
        for (int col : unique_col)
        {
            colIdx.push_back(col);
            rowIdx.push_back(i);
        }
    }
    C_prime.push_back(rowIdx);
    C_prime.push_back(colIdx);
    return make_tuple(rowIdx, colIdx);
}

// CUDA kernel for bitmask conversion

__global__ void bitmask_conversion_kernel(int *matrix_B, int *matrix_mask_B, int num_elements)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < num_elements)
    {
        matrix_mask_B[tid] = (matrix_B[tid] != 0) ? 1 : 0;
    }
}

// CUDA kernel for Step 2 and Step 3

__global__ void step2_and_step3_kernel(int *matrix_A, int *matrix_B, int *tileRowidx_C, int *tileColidx_C, int *tilePtr_A,
                                      int *tilePtr_B, int *tileColPtr_B, int *tileColidx_A,
                                      int *tileRowidx_B, int numtileC, int tile_size, int *matrix_C, int *maskc,
                                      int *matrix_mask_B, int num_rows, int num_cols)
{

    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < numtileC)
    {

        int tile_i = tileRowidx_C[tid];
        int tile_j = tileColidx_C[tid];
        int lena = tilePtr_A[tile_i + 1] - tilePtr_A[tile_i];
        int lenb = tileColPtr_B[tile_j + 1] - tileColPtr_B[tile_j];

        // Array to store matched positions
        int matched_posA[MAX_MATCHED_POSITIONS];
        int num_matched = 0;

        // Applying Binary Search to find the matched positions
        if (lena <= lenb)
        {
            for (int value = tilePtr_A[tile_i]; value < tilePtr_A[tile_i + 1]; value++)
            {
                int low = tileColPtr_B[tile_j];
                int high = tileColPtr_B[tile_j + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileRowidx_B[mid] == tileColidx_A[value])
                    {
                        matched_posA[num_matched++] = tileColidx_A[value];
                        break;
                    }
                    else if (tileRowidx_B[mid] < tileColidx_A[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }
        else
        {
            for (int value = tileColPtr_B[tile_j]; value < tileColPtr_B[tile_j + 1]; value++)
            {
                int low = tilePtr_A[tile_i];
                int high = tilePtr_A[tile_i + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileColidx_A[mid] == tileRowidx_B[value])
                    {
                        matched_posA[num_matched++] = tileRowidx_B[value];
                        break;
                    }
                    else if (tileColidx_A[mid] < tileRowidx_B[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }

        for (int matched = 0; matched < num_matched; matched++)
        {
            int j = matched_posA[matched];
            for (int r = 0; r < tile_size; r++)
            {
                for (int s = 0; s < tile_size; s++)
                {
                    int row = tile_size * tile_i + r;
                    int col = tile_size * j + s;
                    if (matrix_A[row * num_cols + col] != 0)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            atomicOr(&maskc[row * num_cols + (tile_size * tile_j + t)], matrix_mask_B[(tile_size * j + s) * num_cols + (tile_size * tile_j + t)]);
                        }
                    }
                }
            }
        }
        int nnz = 0;
        for (int i = 0; i < tile_size; i++)
        {
            for (int j = 0; j < tile_size; j++)
            {
                // printf("%d",maskc[(tile_i * tile_size + i) * num_cols + (tile_j * tile_size + j)]);
                if (maskc[(tile_i * tile_size + i) * num_cols + (tile_j * tile_size + j)] != 0)
                {
                    nnz++;
                }
            }
        }

        if (nnz >= 12)
        {
            for (int matched = 0; matched < num_matched; matched++)
            {
                int j = matched_posA[matched];
                for (int r = 0; r < tile_size; r++)
                {
                    for (int s = 0; s < tile_size; s++)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            matrix_C[(tile_i * tile_size + r) * num_cols + (tile_size * tile_j + s)] += matrix_A[(tile_size * tile_i + r) * num_cols + tile_size * j + t] * matrix_B[(tile_size * j + t) * num_cols + (tile_size * tile_j + s)];
                        }
                    }
                }
            }
        }
        else
        {
            for (int matched = 0; matched < num_matched; matched++)
            {
                int matched_tiles = matched_posA[matched];
                for (int r = 0; r < tile_size; ++r)
                {
                    for (int s = 0; s < tile_size; ++s)
                    {
                        if(matrix_A[(tile_i*tile_size + r)*num_cols + matched_tiles*tile_size +s] == 0){
                            continue;
                        }

                        for (int t = 0; t < tile_size; ++t)
                        {

  matrix_C[(tile_i*tile_size + r)*num_cols + (tile_j*tile_size + t)] += matrix_A[(tile_i*tile_size + r)*num_cols + matched_tiles*tile_size +s] * matrix_B[(tile_size*matched_tiles + s )*num_cols + tile_j*tile_size + t];

                        }
                    }
                }
            }
        }
    }
}
/*
tuple<vector<int>, vector<int>> highLevel_conversion(vector<vector<int>> &matrix, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix.size(); i += tile_size)
    {
        for (int j = 0; j < matrix[i].size(); j += tile_size)
        {
            for (int k = 0; k < tile_size * tile_size; k++)
            {
                int row = k / tile_size;
                int col = k % tile_size;
                if (matrix[i + row][j + col] != 0)
                {
                    colIdx.push_back(j / tile_size);
                    break;
                }
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}
*/


tuple<vector<int>, vector<int>> highLevel_conversion(vector<int> &matrix, int matrix_size, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix_size; i += tile_size)
    {
        for (int j = 0; j < matrix_size; j += tile_size)
        {
            bool hasNonZero = false;
            for (int x = 0; x < tile_size; ++x)
            {
                for (int y = 0; y < tile_size; ++y)
                {
                    int row = i + x;
                    int col = j + y;
                    if (row < matrix_size && col < matrix_size && matrix[row * matrix_size + col] != 0)
                    {
                        colIdx.push_back(j / tile_size);
                        hasNonZero = true;
                        break;
                    }
                }
                if (hasNonZero)
                    break;
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}


tuple<vector<int>, vector<int>> highLevel_conversion_For_MatrixB(vector<int> rowPtr, vector<int> Colidx)
{
    vector<int> colPtr(rowPtr.size(), 0);
    vector<int> rowIdx(Colidx.size(), 0);
    int max = 0;

    for (int i = 0; i < rowPtr.size() - 1; i++)
    {
        for (int j = rowPtr[i]; j < rowPtr[i + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < colPtr.size(); k++)
            {
                colPtr[k]++;
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < max; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = i;
            max++;
        }
    }
    for (int i = 0; i < colPtr.size() - 1; i++)
    {
        sort(rowIdx.begin() + colPtr[i], rowIdx.begin() + colPtr[i + 1]);
    }
    return make_tuple(colPtr, rowIdx);
}
int main() {
    // Parameters
    vector<int> tile_sizes = {64};
    vector<int> matrix_sizes = {128, 256, 384, 512, 640, 768, 896, 1024, 1152, 1280, 1408, 1536, 1664, 1792, 1920, 2048, 2176, 2304, 2432, 2560, 2688, 2816, 2944, 3072, 10000};
    vector<double>time;

    double density = 0.1;
    for (int tile_size : tile_sizes) {
        cout << "Tile Size: " << tile_size << endl;
        for (int matrix_size : matrix_sizes) {
            cout << "Matrix Size: " << matrix_size << "x" << matrix_size << endl;
            vector<int> matrix_A = generateRandomSparseMatrix(matrix_size, matrix_size, density);
            vector<int> matrix_B = generateRandomSparseMatrix(matrix_size, matrix_size, density);

            struct timeval begin, end;
            gettimeofday(&begin, 0);

            // High-level conversion for matrix A
            vector<int> tilerowPtr_A, tileColidx_A;
            tie(tilerowPtr_A, tileColidx_A) = highLevel_conversion(matrix_A, matrix_size, tile_size);

            // High-level conversion for matrix B
            vector<int> tilerowPtr_B, tileColidx_B;
            tie(tilerowPtr_B, tileColidx_B) = highLevel_conversion(matrix_B, matrix_size, tile_size);

            // Symbolic SpGEMM for CSR Format
            vector<int> tileRowidx_C, tileColidx_C;
            tie(tileRowidx_C, tileColidx_C) = symbolic_SpGEMM_For_CSRFormat(tilerowPtr_A, tileColidx_A, tilerowPtr_B, tileColidx_B);

            int numtileC = tileRowidx_C.size();
            vector<int> tileColPtr_B, tileRowidx_B;
            tie(tileColPtr_B, tileRowidx_B) = highLevel_conversion_For_MatrixB(tilerowPtr_B, tileColidx_B);




             // Allocate memory for matrices and vectors on device
              int *d_matrix_A, *d_matrix_B, *d_tileRowidx_C, *d_tileColidx_C, *d_tilePtr_A,
                  *d_tilePtr_B, *d_tileColPtr_B, *d_tileColidx_A,
                  *d_tileRowidx_B, *d_matrix_C, *d_maskc, *d_matrix_mask_B;

              // CUDA memory allocation
              cudaMalloc((void **)&d_matrix_A, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_matrix_B, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_tileRowidx_C, sizeof(int) * tileRowidx_C.size());
              cudaMalloc((void **)&d_tileColidx_C, sizeof(int) * tileColidx_C.size());
              cudaMalloc((void **)&d_tilePtr_A, sizeof(int) * tilerowPtr_A.size());
              cudaMalloc((void **)&d_tilePtr_B, sizeof(int) * tilerowPtr_B.size());
              cudaMalloc((void **)&d_tileColPtr_B, sizeof(int) * tileColPtr_B.size());
              cudaMalloc((void **)&d_tileColidx_A, sizeof(int) * tileColidx_A.size());
              cudaMalloc((void **)&d_tileRowidx_B, sizeof(int) * tileRowidx_B.size());
              cudaMalloc((void **)&d_matrix_C, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_maskc, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_matrix_mask_B, sizeof(int) * matrix_size * matrix_size);

              // Copy data from host to device
              cudaMemcpy(d_matrix_A, matrix_A.data(), sizeof(int) * matrix_size * matrix_size, cudaMemcpyHostToDevice);
              cudaMemcpy(d_matrix_B, matrix_B.data(), sizeof(int) * matrix_size * matrix_size, cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileRowidx_C, tileRowidx_C.data(), sizeof(int) * tileRowidx_C.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColidx_C, tileColidx_C.data(), sizeof(int) * tileColidx_C.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tilePtr_A, tilerowPtr_A.data(), sizeof(int) * tilerowPtr_A.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tilePtr_B, tilerowPtr_B.data(), sizeof(int) * tilerowPtr_B.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColPtr_B, tileColPtr_B.data(), sizeof(int) * tileColPtr_B.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColidx_A, tileColidx_A.data(), sizeof(int) * tileColidx_A.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileRowidx_B, tileRowidx_B.data(), sizeof(int) * tileRowidx_B.size(), cudaMemcpyHostToDevice);

              // Define grid and block dimensions
              int numThreads = 256;
              int numBlocks = (numtileC + numThreads - 1) / numThreads;

              // Launch bitmask conversion kernel
              bitmask_conversion_kernel<<<numBlocks, numThreads>>>(d_matrix_B, d_matrix_mask_B, matrix_size * matrix_size);
              cudaDeviceSynchronize();
              int *mask_B_matrix = new int[matrix_size * matrix_size];
              cudaMemcpy(mask_B_matrix, d_matrix_mask_B, sizeof(int) * matrix_size * matrix_size, cudaMemcpyDeviceToHost);

              // Launch Step 2 and Step 3 kernel
              step2_and_step3_kernel<<<numBlocks, numThreads>>>(d_matrix_A, d_matrix_B, d_tileRowidx_C, d_tileColidx_C, d_tilePtr_A,
                                                                d_tilePtr_B, d_tileColPtr_B, d_tileColidx_A,
                                                                d_tileRowidx_B, numtileC, tile_size, d_matrix_C, d_maskc, d_matrix_mask_B, matrix_size, matrix_size);
              cudaDeviceSynchronize();

              gettimeofday(&end, 0);
            long seconds = end.tv_sec - begin.tv_sec;
            long microseconds = end.tv_usec - begin.tv_usec;
            double elapsed = seconds + microseconds*1e-6;
            time.push_back(elapsed);
            printf("Time measured: %.6f seconds.\n", elapsed);

              int *matrix_C_result = new int[matrix_size * matrix_size];
              cudaMemcpy(matrix_C_result, d_matrix_C, sizeof(int) * matrix_size * matrix_size, cudaMemcpyDeviceToHost);
            // Free host memory
            delete[] matrix_C_result;

            // Copy result matrix from device to host
            // Free device memory
            cudaFree(d_matrix_A);
            cudaFree(d_matrix_B);
            cudaFree(d_tileRowidx_C);
            cudaFree(d_tileColidx_C);
            cudaFree(d_tilePtr_A);
            cudaFree(d_tilePtr_B);
            cudaFree(d_tileColPtr_B);
            cudaFree(d_tileColidx_A);
            cudaFree(d_tileRowidx_B);
            cudaFree(d_matrix_C);
            cudaFree(d_maskc);
            cudaFree(d_matrix_mask_B);
            cout << "hello" << endl;
        }


    }
    return 0;
}

Tile Size: 64
Matrix Size: 128x128
Time measured: 0.347997 seconds.
hello
Matrix Size: 256x256
Time measured: 0.122070 seconds.
hello
Matrix Size: 384x384
Time measured: 0.063753 seconds.
hello
Matrix Size: 512x512
Time measured: 0.050670 seconds.
hello
Matrix Size: 640x640
Time measured: 0.041306 seconds.
hello
Matrix Size: 768x768
Time measured: 0.048014 seconds.
hello
Matrix Size: 896x896
Time measured: 0.057205 seconds.
hello
Matrix Size: 1024x1024
Time measured: 0.061193 seconds.
hello
Matrix Size: 1152x1152
Time measured: 0.065416 seconds.
hello
Matrix Size: 1280x1280
Time measured: 0.067242 seconds.
hello
Matrix Size: 1408x1408
Time measured: 0.069979 seconds.
hello
Matrix Size: 1536x1536
Time measured: 0.099731 seconds.
hello
Matrix Size: 1664x1664
Time measured: 0.096534 seconds.
hello
Matrix Size: 1792x1792
Time measured: 0.097549 seconds.
hello
Matrix Size: 1920x1920
Time measured: 0.102245 seconds.
hello
Matrix Size: 2048x2048
Time measured: 0.101267 seconds.
hello
Matrix S

# Tile 32

In [ ]:
%%cuda
#include <random>
#include<bits/stdc++.h>
#include <algorithm>
#include <vector>
#include <iostream>
#include <sys/time.h>
#include <cuda_runtime.h>
#define MAX_MATCHED_POSITIONS 16

using namespace std;

vector<int> generateRandomSparseMatrix(int rows, int cols, double density) {
    vector<int> matrix(rows*cols);
    std::random_device rd;
    std::mt19937 gen(rd());
    std::uniform_real_distribution<> dis(0.0, 1.0);
    for (int i = 0; i < rows; ++i) {
        for (int j = 0; j < cols; ++j) {
            double randNum = dis(gen);
            if (randNum <= density) {
                matrix[i*rows+j] = 1;
            }
        }
    }
    return matrix;
}

tuple<vector<int>, vector<int>> symbolic_SpGEMM_For_CSRFormat(vector<int> tilerowPtr_A, vector<int> tileColidx_A, vector<int> tilerowPtr_B, vector<int> tileColidx_B)
{
    int n = tilerowPtr_A.size();
    int m = tilerowPtr_B.size();
    vector<vector<int>> C_prime;
    vector<int> rowIdx, colIdx;

    for (int i = 0; i < n - 1; ++i)
    {
        set<int> unique_col;
        for (int j = tilerowPtr_A[i]; j < tilerowPtr_A[i + 1]; ++j)
        {
            int col = tileColidx_A[j];
            for (int i1 = tilerowPtr_B[col]; i1 < tilerowPtr_B[col + 1]; ++i1)
            {
                int col1 = tileColidx_B[i1];
                unique_col.insert(col1);
            }
        }
        for (int col : unique_col)
        {
            colIdx.push_back(col);
            rowIdx.push_back(i);
        }
    }
    C_prime.push_back(rowIdx);
    C_prime.push_back(colIdx);
    return make_tuple(rowIdx, colIdx);
}

// CUDA kernel for bitmask conversion

__global__ void bitmask_conversion_kernel(int *matrix_B, int *matrix_mask_B, int num_elements)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < num_elements)
    {
        matrix_mask_B[tid] = (matrix_B[tid] != 0) ? 1 : 0;
    }
}

// CUDA kernel for Step 2 and Step 3

__global__ void step2_and_step3_kernel(int *matrix_A, int *matrix_B, int *tileRowidx_C, int *tileColidx_C, int *tilePtr_A,
                                      int *tilePtr_B, int *tileColPtr_B, int *tileColidx_A,
                                      int *tileRowidx_B, int numtileC, int tile_size, int *matrix_C, int *maskc,
                                      int *matrix_mask_B, int num_rows, int num_cols)
{

    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < numtileC)
    {

        int tile_i = tileRowidx_C[tid];
        int tile_j = tileColidx_C[tid];
        int lena = tilePtr_A[tile_i + 1] - tilePtr_A[tile_i];
        int lenb = tileColPtr_B[tile_j + 1] - tileColPtr_B[tile_j];

        // Array to store matched positions
        int matched_posA[MAX_MATCHED_POSITIONS];
        int num_matched = 0;

        // Applying Binary Search to find the matched positions
        if (lena <= lenb)
        {
            for (int value = tilePtr_A[tile_i]; value < tilePtr_A[tile_i + 1]; value++)
            {
                int low = tileColPtr_B[tile_j];
                int high = tileColPtr_B[tile_j + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileRowidx_B[mid] == tileColidx_A[value])
                    {
                        matched_posA[num_matched++] = tileColidx_A[value];
                        break;
                    }
                    else if (tileRowidx_B[mid] < tileColidx_A[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }
        else
        {
            for (int value = tileColPtr_B[tile_j]; value < tileColPtr_B[tile_j + 1]; value++)
            {
                int low = tilePtr_A[tile_i];
                int high = tilePtr_A[tile_i + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileColidx_A[mid] == tileRowidx_B[value])
                    {
                        matched_posA[num_matched++] = tileRowidx_B[value];
                        break;
                    }
                    else if (tileColidx_A[mid] < tileRowidx_B[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }

        for (int matched = 0; matched < num_matched; matched++)
        {
            int j = matched_posA[matched];
            for (int r = 0; r < tile_size; r++)
            {
                for (int s = 0; s < tile_size; s++)
                {
                    int row = tile_size * tile_i + r;
                    int col = tile_size * j + s;
                    if (matrix_A[row * num_cols + col] != 0)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            atomicOr(&maskc[row * num_cols + (tile_size * tile_j + t)], matrix_mask_B[(tile_size * j + s) * num_cols + (tile_size * tile_j + t)]);
                        }
                    }
                }
            }
        }
        int nnz = 0;
        for (int i = 0; i < tile_size; i++)
        {
            for (int j = 0; j < tile_size; j++)
            {
                // printf("%d",maskc[(tile_i * tile_size + i) * num_cols + (tile_j * tile_size + j)]);
                if (maskc[(tile_i * tile_size + i) * num_cols + (tile_j * tile_size + j)] != 0)
                {
                    nnz++;
                }
            }
        }

        if (nnz >= 12)
        {
            for (int matched = 0; matched < num_matched; matched++)
            {
                int j = matched_posA[matched];
                for (int r = 0; r < tile_size; r++)
                {
                    for (int s = 0; s < tile_size; s++)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            matrix_C[(tile_i * tile_size + r) * num_cols + (tile_size * tile_j + s)] += matrix_A[(tile_size * tile_i + r) * num_cols + tile_size * j + t] * matrix_B[(tile_size * j + t) * num_cols + (tile_size * tile_j + s)];
                        }
                    }
                }
            }
        }
        else
        {
            for (int matched = 0; matched < num_matched; matched++)
            {
                int matched_tiles = matched_posA[matched];
                for (int r = 0; r < tile_size; ++r)
                {
                    for (int s = 0; s < tile_size; ++s)
                    {
                        if(matrix_A[(tile_i*tile_size + r)*num_cols + matched_tiles*tile_size +s] == 0){
                            continue;
                        }

                        for (int t = 0; t < tile_size; ++t)
                        {

  matrix_C[(tile_i*tile_size + r)*num_cols + (tile_j*tile_size + t)] += matrix_A[(tile_i*tile_size + r)*num_cols + matched_tiles*tile_size +s] * matrix_B[(tile_size*matched_tiles + s )*num_cols + tile_j*tile_size + t];

                        }
                    }
                }
            }
        }
    }
}
/*
tuple<vector<int>, vector<int>> highLevel_conversion(vector<vector<int>> &matrix, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix.size(); i += tile_size)
    {
        for (int j = 0; j < matrix[i].size(); j += tile_size)
        {
            for (int k = 0; k < tile_size * tile_size; k++)
            {
                int row = k / tile_size;
                int col = k % tile_size;
                if (matrix[i + row][j + col] != 0)
                {
                    colIdx.push_back(j / tile_size);
                    break;
                }
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}
*/


tuple<vector<int>, vector<int>> highLevel_conversion(vector<int> &matrix, int matrix_size, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix_size; i += tile_size)
    {
        for (int j = 0; j < matrix_size; j += tile_size)
        {
            bool hasNonZero = false;
            for (int x = 0; x < tile_size; ++x)
            {
                for (int y = 0; y < tile_size; ++y)
                {
                    int row = i + x;
                    int col = j + y;
                    if (row < matrix_size && col < matrix_size && matrix[row * matrix_size + col] != 0)
                    {
                        colIdx.push_back(j / tile_size);
                        hasNonZero = true;
                        break;
                    }
                }
                if (hasNonZero)
                    break;
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}


tuple<vector<int>, vector<int>> highLevel_conversion_For_MatrixB(vector<int> rowPtr, vector<int> Colidx)
{
    vector<int> colPtr(rowPtr.size(), 0);
    vector<int> rowIdx(Colidx.size(), 0);
    int max = 0;

    for (int i = 0; i < rowPtr.size() - 1; i++)
    {
        for (int j = rowPtr[i]; j < rowPtr[i + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < colPtr.size(); k++)
            {
                colPtr[k]++;
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < max; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = i;
            max++;
        }
    }
    for (int i = 0; i < colPtr.size() - 1; i++)
    {
        sort(rowIdx.begin() + colPtr[i], rowIdx.begin() + colPtr[i + 1]);
    }
    return make_tuple(colPtr, rowIdx);
}
int main() {
    // Parameters
    vector<int> tile_sizes = {32};
    vector<int> matrix_sizes = {128, 256, 384, 512, 640, 768, 896, 1024, 1152, 1280, 1408, 1536, 1664, 1792, 1920, 2048, 2176, 2304, 2432, 2560, 2688, 2816, 2944, 3072, 10000};
    vector<double>time;

    double density = 0.1;
    for (int tile_size : tile_sizes) {
        cout << "Tile Size: " << tile_size << endl;
        for (int matrix_size : matrix_sizes) {
            cout << "Matrix Size: " << matrix_size << "x" << matrix_size << endl;
            vector<int> matrix_A = generateRandomSparseMatrix(matrix_size, matrix_size, density);
            vector<int> matrix_B = generateRandomSparseMatrix(matrix_size, matrix_size, density);

            struct timeval begin, end;
            gettimeofday(&begin, 0);

            // High-level conversion for matrix A
            vector<int> tilerowPtr_A, tileColidx_A;
            tie(tilerowPtr_A, tileColidx_A) = highLevel_conversion(matrix_A, matrix_size, tile_size);

            // High-level conversion for matrix B
            vector<int> tilerowPtr_B, tileColidx_B;
            tie(tilerowPtr_B, tileColidx_B) = highLevel_conversion(matrix_B, matrix_size, tile_size);

            // Symbolic SpGEMM for CSR Format
            vector<int> tileRowidx_C, tileColidx_C;
            tie(tileRowidx_C, tileColidx_C) = symbolic_SpGEMM_For_CSRFormat(tilerowPtr_A, tileColidx_A, tilerowPtr_B, tileColidx_B);

            int numtileC = tileRowidx_C.size();
            vector<int> tileColPtr_B, tileRowidx_B;
            tie(tileColPtr_B, tileRowidx_B) = highLevel_conversion_For_MatrixB(tilerowPtr_B, tileColidx_B);




             // Allocate memory for matrices and vectors on device
              int *d_matrix_A, *d_matrix_B, *d_tileRowidx_C, *d_tileColidx_C, *d_tilePtr_A,
                  *d_tilePtr_B, *d_tileColPtr_B, *d_tileColidx_A,
                  *d_tileRowidx_B, *d_matrix_C, *d_maskc, *d_matrix_mask_B;

              // CUDA memory allocation
              cudaMalloc((void **)&d_matrix_A, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_matrix_B, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_tileRowidx_C, sizeof(int) * tileRowidx_C.size());
              cudaMalloc((void **)&d_tileColidx_C, sizeof(int) * tileColidx_C.size());
              cudaMalloc((void **)&d_tilePtr_A, sizeof(int) * tilerowPtr_A.size());
              cudaMalloc((void **)&d_tilePtr_B, sizeof(int) * tilerowPtr_B.size());
              cudaMalloc((void **)&d_tileColPtr_B, sizeof(int) * tileColPtr_B.size());
              cudaMalloc((void **)&d_tileColidx_A, sizeof(int) * tileColidx_A.size());
              cudaMalloc((void **)&d_tileRowidx_B, sizeof(int) * tileRowidx_B.size());
              cudaMalloc((void **)&d_matrix_C, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_maskc, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_matrix_mask_B, sizeof(int) * matrix_size * matrix_size);

              // Copy data from host to device
              cudaMemcpy(d_matrix_A, matrix_A.data(), sizeof(int) * matrix_size * matrix_size, cudaMemcpyHostToDevice);
              cudaMemcpy(d_matrix_B, matrix_B.data(), sizeof(int) * matrix_size * matrix_size, cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileRowidx_C, tileRowidx_C.data(), sizeof(int) * tileRowidx_C.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColidx_C, tileColidx_C.data(), sizeof(int) * tileColidx_C.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tilePtr_A, tilerowPtr_A.data(), sizeof(int) * tilerowPtr_A.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tilePtr_B, tilerowPtr_B.data(), sizeof(int) * tilerowPtr_B.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColPtr_B, tileColPtr_B.data(), sizeof(int) * tileColPtr_B.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColidx_A, tileColidx_A.data(), sizeof(int) * tileColidx_A.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileRowidx_B, tileRowidx_B.data(), sizeof(int) * tileRowidx_B.size(), cudaMemcpyHostToDevice);

              // Define grid and block dimensions
              int numThreads = 256;
              int numBlocks = (numtileC + numThreads - 1) / numThreads;

              // Launch bitmask conversion kernel
              bitmask_conversion_kernel<<<numBlocks, numThreads>>>(d_matrix_B, d_matrix_mask_B, matrix_size * matrix_size);
              cudaDeviceSynchronize();
              int *mask_B_matrix = new int[matrix_size * matrix_size];
              cudaMemcpy(mask_B_matrix, d_matrix_mask_B, sizeof(int) * matrix_size * matrix_size, cudaMemcpyDeviceToHost);

              // Launch Step 2 and Step 3 kernel
              step2_and_step3_kernel<<<numBlocks, numThreads>>>(d_matrix_A, d_matrix_B, d_tileRowidx_C, d_tileColidx_C, d_tilePtr_A,
                                                                d_tilePtr_B, d_tileColPtr_B, d_tileColidx_A,
                                                                d_tileRowidx_B, numtileC, tile_size, d_matrix_C, d_maskc, d_matrix_mask_B, matrix_size, matrix_size);
              cudaDeviceSynchronize();

              gettimeofday(&end, 0);
            long seconds = end.tv_sec - begin.tv_sec;
            long microseconds = end.tv_usec - begin.tv_usec;
            double elapsed = seconds + microseconds*1e-6;
            time.push_back(elapsed);
            printf("Time measured: %.6f seconds.\n", elapsed);

              int *matrix_C_result = new int[matrix_size * matrix_size];
              cudaMemcpy(matrix_C_result, d_matrix_C, sizeof(int) * matrix_size * matrix_size, cudaMemcpyDeviceToHost);
            // Free host memory
            delete[] matrix_C_result;

            // Copy result matrix from device to host
            // Free device memory
            cudaFree(d_matrix_A);
            cudaFree(d_matrix_B);
            cudaFree(d_tileRowidx_C);
            cudaFree(d_tileColidx_C);
            cudaFree(d_tilePtr_A);
            cudaFree(d_tilePtr_B);
            cudaFree(d_tileColPtr_B);
            cudaFree(d_tileColidx_A);
            cudaFree(d_tileRowidx_B);
            cudaFree(d_matrix_C);
            cudaFree(d_maskc);
            cudaFree(d_matrix_mask_B);
            cout << "hello" << endl;
        }


    }
    return 0;
}

Tile Size: 32
Matrix Size: 128x128
Time measured: 0.216782 seconds.
hello
Matrix Size: 256x256
Time measured: 0.022222 seconds.
hello
Matrix Size: 384x384
Time measured: 0.015872 seconds.
hello
Matrix Size: 512x512
Time measured: 0.021370 seconds.
hello
Matrix Size: 640x640
Time measured: 0.023924 seconds.
hello
Matrix Size: 768x768
Time measured: 0.034323 seconds.
hello
Matrix Size: 896x896
Time measured: 0.039579 seconds.
hello
Matrix Size: 1024x1024
Time measured: 0.033411 seconds.
hello
Matrix Size: 1152x1152
Time measured: 0.047916 seconds.
hello
Matrix Size: 1280x1280
Time measured: 0.046588 seconds.
hello
Matrix Size: 1408x1408
Time measured: 0.062996 seconds.
hello
Matrix Size: 1536x1536
Time measured: 0.085326 seconds.
hello
Matrix Size: 1664x1664
Time measured: 0.101356 seconds.
hello
Matrix Size: 1792x1792
Time measured: 0.122329 seconds.
hello
Matrix Size: 1920x1920
Time measured: 0.120297 seconds.
hello
Matrix Size: 2048x2048
Time measured: 0.133956 seconds.
hello
Matrix S

# Tile 16

In [ ]:
%%cuda
#include <random>
#include<bits/stdc++.h>
#include <algorithm>
#include <vector>
#include <iostream>
#include <sys/time.h>
#include <cuda_runtime.h>
#define MAX_MATCHED_POSITIONS 16

using namespace std;

vector<int> generateRandomSparseMatrix(int rows, int cols, double density) {
    vector<int> matrix(rows*cols);
    std::random_device rd;
    std::mt19937 gen(rd());
    std::uniform_real_distribution<> dis(0.0, 1.0);
    for (int i = 0; i < rows; ++i) {
        for (int j = 0; j < cols; ++j) {
            double randNum = dis(gen);
            if (randNum <= density) {
                matrix[i*rows+j] = 1;
            }
        }
    }
    return matrix;
}

tuple<vector<int>, vector<int>> symbolic_SpGEMM_For_CSRFormat(vector<int> tilerowPtr_A, vector<int> tileColidx_A, vector<int> tilerowPtr_B, vector<int> tileColidx_B)
{
    int n = tilerowPtr_A.size();
    int m = tilerowPtr_B.size();
    vector<vector<int>> C_prime;
    vector<int> rowIdx, colIdx;

    for (int i = 0; i < n - 1; ++i)
    {
        set<int> unique_col;
        for (int j = tilerowPtr_A[i]; j < tilerowPtr_A[i + 1]; ++j)
        {
            int col = tileColidx_A[j];
            for (int i1 = tilerowPtr_B[col]; i1 < tilerowPtr_B[col + 1]; ++i1)
            {
                int col1 = tileColidx_B[i1];
                unique_col.insert(col1);
            }
        }
        for (int col : unique_col)
        {
            colIdx.push_back(col);
            rowIdx.push_back(i);
        }
    }
    C_prime.push_back(rowIdx);
    C_prime.push_back(colIdx);
    return make_tuple(rowIdx, colIdx);
}

// CUDA kernel for bitmask conversion

__global__ void bitmask_conversion_kernel(int *matrix_B, int *matrix_mask_B, int num_elements)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < num_elements)
    {
        matrix_mask_B[tid] = (matrix_B[tid] != 0) ? 1 : 0;
    }
}

// CUDA kernel for Step 2 and Step 3

__global__ void step2_and_step3_kernel(int *matrix_A, int *matrix_B, int *tileRowidx_C, int *tileColidx_C, int *tilePtr_A,
                                      int *tilePtr_B, int *tileColPtr_B, int *tileColidx_A,
                                      int *tileRowidx_B, int numtileC, int tile_size, int *matrix_C, int *maskc,
                                      int *matrix_mask_B, int num_rows, int num_cols)
{

    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < numtileC)
    {

        int tile_i = tileRowidx_C[tid];
        int tile_j = tileColidx_C[tid];
        int lena = tilePtr_A[tile_i + 1] - tilePtr_A[tile_i];
        int lenb = tileColPtr_B[tile_j + 1] - tileColPtr_B[tile_j];

        // Array to store matched positions
        int matched_posA[MAX_MATCHED_POSITIONS];
        int num_matched = 0;

        // Applying Binary Search to find the matched positions
        if (lena <= lenb)
        {
            for (int value = tilePtr_A[tile_i]; value < tilePtr_A[tile_i + 1]; value++)
            {
                int low = tileColPtr_B[tile_j];
                int high = tileColPtr_B[tile_j + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileRowidx_B[mid] == tileColidx_A[value])
                    {
                        matched_posA[num_matched++] = tileColidx_A[value];
                        break;
                    }
                    else if (tileRowidx_B[mid] < tileColidx_A[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }
        else
        {
            for (int value = tileColPtr_B[tile_j]; value < tileColPtr_B[tile_j + 1]; value++)
            {
                int low = tilePtr_A[tile_i];
                int high = tilePtr_A[tile_i + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileColidx_A[mid] == tileRowidx_B[value])
                    {
                        matched_posA[num_matched++] = tileRowidx_B[value];
                        break;
                    }
                    else if (tileColidx_A[mid] < tileRowidx_B[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }

        for (int matched = 0; matched < num_matched; matched++)
        {
            int j = matched_posA[matched];
            for (int r = 0; r < tile_size; r++)
            {
                for (int s = 0; s < tile_size; s++)
                {
                    int row = tile_size * tile_i + r;
                    int col = tile_size * j + s;
                    if (matrix_A[row * num_cols + col] != 0)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            atomicOr(&maskc[row * num_cols + (tile_size * tile_j + t)], matrix_mask_B[(tile_size * j + s) * num_cols + (tile_size * tile_j + t)]);
                        }
                    }
                }
            }
        }
        int nnz = 0;
        for (int i = 0; i < tile_size; i++)
        {
            for (int j = 0; j < tile_size; j++)
            {
                // printf("%d",maskc[(tile_i * tile_size + i) * num_cols + (tile_j * tile_size + j)]);
                if (maskc[(tile_i * tile_size + i) * num_cols + (tile_j * tile_size + j)] != 0)
                {
                    nnz++;
                }
            }
        }

        if (nnz >= 12)
        {
            for (int matched = 0; matched < num_matched; matched++)
            {
                int j = matched_posA[matched];
                for (int r = 0; r < tile_size; r++)
                {
                    for (int s = 0; s < tile_size; s++)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            matrix_C[(tile_i * tile_size + r) * num_cols + (tile_size * tile_j + s)] += matrix_A[(tile_size * tile_i + r) * num_cols + tile_size * j + t] * matrix_B[(tile_size * j + t) * num_cols + (tile_size * tile_j + s)];
                        }
                    }
                }
            }
        }
        else
        {
            for (int matched = 0; matched < num_matched; matched++)
            {
                int matched_tiles = matched_posA[matched];
                for (int r = 0; r < tile_size; ++r)
                {
                    for (int s = 0; s < tile_size; ++s)
                    {
                        if(matrix_A[(tile_i*tile_size + r)*num_cols + matched_tiles*tile_size +s] == 0){
                            continue;
                        }

                        for (int t = 0; t < tile_size; ++t)
                        {

  matrix_C[(tile_i*tile_size + r)*num_cols + (tile_j*tile_size + t)] += matrix_A[(tile_i*tile_size + r)*num_cols + matched_tiles*tile_size +s] * matrix_B[(tile_size*matched_tiles + s )*num_cols + tile_j*tile_size + t];

                        }
                    }
                }
            }
        }
    }
}
/*
tuple<vector<int>, vector<int>> highLevel_conversion(vector<vector<int>> &matrix, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix.size(); i += tile_size)
    {
        for (int j = 0; j < matrix[i].size(); j += tile_size)
        {
            for (int k = 0; k < tile_size * tile_size; k++)
            {
                int row = k / tile_size;
                int col = k % tile_size;
                if (matrix[i + row][j + col] != 0)
                {
                    colIdx.push_back(j / tile_size);
                    break;
                }
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}
*/


tuple<vector<int>, vector<int>> highLevel_conversion(vector<int> &matrix, int matrix_size, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix_size; i += tile_size)
    {
        for (int j = 0; j < matrix_size; j += tile_size)
        {
            bool hasNonZero = false;
            for (int x = 0; x < tile_size; ++x)
            {
                for (int y = 0; y < tile_size; ++y)
                {
                    int row = i + x;
                    int col = j + y;
                    if (row < matrix_size && col < matrix_size && matrix[row * matrix_size + col] != 0)
                    {
                        colIdx.push_back(j / tile_size);
                        hasNonZero = true;
                        break;
                    }
                }
                if (hasNonZero)
                    break;
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}


tuple<vector<int>, vector<int>> highLevel_conversion_For_MatrixB(vector<int> rowPtr, vector<int> Colidx)
{
    vector<int> colPtr(rowPtr.size(), 0);
    vector<int> rowIdx(Colidx.size(), 0);
    int max = 0;

    for (int i = 0; i < rowPtr.size() - 1; i++)
    {
        for (int j = rowPtr[i]; j < rowPtr[i + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < colPtr.size(); k++)
            {
                colPtr[k]++;
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < max; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = i;
            max++;
        }
    }
    for (int i = 0; i < colPtr.size() - 1; i++)
    {
        sort(rowIdx.begin() + colPtr[i], rowIdx.begin() + colPtr[i + 1]);
    }
    return make_tuple(colPtr, rowIdx);
}
int main() {
    // Parameters
    vector<int> tile_sizes = {16};
    vector<int> matrix_sizes = {128, 256, 384, 512, 640, 768, 896, 1024, 1152, 1280, 1408, 1536, 1664, 1792, 1920, 2048, 2176, 2304, 2432, 2560, 2688, 2816, 2944, 3072, 10000};
    vector<double>time;

    double density = 0.1;
    for (int tile_size : tile_sizes) {
        cout << "Tile Size: " << tile_size << endl;
        for (int matrix_size : matrix_sizes) {
            cout << "Matrix Size: " << matrix_size << "x" << matrix_size << endl;
            vector<int> matrix_A = generateRandomSparseMatrix(matrix_size, matrix_size, density);
            vector<int> matrix_B = generateRandomSparseMatrix(matrix_size, matrix_size, density);

            struct timeval begin, end;
            gettimeofday(&begin, 0);

            // High-level conversion for matrix A
            vector<int> tilerowPtr_A, tileColidx_A;
            tie(tilerowPtr_A, tileColidx_A) = highLevel_conversion(matrix_A, matrix_size, tile_size);

            // High-level conversion for matrix B
            vector<int> tilerowPtr_B, tileColidx_B;
            tie(tilerowPtr_B, tileColidx_B) = highLevel_conversion(matrix_B, matrix_size, tile_size);

            // Symbolic SpGEMM for CSR Format
            vector<int> tileRowidx_C, tileColidx_C;
            tie(tileRowidx_C, tileColidx_C) = symbolic_SpGEMM_For_CSRFormat(tilerowPtr_A, tileColidx_A, tilerowPtr_B, tileColidx_B);

            int numtileC = tileRowidx_C.size();
            vector<int> tileColPtr_B, tileRowidx_B;
            tie(tileColPtr_B, tileRowidx_B) = highLevel_conversion_For_MatrixB(tilerowPtr_B, tileColidx_B);




             // Allocate memory for matrices and vectors on device
              int *d_matrix_A, *d_matrix_B, *d_tileRowidx_C, *d_tileColidx_C, *d_tilePtr_A,
                  *d_tilePtr_B, *d_tileColPtr_B, *d_tileColidx_A,
                  *d_tileRowidx_B, *d_matrix_C, *d_maskc, *d_matrix_mask_B;

              // CUDA memory allocation
              cudaMalloc((void **)&d_matrix_A, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_matrix_B, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_tileRowidx_C, sizeof(int) * tileRowidx_C.size());
              cudaMalloc((void **)&d_tileColidx_C, sizeof(int) * tileColidx_C.size());
              cudaMalloc((void **)&d_tilePtr_A, sizeof(int) * tilerowPtr_A.size());
              cudaMalloc((void **)&d_tilePtr_B, sizeof(int) * tilerowPtr_B.size());
              cudaMalloc((void **)&d_tileColPtr_B, sizeof(int) * tileColPtr_B.size());
              cudaMalloc((void **)&d_tileColidx_A, sizeof(int) * tileColidx_A.size());
              cudaMalloc((void **)&d_tileRowidx_B, sizeof(int) * tileRowidx_B.size());
              cudaMalloc((void **)&d_matrix_C, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_maskc, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_matrix_mask_B, sizeof(int) * matrix_size * matrix_size);

              // Copy data from host to device
              cudaMemcpy(d_matrix_A, matrix_A.data(), sizeof(int) * matrix_size * matrix_size, cudaMemcpyHostToDevice);
              cudaMemcpy(d_matrix_B, matrix_B.data(), sizeof(int) * matrix_size * matrix_size, cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileRowidx_C, tileRowidx_C.data(), sizeof(int) * tileRowidx_C.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColidx_C, tileColidx_C.data(), sizeof(int) * tileColidx_C.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tilePtr_A, tilerowPtr_A.data(), sizeof(int) * tilerowPtr_A.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tilePtr_B, tilerowPtr_B.data(), sizeof(int) * tilerowPtr_B.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColPtr_B, tileColPtr_B.data(), sizeof(int) * tileColPtr_B.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColidx_A, tileColidx_A.data(), sizeof(int) * tileColidx_A.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileRowidx_B, tileRowidx_B.data(), sizeof(int) * tileRowidx_B.size(), cudaMemcpyHostToDevice);

              // Define grid and block dimensions
              int numThreads = 256;
              int numBlocks = (numtileC + numThreads - 1) / numThreads;

              // Launch bitmask conversion kernel
              bitmask_conversion_kernel<<<numBlocks, numThreads>>>(d_matrix_B, d_matrix_mask_B, matrix_size * matrix_size);
              cudaDeviceSynchronize();
              int *mask_B_matrix = new int[matrix_size * matrix_size];
              cudaMemcpy(mask_B_matrix, d_matrix_mask_B, sizeof(int) * matrix_size * matrix_size, cudaMemcpyDeviceToHost);

              // Launch Step 2 and Step 3 kernel
              step2_and_step3_kernel<<<numBlocks, numThreads>>>(d_matrix_A, d_matrix_B, d_tileRowidx_C, d_tileColidx_C, d_tilePtr_A,
                                                                d_tilePtr_B, d_tileColPtr_B, d_tileColidx_A,
                                                                d_tileRowidx_B, numtileC, tile_size, d_matrix_C, d_maskc, d_matrix_mask_B, matrix_size, matrix_size);
              cudaDeviceSynchronize();

              gettimeofday(&end, 0);
            long seconds = end.tv_sec - begin.tv_sec;
            long microseconds = end.tv_usec - begin.tv_usec;
            double elapsed = seconds + microseconds*1e-6;
            time.push_back(elapsed);
            printf("Time measured: %.6f seconds.\n", elapsed);

              int *matrix_C_result = new int[matrix_size * matrix_size];
              cudaMemcpy(matrix_C_result, d_matrix_C, sizeof(int) * matrix_size * matrix_size, cudaMemcpyDeviceToHost);

            // Free host memory
            delete[] matrix_C_result;

            // Copy result matrix from device to host
            // Free device memory
            cudaFree(d_matrix_A);
            cudaFree(d_matrix_B);
            cudaFree(d_tileRowidx_C);
            cudaFree(d_tileColidx_C);
            cudaFree(d_tilePtr_A);
            cudaFree(d_tilePtr_B);
            cudaFree(d_tileColPtr_B);
            cudaFree(d_tileColidx_A);
            cudaFree(d_tileRowidx_B);
            cudaFree(d_matrix_C);
            cudaFree(d_maskc);
            cudaFree(d_matrix_mask_B);
            cout << "hello" << endl;
        }


    }
    return 0;
}

Tile Size: 16
Matrix Size: 128x128
Time measured: 0.212693 seconds.
hello
Matrix Size: 256x256
Time measured: 0.005298 seconds.
hello
Matrix Size: 384x384
Time measured: 0.009267 seconds.
hello
Matrix Size: 512x512
Time measured: 0.016020 seconds.
hello
Matrix Size: 640x640
Time measured: 0.030048 seconds.
hello
Matrix Size: 768x768
Time measured: 0.052976 seconds.
hello
Matrix Size: 896x896
Time measured: 0.085661 seconds.
hello
Matrix Size: 1024x1024
Time measured: 0.131731 seconds.
hello
Matrix Size: 1152x1152
Time measured: 0.198508 seconds.
hello
Matrix Size: 1280x1280
Time measured: 0.235554 seconds.
hello
Matrix Size: 1408x1408
Time measured: 0.264777 seconds.
hello
Matrix Size: 1536x1536
Time measured: 0.357402 seconds.
hello
Matrix Size: 1664x1664
Time measured: 0.458635 seconds.
hello
Matrix Size: 1792x1792
Time measured: 0.603061 seconds.
hello
Matrix Size: 1920x1920
Time measured: 0.773268 seconds.
hello
Matrix Size: 2048x2048
Time measured: 0.976421 seconds.
hello
Matrix S

# Tile 8

In [ ]:
%%cuda
#include <random>
#include<bits/stdc++.h>
#include <algorithm>
#include <vector>
#include <iostream>
#include <sys/time.h>
#include <cuda_runtime.h>
#define MAX_MATCHED_POSITIONS 16

using namespace std;

vector<int> generateRandomSparseMatrix(int rows, int cols, double density) {
    vector<int> matrix(rows*cols);
    std::random_device rd;
    std::mt19937 gen(rd());
    std::uniform_real_distribution<> dis(0.0, 1.0);
    for (int i = 0; i < rows; ++i) {
        for (int j = 0; j < cols; ++j) {
            double randNum = dis(gen);
            if (randNum <= density) {
                matrix[i*rows+j] = 1;
            }
        }
    }
    return matrix;
}

tuple<vector<int>, vector<int>> symbolic_SpGEMM_For_CSRFormat(vector<int> tilerowPtr_A, vector<int> tileColidx_A, vector<int> tilerowPtr_B, vector<int> tileColidx_B)
{
    int n = tilerowPtr_A.size();
    int m = tilerowPtr_B.size();
    vector<vector<int>> C_prime;
    vector<int> rowIdx, colIdx;

    for (int i = 0; i < n - 1; ++i)
    {
        set<int> unique_col;
        for (int j = tilerowPtr_A[i]; j < tilerowPtr_A[i + 1]; ++j)
        {
            int col = tileColidx_A[j];
            for (int i1 = tilerowPtr_B[col]; i1 < tilerowPtr_B[col + 1]; ++i1)
            {
                int col1 = tileColidx_B[i1];
                unique_col.insert(col1);
            }
        }
        for (int col : unique_col)
        {
            colIdx.push_back(col);
            rowIdx.push_back(i);
        }
    }
    C_prime.push_back(rowIdx);
    C_prime.push_back(colIdx);
    return make_tuple(rowIdx, colIdx);
}

// CUDA kernel for bitmask conversion

__global__ void bitmask_conversion_kernel(int *matrix_B, int *matrix_mask_B, int num_elements)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < num_elements)
    {
        matrix_mask_B[tid] = (matrix_B[tid] != 0) ? 1 : 0;
    }
}

// CUDA kernel for Step 2 and Step 3

__global__ void step2_and_step3_kernel(int *matrix_A, int *matrix_B, int *tileRowidx_C, int *tileColidx_C, int *tilePtr_A,
                                      int *tilePtr_B, int *tileColPtr_B, int *tileColidx_A,
                                      int *tileRowidx_B, int numtileC, int tile_size, int *matrix_C, int *maskc,
                                      int *matrix_mask_B, int num_rows, int num_cols)
{

    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < numtileC)
    {

        int tile_i = tileRowidx_C[tid];
        int tile_j = tileColidx_C[tid];
        int lena = tilePtr_A[tile_i + 1] - tilePtr_A[tile_i];
        int lenb = tileColPtr_B[tile_j + 1] - tileColPtr_B[tile_j];

        // Array to store matched positions
        int matched_posA[MAX_MATCHED_POSITIONS];
        int num_matched = 0;

        // Applying Binary Search to find the matched positions
        if (lena <= lenb)
        {
            for (int value = tilePtr_A[tile_i]; value < tilePtr_A[tile_i + 1]; value++)
            {
                int low = tileColPtr_B[tile_j];
                int high = tileColPtr_B[tile_j + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileRowidx_B[mid] == tileColidx_A[value])
                    {
                        matched_posA[num_matched++] = tileColidx_A[value];
                        break;
                    }
                    else if (tileRowidx_B[mid] < tileColidx_A[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }
        else
        {
            for (int value = tileColPtr_B[tile_j]; value < tileColPtr_B[tile_j + 1]; value++)
            {
                int low = tilePtr_A[tile_i];
                int high = tilePtr_A[tile_i + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileColidx_A[mid] == tileRowidx_B[value])
                    {
                        matched_posA[num_matched++] = tileRowidx_B[value];
                        break;
                    }
                    else if (tileColidx_A[mid] < tileRowidx_B[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }

        for (int matched = 0; matched < num_matched; matched++)
        {
            int j = matched_posA[matched];
            for (int r = 0; r < tile_size; r++)
            {
                for (int s = 0; s < tile_size; s++)
                {
                    int row = tile_size * tile_i + r;
                    int col = tile_size * j + s;
                    if (matrix_A[row * num_cols + col] != 0)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            atomicOr(&maskc[row * num_cols + (tile_size * tile_j + t)], matrix_mask_B[(tile_size * j + s) * num_cols + (tile_size * tile_j + t)]);
                        }
                    }
                }
            }
        }
        int nnz = 0;
        for (int i = 0; i < tile_size; i++)
        {
            for (int j = 0; j < tile_size; j++)
            {
                // printf("%d",maskc[(tile_i * tile_size + i) * num_cols + (tile_j * tile_size + j)]);
                if (maskc[(tile_i * tile_size + i) * num_cols + (tile_j * tile_size + j)] != 0)
                {
                    nnz++;
                }
            }
        }

        if (nnz >= 12)
        {
            for (int matched = 0; matched < num_matched; matched++)
            {
                int j = matched_posA[matched];
                for (int r = 0; r < tile_size; r++)
                {
                    for (int s = 0; s < tile_size; s++)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            matrix_C[(tile_i * tile_size + r) * num_cols + (tile_size * tile_j + s)] += matrix_A[(tile_size * tile_i + r) * num_cols + tile_size * j + t] * matrix_B[(tile_size * j + t) * num_cols + (tile_size * tile_j + s)];
                        }
                    }
                }
            }
        }
        else
        {
            for (int matched = 0; matched < num_matched; matched++)
            {
                int matched_tiles = matched_posA[matched];
                for (int r = 0; r < tile_size; ++r)
                {
                    for (int s = 0; s < tile_size; ++s)
                    {
                        if(matrix_A[(tile_i*tile_size + r)*num_cols + matched_tiles*tile_size +s] == 0){
                            continue;
                        }

                        for (int t = 0; t < tile_size; ++t)
                        {

  matrix_C[(tile_i*tile_size + r)*num_cols + (tile_j*tile_size + t)] += matrix_A[(tile_i*tile_size + r)*num_cols + matched_tiles*tile_size +s] * matrix_B[(tile_size*matched_tiles + s )*num_cols + tile_j*tile_size + t];

                        }
                    }
                }
            }
        }
    }
}
/*
tuple<vector<int>, vector<int>> highLevel_conversion(vector<vector<int>> &matrix, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix.size(); i += tile_size)
    {
        for (int j = 0; j < matrix[i].size(); j += tile_size)
        {
            for (int k = 0; k < tile_size * tile_size; k++)
            {
                int row = k / tile_size;
                int col = k % tile_size;
                if (matrix[i + row][j + col] != 0)
                {
                    colIdx.push_back(j / tile_size);
                    break;
                }
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}
*/


tuple<vector<int>, vector<int>> highLevel_conversion(vector<int> &matrix, int matrix_size, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix_size; i += tile_size)
    {
        for (int j = 0; j < matrix_size; j += tile_size)
        {
            bool hasNonZero = false;
            for (int x = 0; x < tile_size; ++x)
            {
                for (int y = 0; y < tile_size; ++y)
                {
                    int row = i + x;
                    int col = j + y;
                    if (row < matrix_size && col < matrix_size && matrix[row * matrix_size + col] != 0)
                    {
                        colIdx.push_back(j / tile_size);
                        hasNonZero = true;
                        break;
                    }
                }
                if (hasNonZero)
                    break;
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}


tuple<vector<int>, vector<int>> highLevel_conversion_For_MatrixB(vector<int> rowPtr, vector<int> Colidx)
{
    vector<int> colPtr(rowPtr.size(), 0);
    vector<int> rowIdx(Colidx.size(), 0);
    int max = 0;

    for (int i = 0; i < rowPtr.size() - 1; i++)
    {
        for (int j = rowPtr[i]; j < rowPtr[i + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < colPtr.size(); k++)
            {
                colPtr[k]++;
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < max; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = i;
            max++;
        }
    }
    for (int i = 0; i < colPtr.size() - 1; i++)
    {
        sort(rowIdx.begin() + colPtr[i], rowIdx.begin() + colPtr[i + 1]);
    }
    return make_tuple(colPtr, rowIdx);
}
int main() {
    // Parameters
    vector<int> tile_sizes = {8};
    vector<int> matrix_sizes = {128, 256, 384, 512, 640, 768, 896, 1024, 1152, 1280, 1408, 1536, 1664, 1792, 1920, 2048, 2176, 2304, 2432, 2560, 2688, 2816, 2944, 3072};
    vector<double>time;

    double density = 0.1;
    for (int tile_size : tile_sizes) {
        cout << "Tile Size: " << tile_size << endl;
        for (int matrix_size : matrix_sizes) {
            cout << "Matrix Size: " << matrix_size << "x" << matrix_size << endl;
            vector<int> matrix_A = generateRandomSparseMatrix(matrix_size, matrix_size, density);
            vector<int> matrix_B = generateRandomSparseMatrix(matrix_size, matrix_size, density);

            struct timeval begin, end;
            gettimeofday(&begin, 0);

            // High-level conversion for matrix A
            vector<int> tilerowPtr_A, tileColidx_A;
            tie(tilerowPtr_A, tileColidx_A) = highLevel_conversion(matrix_A, matrix_size, tile_size);

            // High-level conversion for matrix B
            vector<int> tilerowPtr_B, tileColidx_B;
            tie(tilerowPtr_B, tileColidx_B) = highLevel_conversion(matrix_B, matrix_size, tile_size);

            // Symbolic SpGEMM for CSR Format
            vector<int> tileRowidx_C, tileColidx_C;
            tie(tileRowidx_C, tileColidx_C) = symbolic_SpGEMM_For_CSRFormat(tilerowPtr_A, tileColidx_A, tilerowPtr_B, tileColidx_B);

            int numtileC = tileRowidx_C.size();
            vector<int> tileColPtr_B, tileRowidx_B;
            tie(tileColPtr_B, tileRowidx_B) = highLevel_conversion_For_MatrixB(tilerowPtr_B, tileColidx_B);




             // Allocate memory for matrices and vectors on device
              int *d_matrix_A, *d_matrix_B, *d_tileRowidx_C, *d_tileColidx_C, *d_tilePtr_A,
                  *d_tilePtr_B, *d_tileColPtr_B, *d_tileColidx_A,
                  *d_tileRowidx_B, *d_matrix_C, *d_maskc, *d_matrix_mask_B;

              // CUDA memory allocation
              cudaMalloc((void **)&d_matrix_A, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_matrix_B, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_tileRowidx_C, sizeof(int) * tileRowidx_C.size());
              cudaMalloc((void **)&d_tileColidx_C, sizeof(int) * tileColidx_C.size());
              cudaMalloc((void **)&d_tilePtr_A, sizeof(int) * tilerowPtr_A.size());
              cudaMalloc((void **)&d_tilePtr_B, sizeof(int) * tilerowPtr_B.size());
              cudaMalloc((void **)&d_tileColPtr_B, sizeof(int) * tileColPtr_B.size());
              cudaMalloc((void **)&d_tileColidx_A, sizeof(int) * tileColidx_A.size());
              cudaMalloc((void **)&d_tileRowidx_B, sizeof(int) * tileRowidx_B.size());
              cudaMalloc((void **)&d_matrix_C, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_maskc, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_matrix_mask_B, sizeof(int) * matrix_size * matrix_size);

              // Copy data from host to device
              cudaMemcpy(d_matrix_A, matrix_A.data(), sizeof(int) * matrix_size * matrix_size, cudaMemcpyHostToDevice);
              cudaMemcpy(d_matrix_B, matrix_B.data(), sizeof(int) * matrix_size * matrix_size, cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileRowidx_C, tileRowidx_C.data(), sizeof(int) * tileRowidx_C.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColidx_C, tileColidx_C.data(), sizeof(int) * tileColidx_C.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tilePtr_A, tilerowPtr_A.data(), sizeof(int) * tilerowPtr_A.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tilePtr_B, tilerowPtr_B.data(), sizeof(int) * tilerowPtr_B.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColPtr_B, tileColPtr_B.data(), sizeof(int) * tileColPtr_B.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColidx_A, tileColidx_A.data(), sizeof(int) * tileColidx_A.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileRowidx_B, tileRowidx_B.data(), sizeof(int) * tileRowidx_B.size(), cudaMemcpyHostToDevice);

              // Define grid and block dimensions
              int numThreads = 256;
              int numBlocks = (numtileC + numThreads - 1) / numThreads;

              // Launch bitmask conversion kernel
              bitmask_conversion_kernel<<<numBlocks, numThreads>>>(d_matrix_B, d_matrix_mask_B, matrix_size * matrix_size);
              cudaDeviceSynchronize();
              int *mask_B_matrix = new int[matrix_size * matrix_size];
              cudaMemcpy(mask_B_matrix, d_matrix_mask_B, sizeof(int) * matrix_size * matrix_size, cudaMemcpyDeviceToHost);

              // Launch Step 2 and Step 3 kernel
              step2_and_step3_kernel<<<numBlocks, numThreads>>>(d_matrix_A, d_matrix_B, d_tileRowidx_C, d_tileColidx_C, d_tilePtr_A,
                                                                d_tilePtr_B, d_tileColPtr_B, d_tileColidx_A,
                                                                d_tileRowidx_B, numtileC, tile_size, d_matrix_C, d_maskc, d_matrix_mask_B, matrix_size, matrix_size);
              cudaDeviceSynchronize();

              gettimeofday(&end, 0);
            long seconds = end.tv_sec - begin.tv_sec;
            long microseconds = end.tv_usec - begin.tv_usec;
            double elapsed = seconds + microseconds*1e-6;
            time.push_back(elapsed);
            printf("Time measured: %.6f seconds.\n", elapsed);

              int *matrix_C_result = new int[matrix_size * matrix_size];
              cudaMemcpy(matrix_C_result, d_matrix_C, sizeof(int) * matrix_size * matrix_size, cudaMemcpyDeviceToHost);

            // Free host memory
            delete[] matrix_C_result;

            // Copy result matrix from device to host
            // Free device memory
            cudaFree(d_matrix_A);
            cudaFree(d_matrix_B);
            cudaFree(d_tileRowidx_C);
            cudaFree(d_tileColidx_C);
            cudaFree(d_tilePtr_A);
            cudaFree(d_tilePtr_B);
            cudaFree(d_tileColPtr_B);
            cudaFree(d_tileColidx_A);
            cudaFree(d_tileRowidx_B);
            cudaFree(d_matrix_C);
            cudaFree(d_maskc);
            cudaFree(d_matrix_mask_B);
            cout << "hello" << endl;
        }


    }
    return 0;
}

Tile Size: 8
Matrix Size: 128x128
Time measured: 0.199112 seconds.
hello
Matrix Size: 256x256
Time measured: 0.009834 seconds.
hello
Matrix Size: 384x384
Time measured: 0.036622 seconds.
hello
Matrix Size: 512x512
Time measured: 0.091815 seconds.
hello
Matrix Size: 640x640
Time measured: 0.192753 seconds.
hello
Matrix Size: 768x768
Time measured: 0.344575 seconds.
hello
Matrix Size: 896x896
Time measured: 0.587618 seconds.
hello
Matrix Size: 1024x1024
Time measured: 0.932178 seconds.
hello
Matrix Size: 1152x1152
Time measured: 1.388752 seconds.
hello
Matrix Size: 1280x1280
Time measured: 2.067790 seconds.
hello
Matrix Size: 1408x1408
Time measured: 4.510172 seconds.
hello
Matrix Size: 1536x1536
Time measured: 4.086794 seconds.
hello
Matrix Size: 1664x1664
Time measured: 5.507133 seconds.
hello
Matrix Size: 1792x1792
Time measured: 6.654089 seconds.
hello
Matrix Size: 1920x1920
Time measured: 8.923974 seconds.
hello
Matrix Size: 2048x2048
Time measured: 11.043611 seconds.
hello
Matrix S

# Tile 4

In [ ]:
%%cuda
#include <random>
#include<bits/stdc++.h>
#include <algorithm>
#include <vector>
#include <iostream>
#include <sys/time.h>
#include <cuda_runtime.h>
#define MAX_MATCHED_POSITIONS 16

using namespace std;

vector<int> generateRandomSparseMatrix(int rows, int cols, double density) {
    vector<int> matrix(rows*cols);
    std::random_device rd;
    std::mt19937 gen(rd());
    std::uniform_real_distribution<> dis(0.0, 1.0);
    for (int i = 0; i < rows; ++i) {
        for (int j = 0; j < cols; ++j) {
            double randNum = dis(gen);
            if (randNum <= density) {
                matrix[i*rows+j] = 1;
            }
        }
    }
    return matrix;
}

tuple<vector<int>, vector<int>> symbolic_SpGEMM_For_CSRFormat(vector<int> tilerowPtr_A, vector<int> tileColidx_A, vector<int> tilerowPtr_B, vector<int> tileColidx_B)
{
    int n = tilerowPtr_A.size();
    int m = tilerowPtr_B.size();
    vector<vector<int>> C_prime;
    vector<int> rowIdx, colIdx;

    for (int i = 0; i < n - 1; ++i)
    {
        set<int> unique_col;
        for (int j = tilerowPtr_A[i]; j < tilerowPtr_A[i + 1]; ++j)
        {
            int col = tileColidx_A[j];
            for (int i1 = tilerowPtr_B[col]; i1 < tilerowPtr_B[col + 1]; ++i1)
            {
                int col1 = tileColidx_B[i1];
                unique_col.insert(col1);
            }
        }
        for (int col : unique_col)
        {
            colIdx.push_back(col);
            rowIdx.push_back(i);
        }
    }
    C_prime.push_back(rowIdx);
    C_prime.push_back(colIdx);
    return make_tuple(rowIdx, colIdx);
}

// CUDA kernel for bitmask conversion

__global__ void bitmask_conversion_kernel(int *matrix_B, int *matrix_mask_B, int num_elements)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < num_elements)
    {
        matrix_mask_B[tid] = (matrix_B[tid] != 0) ? 1 : 0;
    }
}

// CUDA kernel for Step 2 and Step 3

__global__ void step2_and_step3_kernel(int *matrix_A, int *matrix_B, int *tileRowidx_C, int *tileColidx_C, int *tilePtr_A,
                                      int *tilePtr_B, int *tileColPtr_B, int *tileColidx_A,
                                      int *tileRowidx_B, int numtileC, int tile_size, int *matrix_C, int *maskc,
                                      int *matrix_mask_B, int num_rows, int num_cols)
{

    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < numtileC)
    {

        int tile_i = tileRowidx_C[tid];
        int tile_j = tileColidx_C[tid];
        int lena = tilePtr_A[tile_i + 1] - tilePtr_A[tile_i];
        int lenb = tileColPtr_B[tile_j + 1] - tileColPtr_B[tile_j];

        // Array to store matched positions
        int matched_posA[MAX_MATCHED_POSITIONS];
        int num_matched = 0;

        // Applying Binary Search to find the matched positions
        if (lena <= lenb)
        {
            for (int value = tilePtr_A[tile_i]; value < tilePtr_A[tile_i + 1]; value++)
            {
                int low = tileColPtr_B[tile_j];
                int high = tileColPtr_B[tile_j + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileRowidx_B[mid] == tileColidx_A[value])
                    {
                        matched_posA[num_matched++] = tileColidx_A[value];
                        break;
                    }
                    else if (tileRowidx_B[mid] < tileColidx_A[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }
        else
        {
            for (int value = tileColPtr_B[tile_j]; value < tileColPtr_B[tile_j + 1]; value++)
            {
                int low = tilePtr_A[tile_i];
                int high = tilePtr_A[tile_i + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileColidx_A[mid] == tileRowidx_B[value])
                    {
                        matched_posA[num_matched++] = tileRowidx_B[value];
                        break;
                    }
                    else if (tileColidx_A[mid] < tileRowidx_B[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }

        for (int matched = 0; matched < num_matched; matched++)
        {
            int j = matched_posA[matched];
            for (int r = 0; r < tile_size; r++)
            {
                for (int s = 0; s < tile_size; s++)
                {
                    int row = tile_size * tile_i + r;
                    int col = tile_size * j + s;
                    if (matrix_A[row * num_cols + col] != 0)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            atomicOr(&maskc[row * num_cols + (tile_size * tile_j + t)], matrix_mask_B[(tile_size * j + s) * num_cols + (tile_size * tile_j + t)]);
                        }
                    }
                }
            }
        }
        int nnz = 0;
        for (int i = 0; i < tile_size; i++)
        {
            for (int j = 0; j < tile_size; j++)
            {
                // printf("%d",maskc[(tile_i * tile_size + i) * num_cols + (tile_j * tile_size + j)]);
                if (maskc[(tile_i * tile_size + i) * num_cols + (tile_j * tile_size + j)] != 0)
                {
                    nnz++;
                }
            }
        }

        if (nnz >= 12)
        {
            for (int matched = 0; matched < num_matched; matched++)
            {
                int j = matched_posA[matched];
                for (int r = 0; r < tile_size; r++)
                {
                    for (int s = 0; s < tile_size; s++)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            matrix_C[(tile_i * tile_size + r) * num_cols + (tile_size * tile_j + s)] += matrix_A[(tile_size * tile_i + r) * num_cols + tile_size * j + t] * matrix_B[(tile_size * j + t) * num_cols + (tile_size * tile_j + s)];
                        }
                    }
                }
            }
        }
        else
        {
            for (int matched = 0; matched < num_matched; matched++)
            {
                int matched_tiles = matched_posA[matched];
                for (int r = 0; r < tile_size; ++r)
                {
                    for (int s = 0; s < tile_size; ++s)
                    {
                        if(matrix_A[(tile_i*tile_size + r)*num_cols + matched_tiles*tile_size +s] == 0){
                            continue;
                        }

                        for (int t = 0; t < tile_size; ++t)
                        {

  matrix_C[(tile_i*tile_size + r)*num_cols + (tile_j*tile_size + t)] += matrix_A[(tile_i*tile_size + r)*num_cols + matched_tiles*tile_size +s] * matrix_B[(tile_size*matched_tiles + s )*num_cols + tile_j*tile_size + t];

                        }
                    }
                }
            }
        }
    }
}
/*
tuple<vector<int>, vector<int>> highLevel_conversion(vector<vector<int>> &matrix, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix.size(); i += tile_size)
    {
        for (int j = 0; j < matrix[i].size(); j += tile_size)
        {
            for (int k = 0; k < tile_size * tile_size; k++)
            {
                int row = k / tile_size;
                int col = k % tile_size;
                if (matrix[i + row][j + col] != 0)
                {
                    colIdx.push_back(j / tile_size);
                    break;
                }
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}
*/


tuple<vector<int>, vector<int>> highLevel_conversion(vector<int> &matrix, int matrix_size, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix_size; i += tile_size)
    {
        for (int j = 0; j < matrix_size; j += tile_size)
        {
            bool hasNonZero = false;
            for (int x = 0; x < tile_size; ++x)
            {
                for (int y = 0; y < tile_size; ++y)
                {
                    int row = i + x;
                    int col = j + y;
                    if (row < matrix_size && col < matrix_size && matrix[row * matrix_size + col] != 0)
                    {
                        colIdx.push_back(j / tile_size);
                        hasNonZero = true;
                        break;
                    }
                }
                if (hasNonZero)
                    break;
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}


tuple<vector<int>, vector<int>> highLevel_conversion_For_MatrixB(vector<int> rowPtr, vector<int> Colidx)
{
    vector<int> colPtr(rowPtr.size(), 0);
    vector<int> rowIdx(Colidx.size(), 0);
    int max = 0;

    for (int i = 0; i < rowPtr.size() - 1; i++)
    {
        for (int j = rowPtr[i]; j < rowPtr[i + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < colPtr.size(); k++)
            {
                colPtr[k]++;
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < max; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = i;
            max++;
        }
    }
    for (int i = 0; i < colPtr.size() - 1; i++)
    {
        sort(rowIdx.begin() + colPtr[i], rowIdx.begin() + colPtr[i + 1]);
    }
    return make_tuple(colPtr, rowIdx);
}
int main() {
    // Parameters
    vector<int> tile_sizes = {4};
    vector<int> matrix_sizes = {128, 256, 384, 512, 640, 768, 896, 1024, 1152, 1280, 1408, 1536, 1664, 1792, 1920, 2048, 2176, 2304, 2432, 2560, 2688, 2816, 2944, 3072};
    vector<double>time;

    double density = 0.1;
    for (int tile_size : tile_sizes) {
        cout << "Tile Size: " << tile_size << endl;
        for (int matrix_size : matrix_sizes) {
            cout << "Matrix Size: " << matrix_size << "x" << matrix_size << endl;
            vector<int> matrix_A = generateRandomSparseMatrix(matrix_size, matrix_size, density);
            vector<int> matrix_B = generateRandomSparseMatrix(matrix_size, matrix_size, density);

            struct timeval begin, end;
            gettimeofday(&begin, 0);

            // High-level conversion for matrix A
            vector<int> tilerowPtr_A, tileColidx_A;
            tie(tilerowPtr_A, tileColidx_A) = highLevel_conversion(matrix_A, matrix_size, tile_size);

            // High-level conversion for matrix B
            vector<int> tilerowPtr_B, tileColidx_B;
            tie(tilerowPtr_B, tileColidx_B) = highLevel_conversion(matrix_B, matrix_size, tile_size);

            // Symbolic SpGEMM for CSR Format
            vector<int> tileRowidx_C, tileColidx_C;
            tie(tileRowidx_C, tileColidx_C) = symbolic_SpGEMM_For_CSRFormat(tilerowPtr_A, tileColidx_A, tilerowPtr_B, tileColidx_B);

            int numtileC = tileRowidx_C.size();
            vector<int> tileColPtr_B, tileRowidx_B;
            tie(tileColPtr_B, tileRowidx_B) = highLevel_conversion_For_MatrixB(tilerowPtr_B, tileColidx_B);




             // Allocate memory for matrices and vectors on device
              int *d_matrix_A, *d_matrix_B, *d_tileRowidx_C, *d_tileColidx_C, *d_tilePtr_A,
                  *d_tilePtr_B, *d_tileColPtr_B, *d_tileColidx_A,
                  *d_tileRowidx_B, *d_matrix_C, *d_maskc, *d_matrix_mask_B;

              // CUDA memory allocation
              cudaMalloc((void **)&d_matrix_A, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_matrix_B, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_tileRowidx_C, sizeof(int) * tileRowidx_C.size());
              cudaMalloc((void **)&d_tileColidx_C, sizeof(int) * tileColidx_C.size());
              cudaMalloc((void **)&d_tilePtr_A, sizeof(int) * tilerowPtr_A.size());
              cudaMalloc((void **)&d_tilePtr_B, sizeof(int) * tilerowPtr_B.size());
              cudaMalloc((void **)&d_tileColPtr_B, sizeof(int) * tileColPtr_B.size());
              cudaMalloc((void **)&d_tileColidx_A, sizeof(int) * tileColidx_A.size());
              cudaMalloc((void **)&d_tileRowidx_B, sizeof(int) * tileRowidx_B.size());
              cudaMalloc((void **)&d_matrix_C, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_maskc, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_matrix_mask_B, sizeof(int) * matrix_size * matrix_size);

              // Copy data from host to device
              cudaMemcpy(d_matrix_A, matrix_A.data(), sizeof(int) * matrix_size * matrix_size, cudaMemcpyHostToDevice);
              cudaMemcpy(d_matrix_B, matrix_B.data(), sizeof(int) * matrix_size * matrix_size, cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileRowidx_C, tileRowidx_C.data(), sizeof(int) * tileRowidx_C.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColidx_C, tileColidx_C.data(), sizeof(int) * tileColidx_C.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tilePtr_A, tilerowPtr_A.data(), sizeof(int) * tilerowPtr_A.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tilePtr_B, tilerowPtr_B.data(), sizeof(int) * tilerowPtr_B.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColPtr_B, tileColPtr_B.data(), sizeof(int) * tileColPtr_B.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColidx_A, tileColidx_A.data(), sizeof(int) * tileColidx_A.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileRowidx_B, tileRowidx_B.data(), sizeof(int) * tileRowidx_B.size(), cudaMemcpyHostToDevice);

              // Define grid and block dimensions
              int numThreads = 256;
              int numBlocks = (numtileC + numThreads - 1) / numThreads;

              // Launch bitmask conversion kernel
              bitmask_conversion_kernel<<<numBlocks, numThreads>>>(d_matrix_B, d_matrix_mask_B, matrix_size * matrix_size);
              cudaDeviceSynchronize();
              int *mask_B_matrix = new int[matrix_size * matrix_size];
              cudaMemcpy(mask_B_matrix, d_matrix_mask_B, sizeof(int) * matrix_size * matrix_size, cudaMemcpyDeviceToHost);

              // Launch Step 2 and Step 3 kernel
              step2_and_step3_kernel<<<numBlocks, numThreads>>>(d_matrix_A, d_matrix_B, d_tileRowidx_C, d_tileColidx_C, d_tilePtr_A,
                                                                d_tilePtr_B, d_tileColPtr_B, d_tileColidx_A,
                                                                d_tileRowidx_B, numtileC, tile_size, d_matrix_C, d_maskc, d_matrix_mask_B, matrix_size, matrix_size);
              cudaDeviceSynchronize();

              gettimeofday(&end, 0);
            long seconds = end.tv_sec - begin.tv_sec;
            long microseconds = end.tv_usec - begin.tv_usec;
            double elapsed = seconds + microseconds*1e-6;
            time.push_back(elapsed);
            printf("Time measured: %.6f seconds.\n", elapsed);

              int *matrix_C_result = new int[matrix_size * matrix_size];
              cudaMemcpy(matrix_C_result, d_matrix_C, sizeof(int) * matrix_size * matrix_size, cudaMemcpyDeviceToHost);
            // Free host memory
            delete[] matrix_C_result;

            // Copy result matrix from device to host
            // Free device memory
            cudaFree(d_matrix_A);
            cudaFree(d_matrix_B);
            cudaFree(d_tileRowidx_C);
            cudaFree(d_tileColidx_C);
            cudaFree(d_tilePtr_A);
            cudaFree(d_tilePtr_B);
            cudaFree(d_tileColPtr_B);
            cudaFree(d_tileColidx_A);
            cudaFree(d_tileRowidx_B);
            cudaFree(d_matrix_C);
            cudaFree(d_maskc);
            cudaFree(d_matrix_mask_B);
            cout << "hello" << endl;
        }


    }
    return 0;
}

Tile Size: 4
Matrix Size: 128x128
Time measured: 0.310486 seconds.
hello
Matrix Size: 256x256
Time measured: 0.064172 seconds.
hello
Matrix Size: 384x384
Time measured: 0.252681 seconds.
hello
Matrix Size: 512x512
Time measured: 0.687308 seconds.
hello
Matrix Size: 640x640
Time measured: 2.116419 seconds.
hello
Matrix Size: 768x768
Time measured: 3.558099 seconds.
hello
Matrix Size: 896x896
Time measured: 4.782732 seconds.
hello
Matrix Size: 1024x1024
Time measured: 8.135314 seconds.
hello
Matrix Size: 1152x1152
Time measured: 13.004249 seconds.
hello
Matrix Size: 1280x1280
Time measured: 17.237437 seconds.
hello
Matrix Size: 1408x1408
Time measured: 25.211076 seconds.
hello
Matrix Size: 1536x1536
Time measured: 34.130772 seconds.
hello
Matrix Size: 1664x1664
Time measured: 46.719020 seconds.
hello
Matrix Size: 1792x1792
Time measured: 69.613498 seconds.
hello
Matrix Size: 1920x1920
Time measured: 80.638414 seconds.
hello
Matrix Size: 2048x2048
Time measured: 101.030979 seconds.
hello


# Density 0.25

# using Tile which is optimal

In [ ]:
%%cuda
#include <random>
#include<bits/stdc++.h>
#include <algorithm>
#include <vector>
#include <iostream>
#include <sys/time.h>
#include <cuda_runtime.h>
#define MAX_MATCHED_POSITIONS 16

using namespace std;

vector<int> generateRandomSparseMatrix(int rows, int cols, double density) {
    vector<int> matrix(rows*cols);
    std::random_device rd;
    std::mt19937 gen(rd());
    std::uniform_real_distribution<> dis(0.0, 1.0);
    for (int i = 0; i < rows; ++i) {
        for (int j = 0; j < cols; ++j) {
            double randNum = dis(gen);
            if (randNum <= density) {
                matrix[i*rows+j] = 1;
            }
        }
    }
    return matrix;
}

tuple<vector<int>, vector<int>> symbolic_SpGEMM_For_CSRFormat(vector<int> tilerowPtr_A, vector<int> tileColidx_A, vector<int> tilerowPtr_B, vector<int> tileColidx_B)
{
    int n = tilerowPtr_A.size();
    int m = tilerowPtr_B.size();
    vector<vector<int>> C_prime;
    vector<int> rowIdx, colIdx;

    for (int i = 0; i < n - 1; ++i)
    {
        set<int> unique_col;
        for (int j = tilerowPtr_A[i]; j < tilerowPtr_A[i + 1]; ++j)
        {
            int col = tileColidx_A[j];
            for (int i1 = tilerowPtr_B[col]; i1 < tilerowPtr_B[col + 1]; ++i1)
            {
                int col1 = tileColidx_B[i1];
                unique_col.insert(col1);
            }
        }
        for (int col : unique_col)
        {
            colIdx.push_back(col);
            rowIdx.push_back(i);
        }
    }
    C_prime.push_back(rowIdx);
    C_prime.push_back(colIdx);
    return make_tuple(rowIdx, colIdx);
}

// CUDA kernel for bitmask conversion

__global__ void bitmask_conversion_kernel(int *matrix_B, int *matrix_mask_B, int num_elements)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < num_elements)
    {
        matrix_mask_B[tid] = (matrix_B[tid] != 0) ? 1 : 0;
    }
}

// CUDA kernel for Step 2 and Step 3

__global__ void step2_and_step3_kernel(int *matrix_A, int *matrix_B, int *tileRowidx_C, int *tileColidx_C, int *tilePtr_A,
                                      int *tilePtr_B, int *tileColPtr_B, int *tileColidx_A,
                                      int *tileRowidx_B, int numtileC, int tile_size, int *matrix_C, int *maskc,
                                      int *matrix_mask_B, int num_rows, int num_cols)
{

    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < numtileC)
    {

        int tile_i = tileRowidx_C[tid];
        int tile_j = tileColidx_C[tid];
        int lena = tilePtr_A[tile_i + 1] - tilePtr_A[tile_i];
        int lenb = tileColPtr_B[tile_j + 1] - tileColPtr_B[tile_j];

        // Array to store matched positions
        int matched_posA[MAX_MATCHED_POSITIONS];
        int num_matched = 0;

        // Applying Binary Search to find the matched positions
        if (lena <= lenb)
        {
            for (int value = tilePtr_A[tile_i]; value < tilePtr_A[tile_i + 1]; value++)
            {
                int low = tileColPtr_B[tile_j];
                int high = tileColPtr_B[tile_j + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileRowidx_B[mid] == tileColidx_A[value])
                    {
                        matched_posA[num_matched++] = tileColidx_A[value];
                        break;
                    }
                    else if (tileRowidx_B[mid] < tileColidx_A[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }
        else
        {
            for (int value = tileColPtr_B[tile_j]; value < tileColPtr_B[tile_j + 1]; value++)
            {
                int low = tilePtr_A[tile_i];
                int high = tilePtr_A[tile_i + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileColidx_A[mid] == tileRowidx_B[value])
                    {
                        matched_posA[num_matched++] = tileRowidx_B[value];
                        break;
                    }
                    else if (tileColidx_A[mid] < tileRowidx_B[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }

        for (int matched = 0; matched < num_matched; matched++)
        {
            int j = matched_posA[matched];
            for (int r = 0; r < tile_size; r++)
            {
                for (int s = 0; s < tile_size; s++)
                {
                    int row = tile_size * tile_i + r;
                    int col = tile_size * j + s;
                    if (matrix_A[row * num_cols + col] != 0)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            atomicOr(&maskc[row * num_cols + (tile_size * tile_j + t)], matrix_mask_B[(tile_size * j + s) * num_cols + (tile_size * tile_j + t)]);
                        }
                    }
                }
            }
        }
        int nnz = 0;
        for (int i = 0; i < tile_size; i++)
        {
            for (int j = 0; j < tile_size; j++)
            {
                // printf("%d",maskc[(tile_i * tile_size + i) * num_cols + (tile_j * tile_size + j)]);
                if (maskc[(tile_i * tile_size + i) * num_cols + (tile_j * tile_size + j)] != 0)
                {
                    nnz++;
                }
            }
        }

        if (nnz >= 12)
        {
            for (int matched = 0; matched < num_matched; matched++)
            {
                int j = matched_posA[matched];
                for (int r = 0; r < tile_size; r++)
                {
                    for (int s = 0; s < tile_size; s++)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            matrix_C[(tile_i * tile_size + r) * num_cols + (tile_size * tile_j + s)] += matrix_A[(tile_size * tile_i + r) * num_cols + tile_size * j + t] * matrix_B[(tile_size * j + t) * num_cols + (tile_size * tile_j + s)];
                        }
                    }
                }
            }
        }
        else
        {
            for (int matched = 0; matched < num_matched; matched++)
            {
                int matched_tiles = matched_posA[matched];
                for (int r = 0; r < tile_size; ++r)
                {
                    for (int s = 0; s < tile_size; ++s)
                    {
                        if(matrix_A[(tile_i*tile_size + r)*num_cols + matched_tiles*tile_size +s] == 0){
                            continue;
                        }

                        for (int t = 0; t < tile_size; ++t)
                        {

  matrix_C[(tile_i*tile_size + r)*num_cols + (tile_j*tile_size + t)] += matrix_A[(tile_i*tile_size + r)*num_cols + matched_tiles*tile_size +s] * matrix_B[(tile_size*matched_tiles + s )*num_cols + tile_j*tile_size + t];

                        }
                    }
                }
            }
        }
    }
}
/*
tuple<vector<int>, vector<int>> highLevel_conversion(vector<vector<int>> &matrix, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix.size(); i += tile_size)
    {
        for (int j = 0; j < matrix[i].size(); j += tile_size)
        {
            for (int k = 0; k < tile_size * tile_size; k++)
            {
                int row = k / tile_size;
                int col = k % tile_size;
                if (matrix[i + row][j + col] != 0)
                {
                    colIdx.push_back(j / tile_size);
                    break;
                }
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}
*/


tuple<vector<int>, vector<int>> highLevel_conversion(vector<int> &matrix, int matrix_size, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix_size; i += tile_size)
    {
        for (int j = 0; j < matrix_size; j += tile_size)
        {
            bool hasNonZero = false;
            for (int x = 0; x < tile_size; ++x)
            {
                for (int y = 0; y < tile_size; ++y)
                {
                    int row = i + x;
                    int col = j + y;
                    if (row < matrix_size && col < matrix_size && matrix[row * matrix_size + col] != 0)
                    {
                        colIdx.push_back(j / tile_size);
                        hasNonZero = true;
                        break;
                    }
                }
                if (hasNonZero)
                    break;
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}


tuple<vector<int>, vector<int>> highLevel_conversion_For_MatrixB(vector<int> rowPtr, vector<int> Colidx)
{
    vector<int> colPtr(rowPtr.size(), 0);
    vector<int> rowIdx(Colidx.size(), 0);
    int max = 0;

    for (int i = 0; i < rowPtr.size() - 1; i++)
    {
        for (int j = rowPtr[i]; j < rowPtr[i + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < colPtr.size(); k++)
            {
                colPtr[k]++;
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < max; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = i;
            max++;
        }
    }
    for (int i = 0; i < colPtr.size() - 1; i++)
    {
        sort(rowIdx.begin() + colPtr[i], rowIdx.begin() + colPtr[i + 1]);
    }
    return make_tuple(colPtr, rowIdx);
}
int main() {
    // Parameters
    vector<int> tile_sizes = {64};
    vector<int> matrix_sizes = {128, 256, 384, 512, 640, 768, 896, 1024, 1152, 1280, 1408, 1536, 1664, 1792, 1920, 2048, 2176, 2304, 2432, 2560, 2688, 2816, 2944, 3072};
    vector<double>time;

    double density = 0.1;
    for (int tile_size : tile_sizes) {
        cout << "Tile Size: " << tile_size << endl;
        for (int matrix_size : matrix_sizes) {
            cout << "Matrix Size: " << matrix_size << "x" << matrix_size << endl;
            vector<int> matrix_A = generateRandomSparseMatrix(matrix_size, matrix_size, density);
            vector<int> matrix_B = generateRandomSparseMatrix(matrix_size, matrix_size, density);

            struct timeval begin, end;
            gettimeofday(&begin, 0);

            // High-level conversion for matrix A
            vector<int> tilerowPtr_A, tileColidx_A;
            tie(tilerowPtr_A, tileColidx_A) = highLevel_conversion(matrix_A, matrix_size, tile_size);

            // High-level conversion for matrix B
            vector<int> tilerowPtr_B, tileColidx_B;
            tie(tilerowPtr_B, tileColidx_B) = highLevel_conversion(matrix_B, matrix_size, tile_size);

            // Symbolic SpGEMM for CSR Format
            vector<int> tileRowidx_C, tileColidx_C;
            tie(tileRowidx_C, tileColidx_C) = symbolic_SpGEMM_For_CSRFormat(tilerowPtr_A, tileColidx_A, tilerowPtr_B, tileColidx_B);

            int numtileC = tileRowidx_C.size();
            vector<int> tileColPtr_B, tileRowidx_B;
            tie(tileColPtr_B, tileRowidx_B) = highLevel_conversion_For_MatrixB(tilerowPtr_B, tileColidx_B);




             // Allocate memory for matrices and vectors on device
              int *d_matrix_A, *d_matrix_B, *d_tileRowidx_C, *d_tileColidx_C, *d_tilePtr_A,
                  *d_tilePtr_B, *d_tileColPtr_B, *d_tileColidx_A,
                  *d_tileRowidx_B, *d_matrix_C, *d_maskc, *d_matrix_mask_B;

              // CUDA memory allocation
              cudaMalloc((void **)&d_matrix_A, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_matrix_B, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_tileRowidx_C, sizeof(int) * tileRowidx_C.size());
              cudaMalloc((void **)&d_tileColidx_C, sizeof(int) * tileColidx_C.size());
              cudaMalloc((void **)&d_tilePtr_A, sizeof(int) * tilerowPtr_A.size());
              cudaMalloc((void **)&d_tilePtr_B, sizeof(int) * tilerowPtr_B.size());
              cudaMalloc((void **)&d_tileColPtr_B, sizeof(int) * tileColPtr_B.size());
              cudaMalloc((void **)&d_tileColidx_A, sizeof(int) * tileColidx_A.size());
              cudaMalloc((void **)&d_tileRowidx_B, sizeof(int) * tileRowidx_B.size());
              cudaMalloc((void **)&d_matrix_C, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_maskc, sizeof(int) * matrix_size * matrix_size);
              cudaMalloc((void **)&d_matrix_mask_B, sizeof(int) * matrix_size * matrix_size);

              // Copy data from host to device
              cudaMemcpy(d_matrix_A, matrix_A.data(), sizeof(int) * matrix_size * matrix_size, cudaMemcpyHostToDevice);
              cudaMemcpy(d_matrix_B, matrix_B.data(), sizeof(int) * matrix_size * matrix_size, cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileRowidx_C, tileRowidx_C.data(), sizeof(int) * tileRowidx_C.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColidx_C, tileColidx_C.data(), sizeof(int) * tileColidx_C.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tilePtr_A, tilerowPtr_A.data(), sizeof(int) * tilerowPtr_A.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tilePtr_B, tilerowPtr_B.data(), sizeof(int) * tilerowPtr_B.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColPtr_B, tileColPtr_B.data(), sizeof(int) * tileColPtr_B.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileColidx_A, tileColidx_A.data(), sizeof(int) * tileColidx_A.size(), cudaMemcpyHostToDevice);
              cudaMemcpy(d_tileRowidx_B, tileRowidx_B.data(), sizeof(int) * tileRowidx_B.size(), cudaMemcpyHostToDevice);

              // Define grid and block dimensions
              int numThreads = 256;
              int numBlocks = (numtileC + numThreads - 1) / numThreads;

              // Launch bitmask conversion kernel
              bitmask_conversion_kernel<<<numBlocks, numThreads>>>(d_matrix_B, d_matrix_mask_B, matrix_size * matrix_size);
              cudaDeviceSynchronize();
              int *mask_B_matrix = new int[matrix_size * matrix_size];
              cudaMemcpy(mask_B_matrix, d_matrix_mask_B, sizeof(int) * matrix_size * matrix_size, cudaMemcpyDeviceToHost);

              // Launch Step 2 and Step 3 kernel
              step2_and_step3_kernel<<<numBlocks, numThreads>>>(d_matrix_A, d_matrix_B, d_tileRowidx_C, d_tileColidx_C, d_tilePtr_A,
                                                                d_tilePtr_B, d_tileColPtr_B, d_tileColidx_A,
                                                                d_tileRowidx_B, numtileC, tile_size, d_matrix_C, d_maskc, d_matrix_mask_B, matrix_size, matrix_size);
              cudaDeviceSynchronize();

              gettimeofday(&end, 0);
            long seconds = end.tv_sec - begin.tv_sec;
            long microseconds = end.tv_usec - begin.tv_usec;
            double elapsed = seconds + microseconds*1e-6;
            time.push_back(elapsed);
            printf("Time measured: %.6f seconds.\n", elapsed);

              int *matrix_C_result = new int[matrix_size * matrix_size];
              cudaMemcpy(matrix_C_result, d_matrix_C, sizeof(int) * matrix_size * matrix_size, cudaMemcpyDeviceToHost);

            // Free host memory
            delete[] matrix_C_result;

            // Copy result matrix from device to host
            // Free device memory
            cudaFree(d_matrix_A);
            cudaFree(d_matrix_B);
            cudaFree(d_tileRowidx_C);
            cudaFree(d_tileColidx_C);
            cudaFree(d_tilePtr_A);
            cudaFree(d_tilePtr_B);
            cudaFree(d_tileColPtr_B);
            cudaFree(d_tileColidx_A);
            cudaFree(d_tileRowidx_B);
            cudaFree(d_matrix_C);
            cudaFree(d_maskc);
            cudaFree(d_matrix_mask_B);
            cout << "hello" << endl;
        }


    }
    return 0;
}

Tile Size: 64
Matrix Size: 128x128
Time measured: 0.823629 seconds.
hello
Matrix Size: 256x256
Time measured: 0.145960 seconds.
hello
Matrix Size: 384x384
Time measured: 0.071950 seconds.
hello
Matrix Size: 512x512
Time measured: 0.065733 seconds.
hello
Matrix Size: 640x640
Time measured: 0.041699 seconds.
hello
Matrix Size: 768x768
Time measured: 0.049148 seconds.
hello
Matrix Size: 896x896
Time measured: 0.057713 seconds.
hello
Matrix Size: 1024x1024
Time measured: 0.063523 seconds.
hello
Matrix Size: 1152x1152
Time measured: 0.066514 seconds.
hello
Matrix Size: 1280x1280
Time measured: 0.069770 seconds.
hello
Matrix Size: 1408x1408
Time measured: 0.100053 seconds.
hello
Matrix Size: 1536x1536
Time measured: 0.119043 seconds.
hello
Matrix Size: 1664x1664
Time measured: 0.111994 seconds.
hello
Matrix Size: 1792x1792
Time measured: 0.160685 seconds.
hello
Matrix Size: 1920x1920
Time measured: 0.150646 seconds.
hello
Matrix Size: 2048x2048
Time measured: 0.101102 seconds.
hello
Matrix S

# SEQUENTIAL TILESPGEMM

## Density 0.1

### Tile: 64

In [ ]:
%%cuda
#include <random>
#include<bits/stdc++.h>
#include <algorithm>
#include <vector>
#include <iostream>
#include <bits/stdc++.h>
#include <sys/time.h>


using namespace std;
vector<vector<int>> generateRandomSparseMatrix(int rows, int cols, double density) {
    vector<vector<int>> matrix(rows, vector<int>(cols, 0));
    std::random_device rd;
    std::mt19937 gen(rd());
    std::uniform_real_distribution<> dis(0.0, 1.0);
    for (int i = 0; i < rows; ++i) {
        for (int j = 0; j < cols; ++j) {
            double randNum = dis(gen);
            if (randNum <= density) {
                matrix[i][j] = 1;
            }
        }
    }
    return matrix;
}

tuple<vector<int>, vector<int>> symbolic_SpGEMM_For_CSRFormat(vector<int> tilerowPtr_A, vector<int> tileColidx_A, vector<int> tilerowPtr_B, vector<int> tileColidx_B)
{
    int n = tilerowPtr_A.size();
    int m = tilerowPtr_B.size();
    vector<vector<int>> C_prime;
    vector<int> rowIdx, colIdx;

    for (int i = 0; i < n - 1; ++i)
    {
        set<int> unique_col;
        for (int j = tilerowPtr_A[i]; j < tilerowPtr_A[i + 1]; ++j)
        {
            int col = tileColidx_A[j];
            for (int i1 = tilerowPtr_B[col]; i1 < tilerowPtr_B[col + 1]; ++i1)
            {
                int col1 = tileColidx_B[i1];
                unique_col.insert(col1);
            }
        }
        for (int col : unique_col)
        {
            colIdx.push_back(col);
            rowIdx.push_back(i);
        }
    }
    C_prime.push_back(rowIdx);
    C_prime.push_back(colIdx);
    return make_tuple(rowIdx, colIdx);
}

vector<vector<int>> bitmask_conversion(vector<vector<int>> &matrix)
{
    vector<vector<int>> matrix_mask_B(matrix.size(), vector<int>(matrix[0].size(), 0));
    for (int i = 0; i < matrix.size(); i++)
    {
        for (int j = 0; j < matrix[i].size(); j++)
        {
            if (matrix[i][j] != 0)
            {
                matrix_mask_B[i][j] = 1;
            }
        }
    }
    return matrix_mask_B;
}

vector<vector<int>> step2_and_step3(vector<vector<int>> matrix_A, vector<vector<int>> matrix_B, vector<int> &tileRowidx_C, vector<int> &tileColidx_C, vector<int> &tilePtr_A,
                                    vector<int> &tilePtr_B, vector<int> &tileColPtr_B, vector<int> &tileColidx_A,
                                    vector<int> &tileRowidx_B, int numtileC, int tile_size)
{

    vector<vector<int>> matrix_C(matrix_A.size(), vector<int>(matrix_B[0].size(), 0));
    vector<vector<int>> maskc(matrix_A.size(), vector<int>(matrix_B[0].size(), 0));
    vector<vector<int>> matrix_mask_B = bitmask_conversion(matrix_B);
    for (int i = 0; i < numtileC; i++)
    {
        vector<int> matched_posA;
        int tile_i = tileRowidx_C[i];
        int tile_j = tileColidx_C[i];
        int lena = tilePtr_A[tile_i + 1] - tilePtr_A[tile_i];
        int lenb = tileColPtr_B[tile_j + 1] - tileColPtr_B[tile_j];
        // Applying Binary Search to find the matched positions
        if (lena <= lenb)
        {
            for (int value = tilePtr_A[tile_i]; value < tilePtr_A[tile_i + 1]; value++)
            {
                int low = tileColPtr_B[tile_j];
                int high = tileColPtr_B[tile_j + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileRowidx_B[mid] == tileColidx_A[value])
                    {
                        matched_posA.push_back(tileColidx_A[value]);
                        break;
                    }
                    else if (tileRowidx_B[mid] < tileColidx_A[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }
        else
        {
            for (int value = tileColPtr_B[tile_j]; value < tileColPtr_B[tile_j + 1]; value++)
            {
                int low = tilePtr_A[tile_i];
                int high = tilePtr_A[tile_i + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileColidx_A[mid] == tileRowidx_B[value])
                    {
                        matched_posA.push_back(tileRowidx_B[value]);
                        break;
                    }
                    else if (tileColidx_A[mid] < tileRowidx_B[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }

        // Calculaitng the mask of the each tile using the matched positions

        for (int matched = 0; matched < matched_posA.size(); matched++)
        {
            int j = matched_posA[matched];
            for (int r = 0; r < tile_size; r++)
            {
                for (int s = 0; s < tile_size; s++)
                {

                    int row = tile_size * tile_i + r;
                    int col = tile_size * j + s;
                    if (matrix_A[row][col] != 0)
                    {

                        for (int t = 0; t < tile_size; t++)
                        {
                            maskc[row][tile_size * tile_j + t] = maskc[row][tile_size * tile_j + t] || matrix_mask_B[tile_size * j + s][tile_size * tile_j + t];
                        }
                    }
                }
            }
        }

        // Calculating the number of non-zeros values in each tile using the maskc
        int nnz = 0;
        for (int i = 0; i < tile_size; i++)
        {
            for (int j = 0; j < tile_size; j++)
            {
                if (maskc[tile_i * tile_size + i][tile_j * tile_size + j] != 0)
                {
                    nnz++;
                }
            }
        }

        // Step 3: For Calculating the matrix C using the number of non-zero values in each tile
        vector<int> idx, C_ptr;
        C_ptr.push_back(0);

        int sizee = 0;
        for (int it = 0; it < tile_size; it++)
        {
            for (int jt = 0; jt < tile_size; jt++)
            {
                if (maskc[tile_i * tile_size + it][tile_j * tile_size + jt] != 0)
                {
                    nnz++;
                    idx.push_back(jt);
                }
            }
            if (sizee < idx.size())
            {
                C_ptr.push_back(idx.size());
                sizee = idx.size();
            }
        }
        // Dense Accumulator
        int threshold = 0.75*tile_size*tile_size;
        if (nnz >= threshold)
        {
            for (int matched = 0; matched < matched_posA.size(); matched++)
            {
                int j = matched_posA[matched];
                for (int r = 0; r < tile_size; r++)
                {
                    for (int s = 0; s < tile_size; s++)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            matrix_C[tile_i * tile_size + r][tile_size * tile_j + s] += matrix_A[tile_size * tile_i + r][tile_size * j + t] * matrix_B[tile_size * j + t][tile_size * tile_j + s];
                        }
                    }
                }
            }
        }
        // Sparse Accumulator
        else
        {

            for (int matched = 0; matched < matched_posA.size(); matched++)
            {
                int jt = matched_posA[matched];
                for (int it = 0; it < C_ptr.size() - 1; ++it)
                {
                    int start = C_ptr[it];
                    int end = C_ptr[it + 1];
                    for (int j1 = start; j1 < end; ++j1)
                    {
                        int colC = idx[j1];
                        for (int r = 0; r < tile_size; r++)
                        {
                            for (int t = 0; t < tile_size; t++)
                            {
                                matrix_C[tile_i * tile_size + r][tile_size * tile_j + colC] += matrix_A[tile_size * tile_i + r][tile_size * jt + t] * matrix_B[tile_size * jt + t][tile_size * tile_j + colC];
                            }
                        }
                    }
                }
            }
        }
    }
    return matrix_C;
}

tuple<vector<int>, vector<int>> highLevel_conversion(vector<vector<int>> &matrix, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix.size(); i += tile_size)
    {
        for (int j = 0; j < matrix[i].size(); j += tile_size)
        {
            for (int k = 0; k < tile_size * tile_size; k++)
            {
                int row = k / tile_size;
                int col = k % tile_size;
                if (matrix[i + row][j + col] != 0)
                {
                    colIdx.push_back(j / tile_size);
                    break;
                }
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}

tuple<vector<int>, vector<int>> highLevel_conversion_For_MatrixB(vector<int> rowPtr, vector<int> Colidx)
{
    vector<int> colPtr(rowPtr.size(), 0);
    vector<int> rowIdx(Colidx.size(), 0);
    int max = 0;

    for (int i = 0; i < rowPtr.size() - 1; i++)
    {
        for (int j = rowPtr[i]; j < rowPtr[i + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < colPtr.size(); k++)
            {
                colPtr[k]++;
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < max; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = i;
            max++;
        }
    }
    for (int i = 0; i < colPtr.size() - 1; i++)
    {
        sort(rowIdx.begin() + colPtr[i], rowIdx.begin() + colPtr[i + 1]);
    }
    return make_tuple(colPtr, rowIdx);
}
int main() {
// Parameters
    vector<int> tile_sizes = {64};
    vector<int> matrix_sizes = {128, 256, 384, 512, 640, 768, 896, 1024, 1152, 1280, 1408, 1536, 1664, 1792, 1920, 2048, 2176, 2304, 2432, 2560, 2688, 2816, 2944, 3072};
    vector<double>time;

    double density = 0.1;

    // Loop over tile sizes
    for (int tile_size : tile_sizes) {
        cout << "Tile Size: " << tile_size << endl;

        // Loop over matrix sizes
        for (int matrix_size : matrix_sizes) {
            cout << "Matrix Size: " << matrix_size << "x" << matrix_size << endl;

            // Generate random sparse matrices A and B
            vector<vector<int>> matrix_A = generateRandomSparseMatrix(matrix_size, matrix_size, density);
            vector<vector<int>> matrix_B = generateRandomSparseMatrix(matrix_size, matrix_size, density);

            // Start measuring time
            struct timeval begin, end;
            gettimeofday(&begin, 0);

            // High-level conversion for matrix A
            vector<int> tilerowPtr_A, tileColidx_A;
            tie(tilerowPtr_A, tileColidx_A) = highLevel_conversion(matrix_A, tile_size);

            // High-level conversion for matrix B
            vector<int> tilerowPtr_B, tileColidx_B;
            tie(tilerowPtr_B, tileColidx_B) = highLevel_conversion(matrix_B, tile_size);

            // Symbolic SpGEMM for CSR Format
            vector<int> tileRowidx_C, tileColidx_C;
            tie(tileRowidx_C, tileColidx_C) = symbolic_SpGEMM_For_CSRFormat(tilerowPtr_A, tileColidx_A, tilerowPtr_B, tileColidx_B);

            int numtileC = tileRowidx_C.size();
            vector<int> tileColPtr_B, tileRowidx_B;
            tie(tileColPtr_B, tileRowidx_B) = highLevel_conversion_For_MatrixB(tilerowPtr_B, tileColidx_B);
            vector<vector<int>> matrix_C = step2_and_step3(matrix_A, matrix_B, tileRowidx_C, tileColidx_C, tilerowPtr_A, tilerowPtr_B, tileColPtr_B, tileColidx_A, tileRowidx_B, numtileC, tile_size);

            gettimeofday(&end, 0);
            long seconds = end.tv_sec - begin.tv_sec;
            long microseconds = end.tv_usec - begin.tv_usec;
            double elapsed = seconds + microseconds*1e-6;
            time.push_back(elapsed);
            printf("Time measured: %.6f seconds.\n", elapsed);
        }
        cout << "Matrix time:" << endl;
        for (const auto& row : time) {
            printf("%.6f, ", row);
        }
        printf("\n");

    }
    return 0;
}

Tile Size: 64
Matrix Size: 128x128
Time measured: 0.070780 seconds.
Matrix Size: 256x256
Time measured: 0.405569 seconds.
Matrix Size: 384x384
Time measured: 0.756715 seconds.
Matrix Size: 512x512
Time measured: 1.424304 seconds.
Matrix Size: 640x640
Time measured: 3.590994 seconds.
Matrix Size: 768x768
Time measured: 1.587158 seconds.
Matrix Size: 896x896
Time measured: 2.129216 seconds.
Matrix Size: 1024x1024
Time measured: 3.497337 seconds.
Matrix Size: 1152x1152
Time measured: 4.671205 seconds.
Matrix Size: 1280x1280
Time measured: 4.327887 seconds.
Matrix Size: 1408x1408
Time measured: 5.648005 seconds.
Matrix Size: 1536x1536
Time measured: 6.576140 seconds.
Matrix Size: 1664x1664
Time measured: 8.273040 seconds.
Matrix Size: 1792x1792
Time measured: 9.211680 seconds.
Matrix Size: 1920x1920
Time measured: 9.661595 seconds.
Matrix Size: 2048x2048
Time measured: 12.061053 seconds.
Matrix Size: 2176x2176
Time measured: 13.738149 seconds.
Matrix Size: 2304x2304
Time measured: 15.28139

### Tile : 32

In [ ]:
%%cuda
#include <random>
#include<bits/stdc++.h>
#include <algorithm>
#include <vector>
#include <iostream>
#include <bits/stdc++.h>
#include <sys/time.h>


using namespace std;
vector<vector<int>> generateRandomSparseMatrix(int rows, int cols, double density) {
    vector<vector<int>> matrix(rows, vector<int>(cols, 0));
    std::random_device rd;
    std::mt19937 gen(rd());
    std::uniform_real_distribution<> dis(0.0, 1.0);
    for (int i = 0; i < rows; ++i) {
        for (int j = 0; j < cols; ++j) {
            double randNum = dis(gen);
            if (randNum <= density) {
                matrix[i][j] = 1;
            }
        }
    }
    return matrix;
}

tuple<vector<int>, vector<int>> symbolic_SpGEMM_For_CSRFormat(vector<int> tilerowPtr_A, vector<int> tileColidx_A, vector<int> tilerowPtr_B, vector<int> tileColidx_B)
{
    int n = tilerowPtr_A.size();
    int m = tilerowPtr_B.size();
    vector<vector<int>> C_prime;
    vector<int> rowIdx, colIdx;

    for (int i = 0; i < n - 1; ++i)
    {
        set<int> unique_col;
        for (int j = tilerowPtr_A[i]; j < tilerowPtr_A[i + 1]; ++j)
        {
            int col = tileColidx_A[j];
            for (int i1 = tilerowPtr_B[col]; i1 < tilerowPtr_B[col + 1]; ++i1)
            {
                int col1 = tileColidx_B[i1];
                unique_col.insert(col1);
            }
        }
        for (int col : unique_col)
        {
            colIdx.push_back(col);
            rowIdx.push_back(i);
        }
    }
    C_prime.push_back(rowIdx);
    C_prime.push_back(colIdx);
    return make_tuple(rowIdx, colIdx);
}

vector<vector<int>> bitmask_conversion(vector<vector<int>> &matrix)
{
    vector<vector<int>> matrix_mask_B(matrix.size(), vector<int>(matrix[0].size(), 0));
    for (int i = 0; i < matrix.size(); i++)
    {
        for (int j = 0; j < matrix[i].size(); j++)
        {
            if (matrix[i][j] != 0)
            {
                matrix_mask_B[i][j] = 1;
            }
        }
    }
    return matrix_mask_B;
}

vector<vector<int>> step2_and_step3(vector<vector<int>> matrix_A, vector<vector<int>> matrix_B, vector<int> &tileRowidx_C, vector<int> &tileColidx_C, vector<int> &tilePtr_A,
                                    vector<int> &tilePtr_B, vector<int> &tileColPtr_B, vector<int> &tileColidx_A,
                                    vector<int> &tileRowidx_B, int numtileC, int tile_size)
{

    vector<vector<int>> matrix_C(matrix_A.size(), vector<int>(matrix_B[0].size(), 0));
    vector<vector<int>> maskc(matrix_A.size(), vector<int>(matrix_B[0].size(), 0));
    vector<vector<int>> matrix_mask_B = bitmask_conversion(matrix_B);
    for (int i = 0; i < numtileC; i++)
    {
        vector<int> matched_posA;
        int tile_i = tileRowidx_C[i];
        int tile_j = tileColidx_C[i];
        int lena = tilePtr_A[tile_i + 1] - tilePtr_A[tile_i];
        int lenb = tileColPtr_B[tile_j + 1] - tileColPtr_B[tile_j];
        // Applying Binary Search to find the matched positions
        if (lena <= lenb)
        {
            for (int value = tilePtr_A[tile_i]; value < tilePtr_A[tile_i + 1]; value++)
            {
                int low = tileColPtr_B[tile_j];
                int high = tileColPtr_B[tile_j + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileRowidx_B[mid] == tileColidx_A[value])
                    {
                        matched_posA.push_back(tileColidx_A[value]);
                        break;
                    }
                    else if (tileRowidx_B[mid] < tileColidx_A[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }
        else
        {
            for (int value = tileColPtr_B[tile_j]; value < tileColPtr_B[tile_j + 1]; value++)
            {
                int low = tilePtr_A[tile_i];
                int high = tilePtr_A[tile_i + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileColidx_A[mid] == tileRowidx_B[value])
                    {
                        matched_posA.push_back(tileRowidx_B[value]);
                        break;
                    }
                    else if (tileColidx_A[mid] < tileRowidx_B[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }

        // Calculaitng the mask of the each tile using the matched positions

        for (int matched = 0; matched < matched_posA.size(); matched++)
        {
            int j = matched_posA[matched];
            for (int r = 0; r < tile_size; r++)
            {
                for (int s = 0; s < tile_size; s++)
                {

                    int row = tile_size * tile_i + r;
                    int col = tile_size * j + s;
                    if (matrix_A[row][col] != 0)
                    {

                        for (int t = 0; t < tile_size; t++)
                        {
                            maskc[row][tile_size * tile_j + t] = maskc[row][tile_size * tile_j + t] || matrix_mask_B[tile_size * j + s][tile_size * tile_j + t];
                        }
                    }
                }
            }
        }

        // Calculating the number of non-zeros values in each tile using the maskc
        int nnz = 0;
        for (int i = 0; i < tile_size; i++)
        {
            for (int j = 0; j < tile_size; j++)
            {
                if (maskc[tile_i * tile_size + i][tile_j * tile_size + j] != 0)
                {
                    nnz++;
                }
            }
        }

        // Step 3: For Calculating the matrix C using the number of non-zero values in each tile
        vector<int> idx, C_ptr;
        C_ptr.push_back(0);

        int sizee = 0;
        for (int it = 0; it < tile_size; it++)
        {
            for (int jt = 0; jt < tile_size; jt++)
            {
                if (maskc[tile_i * tile_size + it][tile_j * tile_size + jt] != 0)
                {
                    nnz++;
                    idx.push_back(jt);
                }
            }
            if (sizee < idx.size())
            {
                C_ptr.push_back(idx.size());
                sizee = idx.size();
            }
        }
        // Dense Accumulator
        int threshold = 0.75*tile_size*tile_size;
        if (nnz >= threshold)
        {
            for (int matched = 0; matched < matched_posA.size(); matched++)
            {
                int j = matched_posA[matched];
                for (int r = 0; r < tile_size; r++)
                {
                    for (int s = 0; s < tile_size; s++)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            matrix_C[tile_i * tile_size + r][tile_size * tile_j + s] += matrix_A[tile_size * tile_i + r][tile_size * j + t] * matrix_B[tile_size * j + t][tile_size * tile_j + s];
                        }
                    }
                }
            }
        }
        // Sparse Accumulator
        else
        {

            for (int matched = 0; matched < matched_posA.size(); matched++)
            {
                int jt = matched_posA[matched];
                for (int it = 0; it < C_ptr.size() - 1; ++it)
                {
                    int start = C_ptr[it];
                    int end = C_ptr[it + 1];
                    for (int j1 = start; j1 < end; ++j1)
                    {
                        int colC = idx[j1];
                        for (int r = 0; r < tile_size; r++)
                        {
                            for (int t = 0; t < tile_size; t++)
                            {
                                matrix_C[tile_i * tile_size + r][tile_size * tile_j + colC] += matrix_A[tile_size * tile_i + r][tile_size * jt + t] * matrix_B[tile_size * jt + t][tile_size * tile_j + colC];
                            }
                        }
                    }
                }
            }
        }
    }
    return matrix_C;
}

tuple<vector<int>, vector<int>> highLevel_conversion(vector<vector<int>> &matrix, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix.size(); i += tile_size)
    {
        for (int j = 0; j < matrix[i].size(); j += tile_size)
        {
            for (int k = 0; k < tile_size * tile_size; k++)
            {
                int row = k / tile_size;
                int col = k % tile_size;
                if (matrix[i + row][j + col] != 0)
                {
                    colIdx.push_back(j / tile_size);
                    break;
                }
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}

tuple<vector<int>, vector<int>> highLevel_conversion_For_MatrixB(vector<int> rowPtr, vector<int> Colidx)
{
    vector<int> colPtr(rowPtr.size(), 0);
    vector<int> rowIdx(Colidx.size(), 0);
    int max = 0;

    for (int i = 0; i < rowPtr.size() - 1; i++)
    {
        for (int j = rowPtr[i]; j < rowPtr[i + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < colPtr.size(); k++)
            {
                colPtr[k]++;
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < max; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = i;
            max++;
        }
    }
    for (int i = 0; i < colPtr.size() - 1; i++)
    {
        sort(rowIdx.begin() + colPtr[i], rowIdx.begin() + colPtr[i + 1]);
    }
    return make_tuple(colPtr, rowIdx);
}
int main() {
// Parameters
    vector<int> tile_sizes = {32};
    vector<int> matrix_sizes = {128, 256, 384, 512, 640, 768, 896, 1024, 1152, 1280, 1408, 1536, 1664, 1792, 1920, 2048, 2176, 2304, 2432, 2560, 2688, 2816, 2944, 3072};
    vector<double>time;

    double density = 0.1;

    // Loop over tile sizes
    for (int tile_size : tile_sizes) {
        cout << "Tile Size: " << tile_size << endl;

        // Loop over matrix sizes
        for (int matrix_size : matrix_sizes) {
            cout << "Matrix Size: " << matrix_size << "x" << matrix_size << endl;

            // Generate random sparse matrices A and B
            vector<vector<int>> matrix_A = generateRandomSparseMatrix(matrix_size, matrix_size, density);
            vector<vector<int>> matrix_B = generateRandomSparseMatrix(matrix_size, matrix_size, density);

            // Start measuring time
            struct timeval begin, end;
            gettimeofday(&begin, 0);

            // High-level conversion for matrix A
            vector<int> tilerowPtr_A, tileColidx_A;
            tie(tilerowPtr_A, tileColidx_A) = highLevel_conversion(matrix_A, tile_size);

            // High-level conversion for matrix B
            vector<int> tilerowPtr_B, tileColidx_B;
            tie(tilerowPtr_B, tileColidx_B) = highLevel_conversion(matrix_B, tile_size);

            // Symbolic SpGEMM for CSR Format
            vector<int> tileRowidx_C, tileColidx_C;
            tie(tileRowidx_C, tileColidx_C) = symbolic_SpGEMM_For_CSRFormat(tilerowPtr_A, tileColidx_A, tilerowPtr_B, tileColidx_B);

            int numtileC = tileRowidx_C.size();
            vector<int> tileColPtr_B, tileRowidx_B;
            tie(tileColPtr_B, tileRowidx_B) = highLevel_conversion_For_MatrixB(tilerowPtr_B, tileColidx_B);
            vector<vector<int>> matrix_C = step2_and_step3(matrix_A, matrix_B, tileRowidx_C, tileColidx_C, tilerowPtr_A, tilerowPtr_B, tileColPtr_B, tileColidx_A, tileRowidx_B, numtileC, tile_size);

            gettimeofday(&end, 0);
            long seconds = end.tv_sec - begin.tv_sec;
            long microseconds = end.tv_usec - begin.tv_usec;
            double elapsed = seconds + microseconds*1e-6;
            time.push_back(elapsed);
            printf("Time measured: %.6f seconds.\n", elapsed);
        }
        cout << "Matrix time:" << endl;
        for (const auto& row : time) {
            printf("%.6f, ", row);
        }
        printf("\n");

    }
    return 0;
}

Tile Size: 32
Matrix Size: 128x128
Time measured: 0.025158 seconds.
Matrix Size: 256x256
Time measured: 0.094366 seconds.
Matrix Size: 384x384
Time measured: 0.218950 seconds.
Matrix Size: 512x512
Time measured: 0.385677 seconds.
Matrix Size: 640x640
Time measured: 0.611608 seconds.
Matrix Size: 768x768
Time measured: 0.850229 seconds.
Matrix Size: 896x896
Time measured: 1.148502 seconds.
Matrix Size: 1024x1024
Time measured: 1.503612 seconds.
Matrix Size: 1152x1152
Time measured: 1.931102 seconds.
Matrix Size: 1280x1280
Time measured: 3.398421 seconds.
Matrix Size: 1408x1408
Time measured: 2.853113 seconds.
Matrix Size: 1536x1536
Time measured: 3.409430 seconds.
Matrix Size: 1664x1664
Time measured: 4.968940 seconds.
Matrix Size: 1792x1792
Time measured: 4.660295 seconds.
Matrix Size: 1920x1920
Time measured: 6.059323 seconds.
Matrix Size: 2048x2048
Time measured: 6.253882 seconds.
Matrix Size: 2176x2176
Time measured: 8.377581 seconds.
Matrix Size: 2304x2304
Time measured: 9.140334 s

### Tile: 16

In [ ]:
%%cuda
#include <random>
#include<bits/stdc++.h>
#include <algorithm>
#include <vector>
#include <iostream>
#include <bits/stdc++.h>
#include <sys/time.h>


using namespace std;
vector<vector<int>> generateRandomSparseMatrix(int rows, int cols, double density) {
    vector<vector<int>> matrix(rows, vector<int>(cols, 0));
    std::random_device rd;
    std::mt19937 gen(rd());
    std::uniform_real_distribution<> dis(0.0, 1.0);
    for (int i = 0; i < rows; ++i) {
        for (int j = 0; j < cols; ++j) {
            double randNum = dis(gen);
            if (randNum <= density) {
                matrix[i][j] = 1;
            }
        }
    }
    return matrix;
}

tuple<vector<int>, vector<int>> symbolic_SpGEMM_For_CSRFormat(vector<int> tilerowPtr_A, vector<int> tileColidx_A, vector<int> tilerowPtr_B, vector<int> tileColidx_B)
{
    int n = tilerowPtr_A.size();
    int m = tilerowPtr_B.size();
    vector<vector<int>> C_prime;
    vector<int> rowIdx, colIdx;

    for (int i = 0; i < n - 1; ++i)
    {
        set<int> unique_col;
        for (int j = tilerowPtr_A[i]; j < tilerowPtr_A[i + 1]; ++j)
        {
            int col = tileColidx_A[j];
            for (int i1 = tilerowPtr_B[col]; i1 < tilerowPtr_B[col + 1]; ++i1)
            {
                int col1 = tileColidx_B[i1];
                unique_col.insert(col1);
            }
        }
        for (int col : unique_col)
        {
            colIdx.push_back(col);
            rowIdx.push_back(i);
        }
    }
    C_prime.push_back(rowIdx);
    C_prime.push_back(colIdx);
    return make_tuple(rowIdx, colIdx);
}

vector<vector<int>> bitmask_conversion(vector<vector<int>> &matrix)
{
    vector<vector<int>> matrix_mask_B(matrix.size(), vector<int>(matrix[0].size(), 0));
    for (int i = 0; i < matrix.size(); i++)
    {
        for (int j = 0; j < matrix[i].size(); j++)
        {
            if (matrix[i][j] != 0)
            {
                matrix_mask_B[i][j] = 1;
            }
        }
    }
    return matrix_mask_B;
}

vector<vector<int>> step2_and_step3(vector<vector<int>> matrix_A, vector<vector<int>> matrix_B, vector<int> &tileRowidx_C, vector<int> &tileColidx_C, vector<int> &tilePtr_A,
                                    vector<int> &tilePtr_B, vector<int> &tileColPtr_B, vector<int> &tileColidx_A,
                                    vector<int> &tileRowidx_B, int numtileC, int tile_size)
{

    vector<vector<int>> matrix_C(matrix_A.size(), vector<int>(matrix_B[0].size(), 0));
    vector<vector<int>> maskc(matrix_A.size(), vector<int>(matrix_B[0].size(), 0));
    vector<vector<int>> matrix_mask_B = bitmask_conversion(matrix_B);
    for (int i = 0; i < numtileC; i++)
    {
        vector<int> matched_posA;
        int tile_i = tileRowidx_C[i];
        int tile_j = tileColidx_C[i];
        int lena = tilePtr_A[tile_i + 1] - tilePtr_A[tile_i];
        int lenb = tileColPtr_B[tile_j + 1] - tileColPtr_B[tile_j];
        // Applying Binary Search to find the matched positions
        if (lena <= lenb)
        {
            for (int value = tilePtr_A[tile_i]; value < tilePtr_A[tile_i + 1]; value++)
            {
                int low = tileColPtr_B[tile_j];
                int high = tileColPtr_B[tile_j + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileRowidx_B[mid] == tileColidx_A[value])
                    {
                        matched_posA.push_back(tileColidx_A[value]);
                        break;
                    }
                    else if (tileRowidx_B[mid] < tileColidx_A[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }
        else
        {
            for (int value = tileColPtr_B[tile_j]; value < tileColPtr_B[tile_j + 1]; value++)
            {
                int low = tilePtr_A[tile_i];
                int high = tilePtr_A[tile_i + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileColidx_A[mid] == tileRowidx_B[value])
                    {
                        matched_posA.push_back(tileRowidx_B[value]);
                        break;
                    }
                    else if (tileColidx_A[mid] < tileRowidx_B[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }

        // Calculaitng the mask of the each tile using the matched positions

        for (int matched = 0; matched < matched_posA.size(); matched++)
        {
            int j = matched_posA[matched];
            for (int r = 0; r < tile_size; r++)
            {
                for (int s = 0; s < tile_size; s++)
                {

                    int row = tile_size * tile_i + r;
                    int col = tile_size * j + s;
                    if (matrix_A[row][col] != 0)
                    {

                        for (int t = 0; t < tile_size; t++)
                        {
                            maskc[row][tile_size * tile_j + t] = maskc[row][tile_size * tile_j + t] || matrix_mask_B[tile_size * j + s][tile_size * tile_j + t];
                        }
                    }
                }
            }
        }

        // Calculating the number of non-zeros values in each tile using the maskc
        int nnz = 0;
        for (int i = 0; i < tile_size; i++)
        {
            for (int j = 0; j < tile_size; j++)
            {
                if (maskc[tile_i * tile_size + i][tile_j * tile_size + j] != 0)
                {
                    nnz++;
                }
            }
        }

        // Step 3: For Calculating the matrix C using the number of non-zero values in each tile
        vector<int> idx, C_ptr;
        C_ptr.push_back(0);

        int sizee = 0;
        for (int it = 0; it < tile_size; it++)
        {
            for (int jt = 0; jt < tile_size; jt++)
            {
                if (maskc[tile_i * tile_size + it][tile_j * tile_size + jt] != 0)
                {
                    nnz++;
                    idx.push_back(jt);
                }
            }
            if (sizee < idx.size())
            {
                C_ptr.push_back(idx.size());
                sizee = idx.size();
            }
        }
        // Dense Accumulator
        int threshold = 0.75*tile_size*tile_size;
        if (nnz >= threshold)
        {
            for (int matched = 0; matched < matched_posA.size(); matched++)
            {
                int j = matched_posA[matched];
                for (int r = 0; r < tile_size; r++)
                {
                    for (int s = 0; s < tile_size; s++)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            matrix_C[tile_i * tile_size + r][tile_size * tile_j + s] += matrix_A[tile_size * tile_i + r][tile_size * j + t] * matrix_B[tile_size * j + t][tile_size * tile_j + s];
                        }
                    }
                }
            }
        }
        // Sparse Accumulator
        else
        {

            for (int matched = 0; matched < matched_posA.size(); matched++)
            {
                int jt = matched_posA[matched];
                for (int it = 0; it < C_ptr.size() - 1; ++it)
                {
                    int start = C_ptr[it];
                    int end = C_ptr[it + 1];
                    for (int j1 = start; j1 < end; ++j1)
                    {
                        int colC = idx[j1];
                        for (int r = 0; r < tile_size; r++)
                        {
                            for (int t = 0; t < tile_size; t++)
                            {
                                matrix_C[tile_i * tile_size + r][tile_size * tile_j + colC] += matrix_A[tile_size * tile_i + r][tile_size * jt + t] * matrix_B[tile_size * jt + t][tile_size * tile_j + colC];
                            }
                        }
                    }
                }
            }
        }
    }
    return matrix_C;
}

tuple<vector<int>, vector<int>> highLevel_conversion(vector<vector<int>> &matrix, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix.size(); i += tile_size)
    {
        for (int j = 0; j < matrix[i].size(); j += tile_size)
        {
            for (int k = 0; k < tile_size * tile_size; k++)
            {
                int row = k / tile_size;
                int col = k % tile_size;
                if (matrix[i + row][j + col] != 0)
                {
                    colIdx.push_back(j / tile_size);
                    break;
                }
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}

tuple<vector<int>, vector<int>> highLevel_conversion_For_MatrixB(vector<int> rowPtr, vector<int> Colidx)
{
    vector<int> colPtr(rowPtr.size(), 0);
    vector<int> rowIdx(Colidx.size(), 0);
    int max = 0;

    for (int i = 0; i < rowPtr.size() - 1; i++)
    {
        for (int j = rowPtr[i]; j < rowPtr[i + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < colPtr.size(); k++)
            {
                colPtr[k]++;
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < max; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = i;
            max++;
        }
    }
    for (int i = 0; i < colPtr.size() - 1; i++)
    {
        sort(rowIdx.begin() + colPtr[i], rowIdx.begin() + colPtr[i + 1]);
    }
    return make_tuple(colPtr, rowIdx);
}
int main() {
// Parameters
    vector<int> tile_sizes = {16};
    vector<int> matrix_sizes = {128, 256, 384, 512, 640, 768, 896, 1024, 1152, 1280, 1408, 1536, 1664, 1792, 1920, 2048, 2176, 2304, 2432, 2560, 2688, 2816, 2944, 3072};
    vector<double>time;

    double density = 0.1;

    // Loop over tile sizes
    for (int tile_size : tile_sizes) {
        cout << "Tile Size: " << tile_size << endl;

        // Loop over matrix sizes
        for (int matrix_size : matrix_sizes) {
            cout << "Matrix Size: " << matrix_size << "x" << matrix_size << endl;

            // Generate random sparse matrices A and B
            vector<vector<int>> matrix_A = generateRandomSparseMatrix(matrix_size, matrix_size, density);
            vector<vector<int>> matrix_B = generateRandomSparseMatrix(matrix_size, matrix_size, density);

            // Start measuring time
            struct timeval begin, end;
            gettimeofday(&begin, 0);

            // High-level conversion for matrix A
            vector<int> tilerowPtr_A, tileColidx_A;
            tie(tilerowPtr_A, tileColidx_A) = highLevel_conversion(matrix_A, tile_size);

            // High-level conversion for matrix B
            vector<int> tilerowPtr_B, tileColidx_B;
            tie(tilerowPtr_B, tileColidx_B) = highLevel_conversion(matrix_B, tile_size);

            // Symbolic SpGEMM for CSR Format
            vector<int> tileRowidx_C, tileColidx_C;
            tie(tileRowidx_C, tileColidx_C) = symbolic_SpGEMM_For_CSRFormat(tilerowPtr_A, tileColidx_A, tilerowPtr_B, tileColidx_B);

            int numtileC = tileRowidx_C.size();
            vector<int> tileColPtr_B, tileRowidx_B;
            tie(tileColPtr_B, tileRowidx_B) = highLevel_conversion_For_MatrixB(tilerowPtr_B, tileColidx_B);
            vector<vector<int>> matrix_C = step2_and_step3(matrix_A, matrix_B, tileRowidx_C, tileColidx_C, tilerowPtr_A, tilerowPtr_B, tileColPtr_B, tileColidx_A, tileRowidx_B, numtileC, tile_size);

            gettimeofday(&end, 0);
            long seconds = end.tv_sec - begin.tv_sec;
            long microseconds = end.tv_usec - begin.tv_usec;
            double elapsed = seconds + microseconds*1e-6;
            time.push_back(elapsed);
            printf("Time measured: %.6f seconds.\n", elapsed);
        }
        cout << "Matrix time:" << endl;
        for (const auto& row : time) {
            printf("%.6f, ", row);
        }
        printf("\n");

    }
    return 0;
}

Tile Size: 16
Matrix Size: 128x128
Time measured: 0.146898 seconds.
Matrix Size: 256x256
Time measured: 0.348177 seconds.
Matrix Size: 384x384
Time measured: 0.400042 seconds.
Matrix Size: 512x512
Time measured: 0.728329 seconds.
Matrix Size: 640x640
Time measured: 1.322073 seconds.
Matrix Size: 768x768
Time measured: 2.495736 seconds.
Matrix Size: 896x896
Time measured: 2.289316 seconds.
Matrix Size: 1024x1024
Time measured: 3.008748 seconds.
Matrix Size: 1152x1152
Time measured: 3.881276 seconds.
Matrix Size: 1280x1280
Time measured: 5.392803 seconds.
Matrix Size: 1408x1408
Time measured: 5.745771 seconds.
Matrix Size: 1536x1536
Time measured: 7.977270 seconds.
Matrix Size: 1664x1664
Time measured: 9.177186 seconds.
Matrix Size: 1792x1792
Time measured: 10.426426 seconds.
Matrix Size: 1920x1920
Time measured: 11.961312 seconds.
Matrix Size: 2048x2048
Time measured: 13.444389 seconds.
Matrix Size: 2176x2176
Time measured: 16.508205 seconds.
Matrix Size: 2304x2304
Time measured: 18.239

### Tile: 8

In [ ]:
%%cuda
#include <random>
#include<bits/stdc++.h>
#include <algorithm>
#include <vector>
#include <iostream>
#include <bits/stdc++.h>
#include <sys/time.h>


using namespace std;
vector<vector<int>> generateRandomSparseMatrix(int rows, int cols, double density) {
    vector<vector<int>> matrix(rows, vector<int>(cols, 0));
    std::random_device rd;
    std::mt19937 gen(rd());
    std::uniform_real_distribution<> dis(0.0, 1.0);
    for (int i = 0; i < rows; ++i) {
        for (int j = 0; j < cols; ++j) {
            double randNum = dis(gen);
            if (randNum <= density) {
                matrix[i][j] = 1;
            }
        }
    }
    return matrix;
}

tuple<vector<int>, vector<int>> symbolic_SpGEMM_For_CSRFormat(vector<int> tilerowPtr_A, vector<int> tileColidx_A, vector<int> tilerowPtr_B, vector<int> tileColidx_B)
{
    int n = tilerowPtr_A.size();
    int m = tilerowPtr_B.size();
    vector<vector<int>> C_prime;
    vector<int> rowIdx, colIdx;

    for (int i = 0; i < n - 1; ++i)
    {
        set<int> unique_col;
        for (int j = tilerowPtr_A[i]; j < tilerowPtr_A[i + 1]; ++j)
        {
            int col = tileColidx_A[j];
            for (int i1 = tilerowPtr_B[col]; i1 < tilerowPtr_B[col + 1]; ++i1)
            {
                int col1 = tileColidx_B[i1];
                unique_col.insert(col1);
            }
        }
        for (int col : unique_col)
        {
            colIdx.push_back(col);
            rowIdx.push_back(i);
        }
    }
    C_prime.push_back(rowIdx);
    C_prime.push_back(colIdx);
    return make_tuple(rowIdx, colIdx);
}

vector<vector<int>> bitmask_conversion(vector<vector<int>> &matrix)
{
    vector<vector<int>> matrix_mask_B(matrix.size(), vector<int>(matrix[0].size(), 0));
    for (int i = 0; i < matrix.size(); i++)
    {
        for (int j = 0; j < matrix[i].size(); j++)
        {
            if (matrix[i][j] != 0)
            {
                matrix_mask_B[i][j] = 1;
            }
        }
    }
    return matrix_mask_B;
}

vector<vector<int>> step2_and_step3(vector<vector<int>> matrix_A, vector<vector<int>> matrix_B, vector<int> &tileRowidx_C, vector<int> &tileColidx_C, vector<int> &tilePtr_A,
                                    vector<int> &tilePtr_B, vector<int> &tileColPtr_B, vector<int> &tileColidx_A,
                                    vector<int> &tileRowidx_B, int numtileC, int tile_size)
{

    vector<vector<int>> matrix_C(matrix_A.size(), vector<int>(matrix_B[0].size(), 0));
    vector<vector<int>> maskc(matrix_A.size(), vector<int>(matrix_B[0].size(), 0));
    vector<vector<int>> matrix_mask_B = bitmask_conversion(matrix_B);
    for (int i = 0; i < numtileC; i++)
    {
        vector<int> matched_posA;
        int tile_i = tileRowidx_C[i];
        int tile_j = tileColidx_C[i];
        int lena = tilePtr_A[tile_i + 1] - tilePtr_A[tile_i];
        int lenb = tileColPtr_B[tile_j + 1] - tileColPtr_B[tile_j];
        // Applying Binary Search to find the matched positions
        if (lena <= lenb)
        {
            for (int value = tilePtr_A[tile_i]; value < tilePtr_A[tile_i + 1]; value++)
            {
                int low = tileColPtr_B[tile_j];
                int high = tileColPtr_B[tile_j + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileRowidx_B[mid] == tileColidx_A[value])
                    {
                        matched_posA.push_back(tileColidx_A[value]);
                        break;
                    }
                    else if (tileRowidx_B[mid] < tileColidx_A[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }
        else
        {
            for (int value = tileColPtr_B[tile_j]; value < tileColPtr_B[tile_j + 1]; value++)
            {
                int low = tilePtr_A[tile_i];
                int high = tilePtr_A[tile_i + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileColidx_A[mid] == tileRowidx_B[value])
                    {
                        matched_posA.push_back(tileRowidx_B[value]);
                        break;
                    }
                    else if (tileColidx_A[mid] < tileRowidx_B[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }

        // Calculaitng the mask of the each tile using the matched positions

        for (int matched = 0; matched < matched_posA.size(); matched++)
        {
            int j = matched_posA[matched];
            for (int r = 0; r < tile_size; r++)
            {
                for (int s = 0; s < tile_size; s++)
                {

                    int row = tile_size * tile_i + r;
                    int col = tile_size * j + s;
                    if (matrix_A[row][col] != 0)
                    {

                        for (int t = 0; t < tile_size; t++)
                        {
                            maskc[row][tile_size * tile_j + t] = maskc[row][tile_size * tile_j + t] || matrix_mask_B[tile_size * j + s][tile_size * tile_j + t];
                        }
                    }
                }
            }
        }

        // Calculating the number of non-zeros values in each tile using the maskc
        int nnz = 0;
        for (int i = 0; i < tile_size; i++)
        {
            for (int j = 0; j < tile_size; j++)
            {
                if (maskc[tile_i * tile_size + i][tile_j * tile_size + j] != 0)
                {
                    nnz++;
                }
            }
        }

        // Step 3: For Calculating the matrix C using the number of non-zero values in each tile
        vector<int> idx, C_ptr;
        C_ptr.push_back(0);

        int sizee = 0;
        for (int it = 0; it < tile_size; it++)
        {
            for (int jt = 0; jt < tile_size; jt++)
            {
                if (maskc[tile_i * tile_size + it][tile_j * tile_size + jt] != 0)
                {
                    nnz++;
                    idx.push_back(jt);
                }
            }
            if (sizee < idx.size())
            {
                C_ptr.push_back(idx.size());
                sizee = idx.size();
            }
        }
        // Dense Accumulator
        int threshold = 0.75*tile_size*tile_size;
        if (nnz >= threshold)
        {
            for (int matched = 0; matched < matched_posA.size(); matched++)
            {
                int j = matched_posA[matched];
                for (int r = 0; r < tile_size; r++)
                {
                    for (int s = 0; s < tile_size; s++)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            matrix_C[tile_i * tile_size + r][tile_size * tile_j + s] += matrix_A[tile_size * tile_i + r][tile_size * j + t] * matrix_B[tile_size * j + t][tile_size * tile_j + s];
                        }
                    }
                }
            }
        }
        // Sparse Accumulator
        else
        {

            for (int matched = 0; matched < matched_posA.size(); matched++)
            {
                int jt = matched_posA[matched];
                for (int it = 0; it < C_ptr.size() - 1; ++it)
                {
                    int start = C_ptr[it];
                    int end = C_ptr[it + 1];
                    for (int j1 = start; j1 < end; ++j1)
                    {
                        int colC = idx[j1];
                        for (int r = 0; r < tile_size; r++)
                        {
                            for (int t = 0; t < tile_size; t++)
                            {
                                matrix_C[tile_i * tile_size + r][tile_size * tile_j + colC] += matrix_A[tile_size * tile_i + r][tile_size * jt + t] * matrix_B[tile_size * jt + t][tile_size * tile_j + colC];
                            }
                        }
                    }
                }
            }
        }
    }
    return matrix_C;
}

tuple<vector<int>, vector<int>> highLevel_conversion(vector<vector<int>> &matrix, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix.size(); i += tile_size)
    {
        for (int j = 0; j < matrix[i].size(); j += tile_size)
        {
            for (int k = 0; k < tile_size * tile_size; k++)
            {
                int row = k / tile_size;
                int col = k % tile_size;
                if (matrix[i + row][j + col] != 0)
                {
                    colIdx.push_back(j / tile_size);
                    break;
                }
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}

tuple<vector<int>, vector<int>> highLevel_conversion_For_MatrixB(vector<int> rowPtr, vector<int> Colidx)
{
    vector<int> colPtr(rowPtr.size(), 0);
    vector<int> rowIdx(Colidx.size(), 0);
    int max = 0;

    for (int i = 0; i < rowPtr.size() - 1; i++)
    {
        for (int j = rowPtr[i]; j < rowPtr[i + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < colPtr.size(); k++)
            {
                colPtr[k]++;
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < max; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = i;
            max++;
        }
    }
    for (int i = 0; i < colPtr.size() - 1; i++)
    {
        sort(rowIdx.begin() + colPtr[i], rowIdx.begin() + colPtr[i + 1]);
    }
    return make_tuple(colPtr, rowIdx);
}
int main() {
// Parameters
    vector<int> tile_sizes = {8};
    vector<int> matrix_sizes = {128, 256, 384, 512, 640, 768, 896, 1024, 1152, 1280, 1408, 1536, 1664, 1792, 1920, 2048, 2176, 2304, 2432, 2560, 2688, 2816, 2944, 3072};
    vector<double>time;

    double density = 0.1;

    // Loop over tile sizes
    for (int tile_size : tile_sizes) {
        cout << "Tile Size: " << tile_size << endl;

        // Loop over matrix sizes
        for (int matrix_size : matrix_sizes) {
            cout << "Matrix Size: " << matrix_size << "x" << matrix_size << endl;

            // Generate random sparse matrices A and B
            vector<vector<int>> matrix_A = generateRandomSparseMatrix(matrix_size, matrix_size, density);
            vector<vector<int>> matrix_B = generateRandomSparseMatrix(matrix_size, matrix_size, density);

            // Start measuring time
            struct timeval begin, end;
            gettimeofday(&begin, 0);

            // High-level conversion for matrix A
            vector<int> tilerowPtr_A, tileColidx_A;
            tie(tilerowPtr_A, tileColidx_A) = highLevel_conversion(matrix_A, tile_size);

            // High-level conversion for matrix B
            vector<int> tilerowPtr_B, tileColidx_B;
            tie(tilerowPtr_B, tileColidx_B) = highLevel_conversion(matrix_B, tile_size);

            // Symbolic SpGEMM for CSR Format
            vector<int> tileRowidx_C, tileColidx_C;
            tie(tileRowidx_C, tileColidx_C) = symbolic_SpGEMM_For_CSRFormat(tilerowPtr_A, tileColidx_A, tilerowPtr_B, tileColidx_B);

            int numtileC = tileRowidx_C.size();
            vector<int> tileColPtr_B, tileRowidx_B;
            tie(tileColPtr_B, tileRowidx_B) = highLevel_conversion_For_MatrixB(tilerowPtr_B, tileColidx_B);
            vector<vector<int>> matrix_C = step2_and_step3(matrix_A, matrix_B, tileRowidx_C, tileColidx_C, tilerowPtr_A, tilerowPtr_B, tileColPtr_B, tileColidx_A, tileRowidx_B, numtileC, tile_size);

            gettimeofday(&end, 0);
            long seconds = end.tv_sec - begin.tv_sec;
            long microseconds = end.tv_usec - begin.tv_usec;
            double elapsed = seconds + microseconds*1e-6;
            time.push_back(elapsed);
            printf("Time measured: %.6f seconds.\n", elapsed);
        }
        cout << "Matrix time:" << endl;
        for (const auto& row : time) {
            printf("%.6f, ", row);
        }
        printf("\n");

    }
    return 0;
}

Tile Size: 8
Matrix Size: 128x128
Time measured: 0.020999 seconds.
Matrix Size: 256x256
Time measured: 0.090685 seconds.
Matrix Size: 384x384
Time measured: 0.271941 seconds.
Matrix Size: 512x512
Time measured: 0.787873 seconds.
Matrix Size: 640x640
Time measured: 1.060492 seconds.
Matrix Size: 768x768
Time measured: 1.819662 seconds.
Matrix Size: 896x896
Time measured: 3.349716 seconds.
Matrix Size: 1024x1024
Time measured: 6.304825 seconds.
Matrix Size: 1152x1152
Time measured: 7.261535 seconds.
Matrix Size: 1280x1280
Time measured: 12.722885 seconds.
Matrix Size: 1408x1408
Time measured: 16.906185 seconds.
Matrix Size: 1536x1536
Time measured: 20.096244 seconds.
Matrix Size: 1664x1664
Time measured: 30.126264 seconds.
Matrix Size: 1792x1792
Time measured: 41.373452 seconds.
Matrix Size: 1920x1920
Time measured: 50.104687 seconds.
Matrix Size: 2048x2048
Time measured: 54.183404 seconds.
Matrix Size: 2176x2176
Time measured: 93.530893 seconds.
Matrix Size: 2304x2304
Time measured: 102

### Tile: 4

In [ ]:
%%cuda
#include <random>
#include<bits/stdc++.h>
#include <algorithm>
#include <vector>
#include <iostream>
#include <bits/stdc++.h>
#include <sys/time.h>


using namespace std;
vector<vector<int>> generateRandomSparseMatrix(int rows, int cols, double density) {
    vector<vector<int>> matrix(rows, vector<int>(cols, 0));
    std::random_device rd;
    std::mt19937 gen(rd());
    std::uniform_real_distribution<> dis(0.0, 1.0);
    for (int i = 0; i < rows; ++i) {
        for (int j = 0; j < cols; ++j) {
            double randNum = dis(gen);
            if (randNum <= density) {
                matrix[i][j] = 1;
            }
        }
    }
    return matrix;
}

tuple<vector<int>, vector<int>> symbolic_SpGEMM_For_CSRFormat(vector<int> tilerowPtr_A, vector<int> tileColidx_A, vector<int> tilerowPtr_B, vector<int> tileColidx_B)
{
    int n = tilerowPtr_A.size();
    int m = tilerowPtr_B.size();
    vector<vector<int>> C_prime;
    vector<int> rowIdx, colIdx;

    for (int i = 0; i < n - 1; ++i)
    {
        set<int> unique_col;
        for (int j = tilerowPtr_A[i]; j < tilerowPtr_A[i + 1]; ++j)
        {
            int col = tileColidx_A[j];
            for (int i1 = tilerowPtr_B[col]; i1 < tilerowPtr_B[col + 1]; ++i1)
            {
                int col1 = tileColidx_B[i1];
                unique_col.insert(col1);
            }
        }
        for (int col : unique_col)
        {
            colIdx.push_back(col);
            rowIdx.push_back(i);
        }
    }
    C_prime.push_back(rowIdx);
    C_prime.push_back(colIdx);
    return make_tuple(rowIdx, colIdx);
}

vector<vector<int>> bitmask_conversion(vector<vector<int>> &matrix)
{
    vector<vector<int>> matrix_mask_B(matrix.size(), vector<int>(matrix[0].size(), 0));
    for (int i = 0; i < matrix.size(); i++)
    {
        for (int j = 0; j < matrix[i].size(); j++)
        {
            if (matrix[i][j] != 0)
            {
                matrix_mask_B[i][j] = 1;
            }
        }
    }
    return matrix_mask_B;
}

vector<vector<int>> step2_and_step3(vector<vector<int>> matrix_A, vector<vector<int>> matrix_B, vector<int> &tileRowidx_C, vector<int> &tileColidx_C, vector<int> &tilePtr_A,
                                    vector<int> &tilePtr_B, vector<int> &tileColPtr_B, vector<int> &tileColidx_A,
                                    vector<int> &tileRowidx_B, int numtileC, int tile_size)
{

    vector<vector<int>> matrix_C(matrix_A.size(), vector<int>(matrix_B[0].size(), 0));
    vector<vector<int>> maskc(matrix_A.size(), vector<int>(matrix_B[0].size(), 0));
    vector<vector<int>> matrix_mask_B = bitmask_conversion(matrix_B);
    for (int i = 0; i < numtileC; i++)
    {
        vector<int> matched_posA;
        int tile_i = tileRowidx_C[i];
        int tile_j = tileColidx_C[i];
        int lena = tilePtr_A[tile_i + 1] - tilePtr_A[tile_i];
        int lenb = tileColPtr_B[tile_j + 1] - tileColPtr_B[tile_j];
        // Applying Binary Search to find the matched positions
        if (lena <= lenb)
        {
            for (int value = tilePtr_A[tile_i]; value < tilePtr_A[tile_i + 1]; value++)
            {
                int low = tileColPtr_B[tile_j];
                int high = tileColPtr_B[tile_j + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileRowidx_B[mid] == tileColidx_A[value])
                    {
                        matched_posA.push_back(tileColidx_A[value]);
                        break;
                    }
                    else if (tileRowidx_B[mid] < tileColidx_A[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }
        else
        {
            for (int value = tileColPtr_B[tile_j]; value < tileColPtr_B[tile_j + 1]; value++)
            {
                int low = tilePtr_A[tile_i];
                int high = tilePtr_A[tile_i + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileColidx_A[mid] == tileRowidx_B[value])
                    {
                        matched_posA.push_back(tileRowidx_B[value]);
                        break;
                    }
                    else if (tileColidx_A[mid] < tileRowidx_B[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }

        // Calculaitng the mask of the each tile using the matched positions

        for (int matched = 0; matched < matched_posA.size(); matched++)
        {
            int j = matched_posA[matched];
            for (int r = 0; r < tile_size; r++)
            {
                for (int s = 0; s < tile_size; s++)
                {

                    int row = tile_size * tile_i + r;
                    int col = tile_size * j + s;
                    if (matrix_A[row][col] != 0)
                    {

                        for (int t = 0; t < tile_size; t++)
                        {
                            maskc[row][tile_size * tile_j + t] = maskc[row][tile_size * tile_j + t] || matrix_mask_B[tile_size * j + s][tile_size * tile_j + t];
                        }
                    }
                }
            }
        }

        // Calculating the number of non-zeros values in each tile using the maskc
        int nnz = 0;
        for (int i = 0; i < tile_size; i++)
        {
            for (int j = 0; j < tile_size; j++)
            {
                if (maskc[tile_i * tile_size + i][tile_j * tile_size + j] != 0)
                {
                    nnz++;
                }
            }
        }

        // Step 3: For Calculating the matrix C using the number of non-zero values in each tile
        vector<int> idx, C_ptr;
        C_ptr.push_back(0);

        int sizee = 0;
        for (int it = 0; it < tile_size; it++)
        {
            for (int jt = 0; jt < tile_size; jt++)
            {
                if (maskc[tile_i * tile_size + it][tile_j * tile_size + jt] != 0)
                {
                    nnz++;
                    idx.push_back(jt);
                }
            }
            if (sizee < idx.size())
            {
                C_ptr.push_back(idx.size());
                sizee = idx.size();
            }
        }
        // Dense Accumulator
        int threshold = 0.75*tile_size*tile_size;
        if (nnz >= threshold)
        {
            for (int matched = 0; matched < matched_posA.size(); matched++)
            {
                int j = matched_posA[matched];
                for (int r = 0; r < tile_size; r++)
                {
                    for (int s = 0; s < tile_size; s++)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            matrix_C[tile_i * tile_size + r][tile_size * tile_j + s] += matrix_A[tile_size * tile_i + r][tile_size * j + t] * matrix_B[tile_size * j + t][tile_size * tile_j + s];
                        }
                    }
                }
            }
        }
        // Sparse Accumulator
        else
        {

            for (int matched = 0; matched < matched_posA.size(); matched++)
            {
                int jt = matched_posA[matched];
                for (int it = 0; it < C_ptr.size() - 1; ++it)
                {
                    int start = C_ptr[it];
                    int end = C_ptr[it + 1];
                    for (int j1 = start; j1 < end; ++j1)
                    {
                        int colC = idx[j1];
                        for (int r = 0; r < tile_size; r++)
                        {
                            for (int t = 0; t < tile_size; t++)
                            {
                                matrix_C[tile_i * tile_size + r][tile_size * tile_j + colC] += matrix_A[tile_size * tile_i + r][tile_size * jt + t] * matrix_B[tile_size * jt + t][tile_size * tile_j + colC];
                            }
                        }
                    }
                }
            }
        }
    }
    return matrix_C;
}

tuple<vector<int>, vector<int>> highLevel_conversion(vector<vector<int>> &matrix, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix.size(); i += tile_size)
    {
        for (int j = 0; j < matrix[i].size(); j += tile_size)
        {
            for (int k = 0; k < tile_size * tile_size; k++)
            {
                int row = k / tile_size;
                int col = k % tile_size;
                if (matrix[i + row][j + col] != 0)
                {
                    colIdx.push_back(j / tile_size);
                    break;
                }
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}

tuple<vector<int>, vector<int>> highLevel_conversion_For_MatrixB(vector<int> rowPtr, vector<int> Colidx)
{
    vector<int> colPtr(rowPtr.size(), 0);
    vector<int> rowIdx(Colidx.size(), 0);
    int max = 0;

    for (int i = 0; i < rowPtr.size() - 1; i++)
    {
        for (int j = rowPtr[i]; j < rowPtr[i + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < colPtr.size(); k++)
            {
                colPtr[k]++;
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < max; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = i;
            max++;
        }
    }
    for (int i = 0; i < colPtr.size() - 1; i++)
    {
        sort(rowIdx.begin() + colPtr[i], rowIdx.begin() + colPtr[i + 1]);
    }
    return make_tuple(colPtr, rowIdx);
}
int main() {
// Parameters
    vector<int> tile_sizes = {4};
    vector<int> matrix_sizes = {128, 256, 384, 512, 640, 768, 896, 1024, 1152, 1280, 1408, 1536, 1664, 1792, 1920, 2048, 2176, 2304, 2432, 2560, 2688, 2816, 2944, 3072};
    vector<double>time;

    double density = 0.1;

    // Loop over tile sizes
    for (int tile_size : tile_sizes) {
        cout << "Tile Size: " << tile_size << endl;

        // Loop over matrix sizes
        for (int matrix_size : matrix_sizes) {
            cout << "Matrix Size: " << matrix_size << "x" << matrix_size << endl;

            // Generate random sparse matrices A and B
            vector<vector<int>> matrix_A = generateRandomSparseMatrix(matrix_size, matrix_size, density);
            vector<vector<int>> matrix_B = generateRandomSparseMatrix(matrix_size, matrix_size, density);

            // Start measuring time
            struct timeval begin, end;
            gettimeofday(&begin, 0);

            // High-level conversion for matrix A
            vector<int> tilerowPtr_A, tileColidx_A;
            tie(tilerowPtr_A, tileColidx_A) = highLevel_conversion(matrix_A, tile_size);

            // High-level conversion for matrix B
            vector<int> tilerowPtr_B, tileColidx_B;
            tie(tilerowPtr_B, tileColidx_B) = highLevel_conversion(matrix_B, tile_size);

            // Symbolic SpGEMM for CSR Format
            vector<int> tileRowidx_C, tileColidx_C;
            tie(tileRowidx_C, tileColidx_C) = symbolic_SpGEMM_For_CSRFormat(tilerowPtr_A, tileColidx_A, tilerowPtr_B, tileColidx_B);

            int numtileC = tileRowidx_C.size();
            vector<int> tileColPtr_B, tileRowidx_B;
            tie(tileColPtr_B, tileRowidx_B) = highLevel_conversion_For_MatrixB(tilerowPtr_B, tileColidx_B);
            vector<vector<int>> matrix_C = step2_and_step3(matrix_A, matrix_B, tileRowidx_C, tileColidx_C, tilerowPtr_A, tilerowPtr_B, tileColPtr_B, tileColidx_A, tileRowidx_B, numtileC, tile_size);

            gettimeofday(&end, 0);
            long seconds = end.tv_sec - begin.tv_sec;
            long microseconds = end.tv_usec - begin.tv_usec;
            double elapsed = seconds + microseconds*1e-6;
            time.push_back(elapsed);
            printf("Time measured: %.6f seconds.\n", elapsed);
        }
        cout << "Matrix time:" << endl;
        for (const auto& row : time) {
            printf("%.6f, ", row);
        }
        printf("\n");

    }
    return 0;
}

Tile Size: 4
Matrix Size: 128x128
Time measured: 0.019388 seconds.
Matrix Size: 256x256
Time measured: 0.152447 seconds.
Matrix Size: 384x384
Time measured: 0.552368 seconds.
Matrix Size: 512x512
Time measured: 1.429607 seconds.
Matrix Size: 640x640
Time measured: 2.823004 seconds.
Matrix Size: 768x768
Time measured: 6.013426 seconds.
Matrix Size: 896x896
Time measured: 8.713862 seconds.
Matrix Size: 1024x1024
Time measured: 14.360755 seconds.
Matrix Size: 1152x1152
Time measured: 20.135432 seconds.
Matrix Size: 1280x1280
Time measured: 30.780368 seconds.
Matrix Size: 1408x1408
Time measured: 41.393190 seconds.
Matrix Size: 1536x1536
Time measured: 55.758008 seconds.
Matrix Size: 1664x1664
Time measured: 74.156305 seconds.
Matrix Size: 1792x1792
Time measured: 93.525556 seconds.
Matrix Size: 1920x1920
Time measured: 121.952535 seconds.
Matrix Size: 2048x2048
Time measured: 152.054838 seconds.
Matrix Size: 2176x2176
Time measured: 185.100652 seconds.
Matrix Size: 2304x2304
Time measured

## Density: 0.25

### Tile: 32

In [ ]:
%%cuda
#include <random>
#include<bits/stdc++.h>
#include <algorithm>
#include <vector>
#include <iostream>
#include <bits/stdc++.h>
#include <sys/time.h>


using namespace std;
vector<vector<int>> generateRandomSparseMatrix(int rows, int cols, double density) {
    vector<vector<int>> matrix(rows, vector<int>(cols, 0));
    std::random_device rd;
    std::mt19937 gen(rd());
    std::uniform_real_distribution<> dis(0.0, 1.0);
    for (int i = 0; i < rows; ++i) {
        for (int j = 0; j < cols; ++j) {
            double randNum = dis(gen);
            if (randNum <= density) {
                matrix[i][j] = 1;
            }
        }
    }
    return matrix;
}

tuple<vector<int>, vector<int>> symbolic_SpGEMM_For_CSRFormat(vector<int> tilerowPtr_A, vector<int> tileColidx_A, vector<int> tilerowPtr_B, vector<int> tileColidx_B)
{
    int n = tilerowPtr_A.size();
    int m = tilerowPtr_B.size();
    vector<vector<int>> C_prime;
    vector<int> rowIdx, colIdx;

    for (int i = 0; i < n - 1; ++i)
    {
        set<int> unique_col;
        for (int j = tilerowPtr_A[i]; j < tilerowPtr_A[i + 1]; ++j)
        {
            int col = tileColidx_A[j];
            for (int i1 = tilerowPtr_B[col]; i1 < tilerowPtr_B[col + 1]; ++i1)
            {
                int col1 = tileColidx_B[i1];
                unique_col.insert(col1);
            }
        }
        for (int col : unique_col)
        {
            colIdx.push_back(col);
            rowIdx.push_back(i);
        }
    }
    C_prime.push_back(rowIdx);
    C_prime.push_back(colIdx);
    return make_tuple(rowIdx, colIdx);
}

vector<vector<int>> bitmask_conversion(vector<vector<int>> &matrix)
{
    vector<vector<int>> matrix_mask_B(matrix.size(), vector<int>(matrix[0].size(), 0));
    for (int i = 0; i < matrix.size(); i++)
    {
        for (int j = 0; j < matrix[i].size(); j++)
        {
            if (matrix[i][j] != 0)
            {
                matrix_mask_B[i][j] = 1;
            }
        }
    }
    return matrix_mask_B;
}

vector<vector<int>> step2_and_step3(vector<vector<int>> matrix_A, vector<vector<int>> matrix_B, vector<int> &tileRowidx_C, vector<int> &tileColidx_C, vector<int> &tilePtr_A,
                                    vector<int> &tilePtr_B, vector<int> &tileColPtr_B, vector<int> &tileColidx_A,
                                    vector<int> &tileRowidx_B, int numtileC, int tile_size)
{

    vector<vector<int>> matrix_C(matrix_A.size(), vector<int>(matrix_B[0].size(), 0));
    vector<vector<int>> maskc(matrix_A.size(), vector<int>(matrix_B[0].size(), 0));
    vector<vector<int>> matrix_mask_B = bitmask_conversion(matrix_B);
    for (int i = 0; i < numtileC; i++)
    {
        vector<int> matched_posA;
        int tile_i = tileRowidx_C[i];
        int tile_j = tileColidx_C[i];
        int lena = tilePtr_A[tile_i + 1] - tilePtr_A[tile_i];
        int lenb = tileColPtr_B[tile_j + 1] - tileColPtr_B[tile_j];
        // Applying Binary Search to find the matched positions
        if (lena <= lenb)
        {
            for (int value = tilePtr_A[tile_i]; value < tilePtr_A[tile_i + 1]; value++)
            {
                int low = tileColPtr_B[tile_j];
                int high = tileColPtr_B[tile_j + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileRowidx_B[mid] == tileColidx_A[value])
                    {
                        matched_posA.push_back(tileColidx_A[value]);
                        break;
                    }
                    else if (tileRowidx_B[mid] < tileColidx_A[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }
        else
        {
            for (int value = tileColPtr_B[tile_j]; value < tileColPtr_B[tile_j + 1]; value++)
            {
                int low = tilePtr_A[tile_i];
                int high = tilePtr_A[tile_i + 1] - 1;
                while (low <= high)
                {
                    int mid = low + (high - low) / 2;
                    if (tileColidx_A[mid] == tileRowidx_B[value])
                    {
                        matched_posA.push_back(tileRowidx_B[value]);
                        break;
                    }
                    else if (tileColidx_A[mid] < tileRowidx_B[value])
                    {
                        low = mid + 1;
                    }
                    else
                    {
                        high = mid - 1;
                    }
                }
            }
        }

        // Calculaitng the mask of the each tile using the matched positions

        for (int matched = 0; matched < matched_posA.size(); matched++)
        {
            int j = matched_posA[matched];
            for (int r = 0; r < tile_size; r++)
            {
                for (int s = 0; s < tile_size; s++)
                {

                    int row = tile_size * tile_i + r;
                    int col = tile_size * j + s;
                    if (matrix_A[row][col] != 0)
                    {

                        for (int t = 0; t < tile_size; t++)
                        {
                            maskc[row][tile_size * tile_j + t] = maskc[row][tile_size * tile_j + t] || matrix_mask_B[tile_size * j + s][tile_size * tile_j + t];
                        }
                    }
                }
            }
        }

        // Calculating the number of non-zeros values in each tile using the maskc
        int nnz = 0;
        for (int i = 0; i < tile_size; i++)
        {
            for (int j = 0; j < tile_size; j++)
            {
                if (maskc[tile_i * tile_size + i][tile_j * tile_size + j] != 0)
                {
                    nnz++;
                }
            }
        }
        printf("%d", nnz);
        // Step 3: For Calculating the matrix C using the number of non-zero values in each tile
        vector<int> idx, C_ptr;
        C_ptr.push_back(0);

        int sizee = 0;
        for (int it = 0; it < tile_size; it++)
        {
            for (int jt = 0; jt < tile_size; jt++)
            {
                if (maskc[tile_i * tile_size + it][tile_j * tile_size + jt] != 0)
                {
                    nnz++;
                    idx.push_back(jt);
                }
            }
            if (sizee < idx.size())
            {
                C_ptr.push_back(idx.size());
                sizee = idx.size();
            }
        }
        // Dense Accumulator
        int threshold = 0.75*tile_size*tile_size;
        if (nnz >= threshold)
        {
            for (int matched = 0; matched < matched_posA.size(); matched++)
            {
                int j = matched_posA[matched];
                for (int r = 0; r < tile_size; r++)
                {
                    for (int s = 0; s < tile_size; s++)
                    {
                        for (int t = 0; t < tile_size; t++)
                        {
                            matrix_C[tile_i * tile_size + r][tile_size * tile_j + s] += matrix_A[tile_size * tile_i + r][tile_size * j + t] * matrix_B[tile_size * j + t][tile_size * tile_j + s];
                        }
                    }
                }
            }
        }
        // Sparse Accumulator
        else
        {

            for (int matched = 0; matched < matched_posA.size(); matched++)
            {
                int jt = matched_posA[matched];
                for (int it = 0; it < C_ptr.size() - 1; ++it)
                {
                    int start = C_ptr[it];
                    int end = C_ptr[it + 1];
                    for (int j1 = start; j1 < end; ++j1)
                    {
                        int colC = idx[j1];
                        for (int r = 0; r < tile_size; r++)
                        {
                            for (int t = 0; t < tile_size; t++)
                            {
                                matrix_C[tile_i * tile_size + r][tile_size * tile_j + colC] += matrix_A[tile_size * tile_i + r][tile_size * jt + t] * matrix_B[tile_size * jt + t][tile_size * tile_j + colC];
                            }
                        }
                    }
                }
            }
        }
    }
    return matrix_C;
}

tuple<vector<int>, vector<int>> highLevel_conversion(vector<vector<int>> &matrix, int tile_size)
{
    vector<int> rowPtr = {0};
    vector<int> colIdx;

    for (int i = 0; i < matrix.size(); i += tile_size)
    {
        for (int j = 0; j < matrix[i].size(); j += tile_size)
        {
            for (int k = 0; k < tile_size * tile_size; k++)
            {
                int row = k / tile_size;
                int col = k % tile_size;
                if (matrix[i + row][j + col] != 0)
                {
                    colIdx.push_back(j / tile_size);
                    break;
                }
            }
        }
        rowPtr.push_back(colIdx.size());
    }
    return make_tuple(rowPtr, colIdx);
}

tuple<vector<int>, vector<int>> highLevel_conversion_For_MatrixB(vector<int> rowPtr, vector<int> Colidx)
{
    vector<int> colPtr(rowPtr.size(), 0);
    vector<int> rowIdx(Colidx.size(), 0);
    int max = 0;

    for (int i = 0; i < rowPtr.size() - 1; i++)
    {
        for (int j = rowPtr[i]; j < rowPtr[i + 1]; j++)
        {
            int col = Colidx[j];
            for (int k = col + 1; k < colPtr.size(); k++)
            {
                colPtr[k]++;
            }
            int insertInd = colPtr[col];
            for (int k = insertInd + 1; k < max; k++)
            {
                rowIdx[k] = rowIdx[k - 1];
            }
            rowIdx[insertInd] = i;
            max++;
        }
    }
    for (int i = 0; i < colPtr.size() - 1; i++)
    {
        sort(rowIdx.begin() + colPtr[i], rowIdx.begin() + colPtr[i + 1]);
    }
    return make_tuple(colPtr, rowIdx);
}
int main() {
// Parameters
    vector<int> tile_sizes = {32};
    vector<int> matrix_sizes = {128, 256, 384, 512, 640, 768, 896, 1024, 1152, 1280, 1408, 1536, 1664, 1792, 1920, 2048, 2176, 2304, 2432, 2560, 2688, 2816, 2944, 3072};
    vector<double>time;

    double density = 0.25;

    // Loop over tile sizes
    for (int tile_size : tile_sizes) {
        cout << "Tile Size: " << tile_size << endl;

        // Loop over matrix sizes
        for (int matrix_size : matrix_sizes) {
            cout << "Matrix Size: " << matrix_size << "x" << matrix_size << endl;

            // Generate random sparse matrices A and B
            vector<vector<int>> matrix_A = generateRandomSparseMatrix(matrix_size, matrix_size, density);
            vector<vector<int>> matrix_B = generateRandomSparseMatrix(matrix_size, matrix_size, density);

            // Start measuring time
            struct timeval begin, end;
            gettimeofday(&begin, 0);

            // High-level conversion for matrix A
            vector<int> tilerowPtr_A, tileColidx_A;
            tie(tilerowPtr_A, tileColidx_A) = highLevel_conversion(matrix_A, tile_size);

            // High-level conversion for matrix B
            vector<int> tilerowPtr_B, tileColidx_B;
            tie(tilerowPtr_B, tileColidx_B) = highLevel_conversion(matrix_B, tile_size);

            // Symbolic SpGEMM for CSR Format
            vector<int> tileRowidx_C, tileColidx_C;
            tie(tileRowidx_C, tileColidx_C) = symbolic_SpGEMM_For_CSRFormat(tilerowPtr_A, tileColidx_A, tilerowPtr_B, tileColidx_B);

            int numtileC = tileRowidx_C.size();
            vector<int> tileColPtr_B, tileRowidx_B;
            tie(tileColPtr_B, tileRowidx_B) = highLevel_conversion_For_MatrixB(tilerowPtr_B, tileColidx_B);
            vector<vector<int>> matrix_C = step2_and_step3(matrix_A, matrix_B, tileRowidx_C, tileColidx_C, tilerowPtr_A, tilerowPtr_B, tileColPtr_B, tileColidx_A, tileRowidx_B, numtileC, tile_size);

            gettimeofday(&end, 0);
            long seconds = end.tv_sec - begin.tv_sec;
            long microseconds = end.tv_usec - begin.tv_usec;
            double elapsed = seconds + microseconds*1e-6;
            time.push_back(elapsed);
            printf("Time measured: %.6f seconds.\n", elapsed);
        }
        cout << "Matrix time:" << endl;
        for (const auto& row : time) {
            printf("%.6f, ", row);
        }
        printf("\n");

    }
    return 0;
}

Tile Size: 32
Matrix Size: 128x128
Time measured: 0.027285 seconds.
Matrix Size: 256x256
Time measured: 0.119640 seconds.
Matrix Size: 384x384
Time measured: 0.393547 seconds.
Matrix Size: 512x512
Time measured: 0.696323 seconds.
Matrix Size: 640x640
Time measured: 1.067146 seconds.
Matrix Size: 768x768
Time measured: 0.928622 seconds.
Matrix Size: 896x896
Time measured: 1.275126 seconds.
Matrix Size: 1024x1024
Time measured: 1.632734 seconds.
Matrix Size: 1152x1152
Time measured: 2.083104 seconds.
Matrix Size: 1280x1280
Time measured: 2.586066 seconds.
Matrix Size: 1408x1408
Time measured: 4.120273 seconds.
Matrix Size: 1536x1536
Time measured: 3.752392 seconds.
Matrix Size: 1664x1664
Time measured: 4.666725 seconds.
Matrix Size: 1792x1792
Time measured: 5.437950 seconds.
Matrix Size: 1920x1920
Time measured: 6.448857 seconds.
Matrix Size: 2048x2048
Time measured: 6.655097 seconds.
Matrix Size: 2176x2176
Time measured: 8.487993 seconds.
Matrix Size: 2304x2304
Time measured: 9.474031 s

# SPGEMM

### Density: 0.1

In [ ]:
%%cuda
#include <iostream>
#include <vector>
#include <sys/time.h>
#include <chrono>
#include<random>
using namespace std;

vector<vector<int>> symbolic_SpGEMM(const vector<vector<int>> &A_prime, const vector<vector<int>> &B_prime)
{
    int n = A_prime.size();
    int m = A_prime[0].size();
    int k = B_prime[0].size();

    vector<vector<int>> C_prime(n, vector<int>(k, 0));
    for (int i = 0; i < n; ++i)
    {
        for (int j = 0; j < m; ++j)
        {
            if (A_prime[i][j] == 0)
            {
                continue;
            }
            for (int p = 0; p < k; ++p)
            {
                C_prime[i][p] += A_prime[i][j] * B_prime[j][p];
            }
        }
    }
    return C_prime;
}

vector<vector<int>> generateRandomSparseMatrix(int rows, int cols, double density) {
    vector<vector<int>> matrix(rows, vector<int>(cols, 0));
    std::random_device rd;
    std::mt19937 gen(rd());
    std::uniform_real_distribution<> dis(0.0, 1.0);
    for (int i = 0; i < rows; ++i) {
        for (int j = 0; j < cols; ++j) {
            double randNum = dis(gen);
            if (randNum <= density) {
                matrix[i][j] = 1;
            }
        }
    }
    return matrix;
}

int main() {
    // Size of the matrices
    int matrix_size = 3072;

    // Generate random sparse matrices A and B
    vector<vector<int>> A_prime = generateRandomSparseMatrix(matrix_size, matrix_size, 0.1);
    vector<vector<int>> B_prime = generateRandomSparseMatrix(matrix_size, matrix_size, 0.1);


// Start measuring time
            struct timeval begin, end;
            gettimeofday(&begin, 0);


    vector<vector<int>> matrix_C = symbolic_SpGEMM(A_prime, B_prime);
    gettimeofday(&end, 0);
            long seconds = end.tv_sec - begin.tv_sec;
            long microseconds = end.tv_usec - begin.tv_usec;
            double elapsed = seconds + microseconds*1e-6;

            printf("Time measured: %.6f seconds.\n", elapsed);

    // Optional: Print matrix C
    /*
    cout << "Matrix C:" << endl;
    for (const auto& row : matrix_C) {
        for (int val : row) {
            cout << val << " ";
        }
        cout << endl;
    }
    */

    return 0;
}

Time measured: 53.630491 seconds.



### Density: 0.25

In [ ]:
%%cuda
#include <iostream>
#include <vector>
#include <sys/time.h>
#include <chrono>

using namespace std;

vector<vector<int>> symbolic_SpGEMM(const vector<vector<int>> &A_prime, const vector<vector<int>> &B_prime)
{
    int n = A_prime.size();
    int m = A_prime[0].size();
    int k = B_prime[0].size();

    vector<vector<int>> C_prime(n, vector<int>(k, 0));
    for (int i = 0; i < n; ++i)
    {
        for (int j = 0; j < m; ++j)
        {
            if (A_prime[i][j] == 0)
            {
                continue;
            }
            for (int p = 0; p < k; ++p)
            {
                C_prime[i][p] += A_prime[i][j] * B_prime[j][p];
            }
        }
    }
    return C_prime;
}

vector<vector<int>> generateRandomSparseMatrix(int rows, int cols) {
    vector<vector<int>> matrix(rows, vector<int>(cols, 0));
    for (int i = 0; i < rows; ++i) {
        for (int j = 0; j < cols; ++j) {
            if(i%2 == 0 || j%2==0)
              matrix[i][j] = 0;
            else matrix[i][j] = 1;
        }
    }

    return matrix;
}

int main() {
    // Size of the matrices
    int matrix_size = 1024;

    // Generate random sparse matrices A and B
    vector<vector<int>> A_prime = generateRandomSparseMatrix(matrix_size, matrix_size);
    vector<vector<int>> B_prime = generateRandomSparseMatrix(matrix_size, matrix_size);


// Start measuring time
            struct timeval begin, end;
            gettimeofday(&begin, 0);


    vector<vector<int>> matrix_C = symbolic_SpGEMM(A_prime, B_prime);
    gettimeofday(&end, 0);
            long seconds = end.tv_sec - begin.tv_sec;
            long microseconds = end.tv_usec - begin.tv_usec;
            double elapsed = seconds + microseconds*1e-6;

            printf("Time measured: %.6f seconds.\n", elapsed);

    // Optional: Print matrix C
    /*
    cout << "Matrix C:" << endl;
    for (const auto& row : matrix_C) {
        for (int val : row) {
            cout << val << " ";
        }
        cout << endl;
    }
    */

    return 0;
}

Time measured: 4.594221 seconds.



In [ ]:
%%cuda
#include <iostream>
#include <vector>
#include <sys/time.h>
#include <chrono>

using namespace std;

vector<vector<int>> symbolic_SpGEMM(const vector<vector<int>> &A_prime, const vector<vector<int>> &B_prime)
{
    int n = A_prime.size();
    int m = A_prime[0].size();
    int k = B_prime[0].size();

    vector<vector<int>> C_prime(n, vector<int>(k, 0));
    for (int i = 0; i < n; ++i)
    {
        for (int j = 0; j < m; ++j)
        {
            if (A_prime[i][j] == 0)
            {
                continue;
            }
            for (int p = 0; p < k; ++p)
            {
                C_prime[i][p] += A_prime[i][j] * B_prime[j][p];
            }
        }
    }
    return C_prime;
}

vector<vector<int>> generateRandomSparseMatrix(int rows, int cols) {
    vector<vector<int>> matrix(rows, vector<int>(cols, 0));
    for (int i = 0; i < rows; ++i) {
        for (int j = 0; j < cols; ++j) {
            if(i%2 == 0 || j%2==0)
              matrix[i][j] = 0;
            else matrix[i][j] = 1;
        }
    }

    return matrix;
}

int main() {
    // Size of the matrices
    int matrix_size = 2048;

    // Generate random sparse matrices A and B
    vector<vector<int>> A_prime = generateRandomSparseMatrix(matrix_size, matrix_size);
    vector<vector<int>> B_prime = generateRandomSparseMatrix(matrix_size, matrix_size);


// Start measuring time
            struct timeval begin, end;
            gettimeofday(&begin, 0);


    vector<vector<int>> matrix_C = symbolic_SpGEMM(A_prime, B_prime);
    gettimeofday(&end, 0);
            long seconds = end.tv_sec - begin.tv_sec;
            long microseconds = end.tv_usec - begin.tv_usec;
            double elapsed = seconds + microseconds*1e-6;

            printf("Time measured: %.6f seconds.\n", elapsed);

    // Optional: Print matrix C
    /*
    cout << "Matrix C:" << endl;
    for (const auto& row : matrix_C) {
        for (int val : row) {
            cout << val << " ";
        }
        cout << endl;
    }
    */

    return 0;
}

Time measured: 39.762319 seconds.



In [ ]:
%%cuda
#include <iostream>
#include <vector>
#include <sys/time.h>
#include <chrono>

using namespace std;

vector<vector<int>> symbolic_SpGEMM(const vector<vector<int>> &A_prime, const vector<vector<int>> &B_prime)
{
    int n = A_prime.size();
    int m = A_prime[0].size();
    int k = B_prime[0].size();

    vector<vector<int>> C_prime(n, vector<int>(k, 0));
    for (int i = 0; i < n; ++i)
    {
        for (int j = 0; j < m; ++j)
        {
            if (A_prime[i][j] == 0)
            {
                continue;
            }
            for (int p = 0; p < k; ++p)
            {
                C_prime[i][p] += A_prime[i][j] * B_prime[j][p];
            }
        }
    }
    return C_prime;
}

vector<vector<int>> generateRandomSparseMatrix(int rows, int cols) {
    vector<vector<int>> matrix(rows, vector<int>(cols, 0));
    for (int i = 0; i < rows; ++i) {
        for (int j = 0; j < cols; ++j) {
            if(i%2 == 0 || j%2==0)
              matrix[i][j] = 0;
            else matrix[i][j] = 1;
        }
    }

    return matrix;
}

int main() {
    // Size of the matrices
    int matrix_size = 3072;

    // Generate random sparse matrices A and B
    vector<vector<int>> A_prime = generateRandomSparseMatrix(matrix_size, matrix_size);
    vector<vector<int>> B_prime = generateRandomSparseMatrix(matrix_size, matrix_size);


// Start measuring time
            struct timeval begin, end;
            gettimeofday(&begin, 0);


    vector<vector<int>> matrix_C = symbolic_SpGEMM(A_prime, B_prime);
    gettimeofday(&end, 0);
            long seconds = end.tv_sec - begin.tv_sec;
            long microseconds = end.tv_usec - begin.tv_usec;
            double elapsed = seconds + microseconds*1e-6;

            printf("Time measured: %.6f seconds.\n", elapsed);

    // Optional: Print matrix C
    /*
    cout << "Matrix C:" << endl;
    for (const auto& row : matrix_C) {
        for (int val : row) {
            cout << val << " ";
        }
        cout << endl;
    }
    */

    return 0;
}

Time measured: 131.929816 seconds.



## Plots

In [ ]:
import plotly.graph_objects as go

# Data from the table with non-zero values
matrix_sizes = [
    "128 × 128", "256 × 256", "384 × 384", "512 × 512", "640 × 640",
    "768 × 768", "896 × 896", "1024 × 1024", "1152 × 1152", "1280 × 1280",
    "1408 × 1408", "1536 × 1536", "1664 × 1664", "1792 × 1792", "1920 × 1920",
    "2048 × 2048", "2176 × 2176", "2304 × 2304", "2432 × 2432", "2560 × 2560",
    "2668 × 2668", "2816 × 2816", "2944 × 2944", "3072 × 3072"
]

# Performance data for SpGEMM and Tile SpGEMM methods
data = [
    [0.207976, 0.310486, 0.199112, 0.212693, 0.216782, 0.347997],
    [0.002439, 0.064172, 0.009834, 0.005298, 0.022222, 0.122070],
    [0.004907, 0.252681, 0.036622, 0.009267, 0.015872, 0.063753],
    [0.009052, 0.687308, 0.091815, 0.016020, 0.021370, 0.050670],
    [0.014791, 2.116419, 0.192753, 0.030048, 0.023924, 0.041306],
    [0.021782, 3.558099, 0.344575, 0.052976, 0.034323, 0.048014],
    [0.028486, 4.782732, 0.587618, 0.085661, 0.039579, 0.057205],
    [0.039174, 8.135314, 0.932178, 0.131731, 0.033411, 0.061193],
    [0.053052, 13.004249, 1.388752, 0.198508, 0.047916, 0.065416],
    [0.063905, 17.237437, 2.067790, 0.235554, 0.046588, 0.067242],
    [0.077996, 25.211076, 4.510172, 0.264777, 0.062996, 0.069979],
    [0.095125, 34.130772, 4.086794, 0.357402, 0.085326, 0.099731],
    [0.113193, 46.719020, 5.507133, 0.458635, 0.101356, 0.096534],
    [0.133540, 69.613498, 6.654089, 0.603061, 0.122329, 0.097549],
    [0.155880, 80.638414, 8.923974, 0.773268, 0.120297, 0.102245],
    [0.179390, 101.030979, 11.043611, 0.976421, 0.133956, 0.101267],
    [0.207383, 121.935092, 13.775232, 1.187396, 0.213630, 0.100610],
    [0.248631, 151.563901, 15.921598, 1.718799, 0.194512, 0.135486],
    [0.265581, 188.257157, 19.847924, 1.668756, 0.234601, 0.139053],
    [0.377470, 228.197896, 23.531966, 2.009511, 0.256956, 0.146734],
    [0.341013, 270.910376, 28.856130, 2.412802, 0.523822, 0.135765],
    [0.327353, 319.270456, 33.715897, 3.361281, 0.348231, 0.138116],
    [0.344476, 384.014558, 39.336449, 3.244895, 0.398874, 0.136251],
    [0.368696, 445.135530, 47.172449, 4.166786, 0.566285, 0.226653]
]

# Transpose the data for easier plotting
data_transposed = list(map(list, zip(*data)))

# Create traces for SpGEMM and Tile SpGEMM methods
fig = go.Figure()

# Plot SpGEMM method
fig.add_trace(go.Scatter(x=matrix_sizes, y=data_transposed[0], mode='lines+markers', name='SpGEMM'))

# Plot Tile SpGEMM methods
for i in range(1, len(data_transposed)):
    fig.add_trace(go.Scatter(x=matrix_sizes, y=data_transposed[i], mode='lines+markers', name=f'Tile SpGEMM with Tile Size: {2**(i+1)}'))

# Update layout
fig.update_layout(
    title='Performance of SpGEMM and Tile SpGEMM Methods',
    xaxis_title='Matrix Size',
    yaxis_title='Time (seconds)',
    xaxis_tickangle=45,
    xaxis_type='category'
)

# Show plot
fig.show()

In [ ]:
import plotly.graph_objects as go

# New data
matrix_sizes = ["128 × 128", "256 × 256", "384 × 384", "512 × 512", "640 × 640",
                "768 × 768", "896 × 896", "1024 × 1024", "1152 × 1152", "1280 × 1280",
                "1408 × 1408", "1536 × 1536", "1664 × 1664", "1792 × 1792", "1920 × 1920",
                "2048 × 2048", "2176 × 2176", "2304 × 2304", "2432 × 2432", "2560 × 2560",
                "2668 × 2668", "2816 × 2816", "2944 × 2944", "3072 × 3072"]

data = [
    [0.207976, 0.310486, 0.199112, 0.212693, 0.216782, 0.347997],
    [0.002439, 0.064172, 0.009834, 0.005298, 0.022222, 0.122070],
    [0.004907, 0.252681, 0.036622, 0.009267, 0.015872, 0.063753],
    [0.009052, 0.687308, 0.091815, 0.016020, 0.021370, 0.050670],
    [0.014791, 2.116419, 0.192753, 0.030048, 0.023924, 0.041306],
    [0.021782, 3.558099, 0.344575, 0.052976, 0.034323, 0.048014],
    [0.028486, 4.782732, 0.587618, 0.085661, 0.039579, 0.057205],
    [0.039174, 8.135314, 0.932178, 0.131731, 0.033411, 0.061193],
    [0.053052, 13.004249, 1.388752, 0.198508, 0.047916, 0.065416],
    [0.063905, 17.237437, 2.067790, 0.235554, 0.046588, 0.067242],
    [0.077996, 25.211076, 4.510172, 0.264777, 0.062996, 0.069979],
    [0.095125, 34.130772, 4.086794, 0.357402, 0.085326, 0.099731],
    [0.113193, 46.719020, 5.507133, 0.458635, 0.101356, 0.096534],
    [0.133540, 69.613498, 6.654089, 0.603061, 0.122329, 0.097549],
    [0.155880, 80.638414, 8.923974, 0.773268, 0.120297, 0.102245],
    [0.179390, 101.030979, 11.043611, 0.976421, 0.133956, 0.101267],
    [0.207383, 121.935092, 13.775232, 1.187396, 0.213630, 0.100610],
    [0.248631, 151.563901, 15.921598, 1.718799, 0.194512, 0.135486],
    [0.265581, 188.257157, 19.847924, 1.668756, 0.234601, 0.139053],
    [0.377470, 228.197896, 23.531966, 2.009511, 0.256956, 0.146734],
    [0.341013, 270.910376, 28.856130, 2.412802, 0.523822, 0.135765],
    [0.327353, 319.270456, 33.715897, 3.361281, 0.348231, 0.138116],
    [0.344476, 384.014558, 39.336449, 3.244895, 0.398874, 0.136251],
    [0.368696, 445.135530, 47.172449, 4.166786, 0.566285, 0.226653]
]

# Create traces for SpGEMM and Tile SpGEMM methods
fig = go.Figure()

# Plot SpGEMM method
fig.add_trace(go.Scatter(x=matrix_sizes, y=[row[0] for row in data], mode='lines+markers', name='SpGEMM'))

# Plot Tile SpGEMM methods
for i in range(1, len(data[0])):
    fig.add_trace(go.Scatter(x=matrix_sizes, y=[row[i] for row in data], mode='lines+markers', name=f'Tile SpGEMM with Tile size: {2**(i+1)}'))

# Update layout
fig.update_layout(
    title='Performance of SpGEMM and Tile SpGEMM Methods (Logarithmic Scale)',
    xaxis_title='Matrix Size',
    yaxis_title='Time (seconds)',
    xaxis_tickangle=-45,
    yaxis_type='log',
    xaxis_type='category'
)

# Show plot
fig.show()

In [ ]:
import plotly.graph_objects as go

# Data for SpGEMM and Tile SpGEMM 1
matrix_sizes = [
    "128 × 128", "256 × 256", "384 × 384", "512 × 512", "640 × 640",
    "768 × 768", "896 × 896", "1024 × 1024", "1152 × 1152", "1280 × 1280",
    "1408 × 1408", "1536 × 1536", "1664 × 1664", "1792 × 1792", "1920 × 1920",
    "2048 × 2048", "2176 × 2176", "2304 × 2304", "2432 × 2432", "2560 × 2560",
    "2668 × 2668", "2816 × 2816", "2944 × 2944", "3072 × 3072"
]

spgemm_times = [
    0.212877, 0.002432, 0.005166, 0.008598, 0.014841, 0.020979, 0.029586,
    0.038004, 0.051803, 0.084120, 0.101488, 0.117864, 0.141348, 0.133302,
    0.152837, 0.178168, 0.202627, 0.233082, 0.259242, 0.290695, 0.330760,
    0.331293, 0.469497, 0.401495
]

tile_spgemm1_times = [
    0.823629, 0.145960, 0.071950, 0.065733, 0.041699, 0.049148, 0.057713,
    0.063523, 0.066514, 0.069770, 0.100053, 0.119043, 0.111994, 0.160685,
    0.150646, 0.101102, 0.104485, 0.146030, 0.127749, 0.118541, 0.192538,
    0.140258, 0.140874, 0.203803
]

# Create traces for SpGEMM and Tile SpGEMM 1
fig = go.Figure()

# Add trace for SpGEMM
fig.add_trace(go.Scatter(x=matrix_sizes, y=spgemm_times, mode='lines+markers', name='SpGEMM'))

# Add trace for Tile SpGEMM 1
fig.add_trace(go.Scatter(x=matrix_sizes, y=tile_spgemm1_times, mode='lines+markers', name='Tile SpGEMM 1'))

# Update layout
fig.update_layout(
    title='Performance of SpGEMM and Tile SpGEMM 1 Methods',
    xaxis_title='Matrix Size',
    yaxis_title='Time (seconds)',
    xaxis_tickangle=-45,
    yaxis_type='log'
)

# Show plot
fig.show()


In [ ]:
import plotly.graph_objects as go

# Data for SpGEMM and Tile SpGEMM 1
matrix_sizes = [
    "128 × 128", "256 × 256", "384 × 384", "512 × 512", "640 × 640",
    "768 × 768", "896 × 896", "1024 × 1024", "1152 × 1152", "1280 × 1280",
    "1408 × 1408", "1536 × 1536", "1664 × 1664", "1792 × 1792", "1920 × 1920",
    "2048 × 2048", "2176 × 2176", "2304 × 2304", "2432 × 2432", "2560 × 2560",
    "2668 × 2668", "2816 × 2816", "2944 × 2944", "3072 × 3072"
]

spgemm_times = [
    0.212877, 0.002432, 0.005166, 0.008598, 0.014841, 0.020979, 0.029586,
    0.038004, 0.051803, 0.084120, 0.101488, 0.117864, 0.141348, 0.133302,
    0.152837, 0.178168, 0.202627, 0.233082, 0.259242, 0.290695, 0.330760,
    0.331293, 0.469497, 0.401495
]

tile_spgemm1_times = [
    0.823629, 0.145960, 0.071950, 0.065733, 0.041699, 0.049148, 0.057713,
    0.063523, 0.066514, 0.069770, 0.100053, 0.119043, 0.111994, 0.160685,
    0.150646, 0.101102, 0.104485, 0.146030, 0.127749, 0.118541, 0.192538,
    0.140258, 0.140874, 0.203803
]

# Create a figure for the logarithmic scale plot
fig_log = go.Figure()

# Add trace for SpGEMM (log scale)
fig_log.add_trace(go.Scatter(x=matrix_sizes, y=spgemm_times, mode='lines+markers', name='SpGEMM'))

# Add trace for Tile SpGEMM 1 (log scale)
fig_log.add_trace(go.Scatter(x=matrix_sizes, y=tile_spgemm1_times, mode='lines+markers', name='Tile SpGEMM'))

# Update layout for logarithmic scale plot
fig_log.update_layout(
    title='Performance of SpGEMM and Tile SpGEMM Methods (Log Scale)',
    xaxis_title='Matrix Size',
    yaxis_title='Time (seconds)',
    xaxis_tickangle=-45,
    yaxis_type='log'
)

# Show logarithmic scale plot
fig_log.show()

# Create a figure for the linear scale plot
fig_linear = go.Figure()

# Add trace for SpGEMM (linear scale)
fig_linear.add_trace(go.Scatter(x=matrix_sizes, y=spgemm_times, mode='lines+markers', name='SpGEMM'))

# Add trace for Tile SpGEMM 1 (linear scale)
fig_linear.add_trace(go.Scatter(x=matrix_sizes, y=tile_spgemm1_times, mode='lines+markers', name='Tile SpGEMM'))

# Update layout for linear scale plot
fig_linear.update_layout(
    title='Performance of SpGEMM and Tile SpGEMM Methods (Linear Scale)',
    xaxis_title='Matrix Size',
    yaxis_title='Time (seconds)',
    xaxis_tickangle=-45,
    yaxis_type='linear'
)

# Show linear scale plot
fig_linear.show()


## Extra

In [ ]:
l = {128, 160, 192, 224, 256, 288, 320, 352, 384, 416, 448, 480, 512, 544, 576, 608, 640, 672, 704, 736, 768, 800, 832, 864, 896, 928, 960, 992, 1024, 1056}
c=0
i=32
lis = []
while c < 30:
  lis.append(96 + i)
  i+=128
  c +=1
print(lis)

[128, 256, 384, 512, 640, 768, 896, 1024, 1152, 1280, 1408, 1536, 1664, 1792, 1920, 2048, 2176, 2304, 2432, 2560, 2688, 2816, 2944, 3072, 3200, 3328, 3456, 3584, 3712, 3840]
